# CMIP6 Historical Bias Analysis (1979-2014)

Evaluates 10 CMIP6 models against MSWX reference for the historical period 1979-2014.
Produces Figure 10 (annual domain-averaged temperature time series), Supplementary Tables 7A-C
(climatology, OLS trends, Mann-Kendall trends), and the Borda rank-aggregation model selection.

**Before running:** Update all path variables marked `<SET YOUR PATH HERE>` to your local data directories.

**Paper reference:** Jha, S.K., Gupta, V., et al. (2026). A Generative AI Framework for High-Fidelity Probabilistic Downscaling of CMIP6 Daily Temperature Projections over India. *Communications Earth & Environment.*

# Heading: Compute spatial temperature bias of multiple CMIP6 models against MSWX (1979–2014), generate bias maps, grid comparison plots, and model performance metrics (RMSE, pattern correlation, std difference).

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature


# ────────────────────────────
# Helper: Handle calendar
# ────────────────────────────
def get_calendar_max_date(ds, var="time"):
    calendar = ds[var].encoding.get("calendar", "standard")
    if "360_day" in calendar:
        return "2014-12-30"
    elif calendar in ["365_day", "noleap"]:
        return "2014-12-31"
    elif calendar in ["standard", "gregorian"]:
        return "2014-12-31"
    else:
        return str(ds[var].values[-1])[:10]


# ────────────────────────────
# Compute Bias and Metrics
# ────────────────────────────
def compute_bias(mswx_file, cmip6_file, start="1979-01-01"):
    ds_obs = xr.open_dataset(mswx_file)
    ds_mod = xr.open_dataset(cmip6_file)

    # Detect safe end date
    end_mod = get_calendar_max_date(ds_mod, "time")
    end_obs = get_calendar_max_date(ds_obs, "time")
    safe_end = min(end_obs, end_mod)

    obs = ds_obs["air_temperature"].sel(time=slice(start, safe_end))
    mod = ds_mod["tas"].sel(time=slice(start, safe_end))

    # Convert to Celsius if needed
    if mod.max() > 200:
        mod = mod - 273.15

    # Time-mean fields
    obs_mean = obs.mean(dim="time")
    mod_mean = mod.mean(dim="time")
    bias = mod_mean - obs_mean

    return bias, obs_mean, mod_mean


# ────────────────────────────
# Main Function: Grid Plot + Metrics
# ────────────────────────────
def plot_all_bias(mswx_file, cmip6_dir,
                  output_dir="<SET YOUR PATH HERE>",
                  ncols=3):
    os.makedirs(output_dir, exist_ok=True)

    # Get all CMIP6 files
    files = [os.path.join(cmip6_dir, f) for f in os.listdir(cmip6_dir) if f.endswith(".nc")]
    files.sort()

    n_models = len(files)
    nrows = math.ceil(n_models / ncols)

    metrics = []  # store results
    fig = plt.figure(figsize=(ncols * 6, nrows * 4))

    for idx, fpath in enumerate(files, start=1):
        fname = os.path.basename(fpath)
        parts = fname.split("_")
        model_name = parts[2].upper() if len(parts) > 2 else os.path.splitext(fname)[0].upper()

        # Compute bias and reference
        bias, obs_mean, mod_mean = compute_bias(mswx_file, fpath)

        # ─── Metrics ───
        mean_bias = float(bias.mean().values)
        rmse = float(np.sqrt(((mod_mean - obs_mean) ** 2).mean().values))
        std_diff = float(mod_mean.std().values - obs_mean.std().values)

        # Pattern correlation (spatial)
        x = obs_mean.values.flatten()
        y = mod_mean.values.flatten()
        mask = np.isfinite(x) & np.isfinite(y)
        x = x[mask]
        y = y[mask]
        x_anom = x - np.mean(x)
        y_anom = y - np.mean(y)
        pattern_corr = float(np.sum(x_anom * y_anom) /
                             np.sqrt(np.sum(x_anom ** 2) * np.sum(y_anom ** 2)))

        metrics.append([model_name, mean_bias, rmse, pattern_corr, std_diff])

        # ─── Plot in Grid ───
        ax = plt.subplot(nrows, ncols, idx, projection=ccrs.PlateCarree())
        ax.set_extent([50, 100, 0, 50], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
        ax.add_feature(cfeature.LAKES, facecolor="white", edgecolor="black", linewidth=0.3)
        ax.add_feature(cfeature.RIVERS, edgecolor="blue", linewidth=0.2)

        bias.plot(ax=ax, transform=ccrs.PlateCarree(), cmap="coolwarm",
                  vmin=-5, vmax=5, add_colorbar=False)
        ax.set_title(model_name, fontsize=10)

        # ─── Save individual plot ───
        single_fig = plt.figure(figsize=(6, 5))
        ax_single = plt.axes(projection=ccrs.PlateCarree())
        ax_single.set_extent([50, 100, 0, 50], crs=ccrs.PlateCarree())
        ax_single.add_feature(cfeature.COASTLINE, linewidth=1)
        ax_single.add_feature(cfeature.LAKES, facecolor="white", edgecolor="black")
        ax_single.add_feature(cfeature.RIVERS, edgecolor="blue", linewidth=0.3)
        bias.plot(ax=ax_single, transform=ccrs.PlateCarree(), cmap="coolwarm",
                  vmin=-5, vmax=5,
                  cbar_kwargs={"label": "Temperature Bias (°C)", "orientation": "vertical"})
        ax_single.set_title(f"{model_name} BIAS VS MSWX", fontsize=14)
        ax_single.set_xlabel("")
        ax_single.set_ylabel("")
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"{model_name}_BIAS.png"), dpi=300)
        plt.close(single_fig)

        print(f"✅ Saved: {model_name}_BIAS.png")

    # ────────────────────────────
    # Shared colorbar
    # ────────────────────────────
    import matplotlib.cm as cm
    import matplotlib.colors as mcolors

    cmap = plt.get_cmap("coolwarm")
    norm = mcolors.Normalize(vmin=-5, vmax=5)
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    cbar_ax = fig.add_axes([0.25, 0.05, 0.5, 0.02])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
    cbar.set_label("Temperature Bias (°C)", fontsize=12)
    cbar.ax.tick_params(labelsize=10, width=0.5, length=3)

    for ax in fig.axes:
        if hasattr(ax, "get_ylabel") and "latitude" in ax.get_ylabel().lower():
            ax.set_ylabel("")
        if hasattr(ax, "get_xlabel") and "longitude" in ax.get_xlabel().lower():
            ax.set_xlabel("")

    plt.suptitle("CMIP6 Model Bias vs MSWX (1979–2014)", fontsize=18)
    grid_path = os.path.join(output_dir, "ALL_MODELS_BIAS_GRID.png")
    plt.savefig(grid_path, dpi=300)
    plt.show()
    print(f"✅ Full grid saved at: {grid_path}")

    # ────────────────────────────
    # Save metrics summary
    # ────────────────────────────
    df = pd.DataFrame(metrics, columns=["Model", "Mean_Bias(°C)", "RMSE(°C)", "Pattern_Corr", "Std_Diff(°C)"])
    df = df.sort_values("RMSE(°C)").reset_index(drop=True)
    csv_path = os.path.join(output_dir, "Model_Performance_Summary.csv")
    df.to_csv(csv_path, index=False)

    print("\n📊 Model performance summary (sorted by RMSE):")
    print(df.to_string(index=False))
    print(f"\n✅ Metrics saved to: {csv_path}")


# ────────────────────────────
# Example Usage
# ────────────────────────────
mswx_file = "<SET YOUR PATH HERE>"
cmip6_dir = "<SET YOUR PATH HERE>"

plot_all_bias(mswx_file, cmip6_dir)

### Robust composite ranking score using the Borda rank-aggregation method

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ================================
# Step 1: Load CSV
# ================================
csv_path = "<SET YOUR PATH HERE>"

try:
    df = pd.read_csv(csv_path)
except UnicodeDecodeError:
    df = pd.read_csv(csv_path, encoding="latin-1")

# Clean column names
df.columns = df.columns.str.replace(r"[^\x00-\x7F]+", "", regex=True)
df.columns = df.columns.str.strip()
print("✅ Columns:", list(df.columns))

# ================================
# Step 2: Identify metric columns
# ================================
col_bias = [c for c in df.columns if "Bias" in c][0]
col_rmse = [c for c in df.columns if "RMSE" in c][0]
col_corr = [c for c in df.columns if "Pattern" in c or "Corr" in c][0]
col_std = [c for c in df.columns if "Std" in c][0]

metrics = [col_bias, col_rmse, col_corr, col_std]
print(f"Using metrics: {metrics}")

# ================================
# Step 3: Borda Count Ranking
# ================================
df_borda = df.copy()

# Lower-is-better metrics: rank ascending (1 = best)
for col in [col_bias, col_rmse, col_std]:
    df_borda[col + "_rank"] = df_borda[col].rank(method='average', ascending=True)

# Higher-is-better metrics: rank descending (1 = best)
df_borda[col_corr + "_rank"] = df_borda[col_corr].rank(method='average', ascending=False)

# Total Borda score (sum of ranks)
rank_cols = [c for c in df_borda.columns if "_rank" in c]
df_borda["Borda_Score"] = df_borda[rank_cols].sum(axis=1)
df_borda = df_borda.sort_values("Borda_Score").reset_index(drop=True)
df_borda["Borda_Rank"] = df_borda.index + 1

top10_borda = df_borda[["Borda_Rank", "Model", "Borda_Score"] + metrics].head(10)
print("\n===== Top 10 Models by Borda Count =====\n")
print(top10_borda.to_string(index=False))


top10_pca = df_pca[["PCA_Rank", "Model", "PCA_Score"] + metrics].head(10)
print("\n===== Top 10 Models by PCA Composite Score =====\n")
print(top10_pca.to_string(index=False))

# ================================
# Step 5: Save results
# ================================
top10_borda.to_csv("<SET YOUR PATH HERE>", index=False)
top10_pca.to_csv("<SET YOUR PATH HERE>", index=False)
print("\n✅ Results saved to 'top10_models_borda.csv' and 'top10_models_pca.csv'")

In [ ]:
import xarray as xr
import os

def inspect_netcdf_files(folder_path):
    """
    Scans a folder for NetCDF files and prints a detailed summary of each one.

    Args:
        folder_path (str): The path to the directory containing NetCDF files.
    """
    print(f"--- Scanning folder: {folder_path} ---\n")
    
    # --- 1. Find all relevant NetCDF files in the directory ---
    try:
        # Use the same pattern to find the files as in the ensemble script
        all_filenames = [f for f in os.listdir(folder_path) if f.endswith('_QDM_corrected.nc')]
        if not all_filenames:
            print(f"No files ending with '_QDM_corrected.nc' found in '{folder_path}'")
            return
            
        nc_files = [os.path.join(folder_path, f) for f in all_filenames]

    except FileNotFoundError:
        print(f"Error: The specified folder path does not exist: {folder_path}")
        return

    # --- 2. Loop through each file and print its details ---
    for i, file_path in enumerate(nc_files):
        print("="*80)
        print(f"({i+1}/{len(nc_files)}) Details for: {os.path.basename(file_path)}")
        print("="*80)
        
        try:
            # Open the dataset using xarray
            with xr.open_dataset(file_path) as ds:
                # The default print output from xarray is a comprehensive summary
                print(ds)
                
                # You could also print specific parts, for example:
                # print("\n--- Global Attributes ---")
                # for attr, value in ds.attrs.items():
                #     print(f"{attr}: {value}")
                
        except Exception as e:
            print(f"\nCould not read or process file. Error: {e}")
        
        print("\n" * 2) # Add some space before the next file's details

if __name__ == '__main__':
    # --- User-defined parameter ---
    
    # IMPORTANT: Please verify this path is correct for your system.
    FOLDER_PATH = "<SET YOUR PATH HERE>"
    
    # --- Execute the main function ---
    inspect_netcdf_files(FOLDER_PATH)


### Unify CMIP6 Calendars and Calculate Ensemble Mean

In [ ]:
import xarray as xr
import os
import pandas as pd
import warnings
import cftime
from xarray.coding.times import CFDatetimeCoder

# Suppress common warnings that occur when xarray handles different calendars
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

def create_cmip6_ensemble(folder_path, start_year, end_year, variable_name, output_filename):
    """
    Creates a multi-model ensemble mean for CMIP6 data, robustly handling diverse calendars.

    This function reads multiple NetCDF files, slices them using a calendar-aware
    boolean mask (the most reliable method), aligns them to a common 'standard'
    calendar, calculates the ensemble mean, and saves the result to a new file.

    Args:
        folder_path (str): Path to the directory containing CMIP6 model NetCDF files.
        start_year (int): The start year for the time slicing.
        end_year (int): The end year for the time slicing.
        variable_name (str): The name of the climate variable (e.g., 'tas').
        output_filename (str): The name of the output NetCDF file.
    """
    # --- 1. Identify NetCDF files and extract model names ---
    try:
        all_filenames = [f for f in os.listdir(folder_path) if f.endswith('_QDM_corrected.nc')]
        if not all_filenames:
            print(f"Error: No files ending with '_QDM_corrected.nc' found in '{folder_path}'")
            return

        nc_files = [os.path.join(folder_path, f) for f in all_filenames]
        model_names = [f.replace('_QDM_corrected.nc', '') for f in all_filenames]
        print(f"Found {len(model_names)} models to process: {', '.join(model_names)}")

    except FileNotFoundError:
        print(f"Error: The specified folder path does not exist: {folder_path}")
        return

    # --- 2. Open, slice, and align each dataset ---
    processed_data_arrays = []
    
    # Define a standard daily calendar to align all models to.
    target_time_index = xr.cftime_range(
        start=f'{start_year}-01-01',
        end=f'{end_year}-12-31',
        calendar='standard',
        freq='D'
    )
    
    time_coder = CFDatetimeCoder(use_cftime=True)
    successful_models = []

    for i, file_path in enumerate(nc_files):
        model_name = model_names[i]
        print(f"Processing ({i+1}/{len(nc_files)}): {os.path.basename(file_path)}...")
        try:
            with xr.open_dataset(file_path, decode_times=time_coder) as ds:
                
                # THE ROBUST FIX:
                # 1. Get the model's specific calendar.
                model_calendar = ds.time.dt.calendar
                
                # 2. Create start and end dates using the model's OWN calendar.
                start_date = cftime.datetime(start_year, 1, 1, calendar=model_calendar)
                end_date = cftime.datetime(end_year, 12, 31, calendar=model_calendar)

                # 3. Create a boolean mask to select the time range. This avoids direct
                #    comparison errors that `.sel()` can cause with mixed calendars.
                time_mask = (ds.time >= start_date) & (ds.time <= end_date)
                ds_sliced = ds.where(time_mask, drop=True)

                if ds_sliced.time.size == 0:
                    raise ValueError("Slicing resulted in an empty dataset. Check time range.")
                
                # 4. Reindex the correctly-sliced data to the common standard calendar.
                ds_aligned = ds_sliced.reindex(
                    {'time': target_time_index},
                    method='nearest',
                    tolerance='1D'
                )
                
                processed_data_arrays.append(ds_aligned[variable_name])
                successful_models.append(model_name)

        except Exception as e:
            print(f"  -> Could not process {model_name}. Error: {e}. Skipping this model.")

    if not processed_data_arrays:
        print("\nNo datasets were successfully processed. Aborting ensemble creation.")
        return

    print(f"\nSuccessfully processed {len(successful_models)} models.")
    
    # --- 3. Combine all models into a single DataArray ---
    print("Combining models into a single data structure...")
    combined_models = xr.concat(
        processed_data_arrays,
        dim=pd.Index(successful_models, name='model')
    )

    # --- 4. Calculate the ensemble mean ---
    print("Calculating the ensemble mean...")
    ensemble_mean = combined_models.mean(dim='model', skipna=True)

    # --- 5. Add descriptive metadata ---
    ensemble_mean.attrs = combined_models.attrs
    ensemble_mean.attrs['long_name'] = f'Ensemble Mean of Daily {variable_name.capitalize()}'
    ensemble_mean.attrs['description'] = (
        f'Multi-model ensemble mean from {len(successful_models)} CMIP6 models '
        f'for the period {start_year}-{end_year}. Data aligned to a standard calendar.'
    )
    ensemble_mean.attrs['source_models_used'] = ', '.join(successful_models)
    
    ensemble_dataset = ensemble_mean.to_dataset(name=variable_name)

    # --- 6. Save the final ensemble to a NetCDF file ---
    # The output will be saved one level up from the input folder
    output_path = os.path.join(os.path.dirname(folder_path), output_filename)
    try:
        print(f"Saving ensemble to: {output_path}")
        ensemble_dataset.to_netcdf(output_path)
        print("\nEnsemble created successfully!")
    except Exception as e:
        print(f"\nError: Could not save the output file. {e}")


if __name__ == '__main__':
    # --- User-defined parameters ---
    FOLDER_PATH = "<SET YOUR PATH HERE>"
    VARIABLE = "tas" 
    START_YEAR = 1979
    END_YEAR = 2014
    OUTPUT_FILENAME = f"CMIP6_Top10_Ensemble_{VARIABLE}_{START_YEAR}-{END_YEAR}.nc"

    # --- Execute the main function ---
    create_cmip6_ensemble(
        folder_path=FOLDER_PATH,
        start_year=START_YEAR,
        end_year=END_YEAR,
        variable_name=VARIABLE,
        output_filename=OUTPUT_FILENAME
    )

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import rioxarray
import geopandas as gpd
import matplotlib.colors as colors

# -----------------------------
# 1. Paths
# -----------------------------
hr_mswx = "<SET YOUR PATH HERE>"
hr_ddpm = "<SET YOUR PATH HERE>"
india_shp = "<SET YOUR PATH HERE>"

# -----------------------------
# 2. Helper functions
# -----------------------------
def get_file_date(fname):
    """Extracts YYYYDOY from filename."""
    base = os.path.basename(fname)
    base = base.replace("_ddpm", "").replace(".nc", "")
    return int(base)

def doy_to_datetime(yydoy):
    """Converts YYYYDOY to datetime object."""
    year = int(str(yydoy)[:4])
    doy = int(str(yydoy)[4:])
    return pd.to_datetime(f"{year}-{doy:03d}", format="%Y-%j")

def filter_period(files, start=2009219, end=2014365): # Testing period
    """Filters a list of files to be within the start/end YYYYDOY range."""
    valid = []
    for f in files:
        try:
            d = get_file_date(f)
            if start <= d <= end:
                valid.append(f)
        except:
            continue
    return valid

# -----------------------------
# 3. Load files
# -----------------------------
f_mswx = sorted(glob.glob(os.path.join(hr_mswx, "*.nc")))
f_ddpm = sorted(glob.glob(os.path.join(hr_ddpm, "*_ddpm.nc")))

f_mswx = filter_period(f_mswx)
f_ddpm = filter_period(f_ddpm)

print(f"Found {len(f_mswx)} MSWX files and {len(f_ddpm)} DDPM files for the test period.")

# -----------------------------
# 4. Open datasets
# -----------------------------
ds_mswx = xr.open_mfdataset(f_mswx, combine="nested", concat_dim="time", engine="netcdf4")
ds_ddpm = xr.open_mfdataset(f_ddpm, combine="nested", concat_dim="time", engine="netcdf4")

# -----------------------------
# 5. Assign time coordinate
# -----------------------------
if "time" not in ds_mswx.coords:
    doy_list = [get_file_date(f) for f in f_mswx]
    times = [doy_to_datetime(d) for d in doy_list]
    ds_mswx = ds_mswx.assign_coords(time=("time", times))
    ds_ddpm = ds_ddpm.assign_coords(time=("time", times))

# -----------------------------
# 6. Calculate Annual & Overall Mean
# -----------------------------
annual_mswx = ds_mswx["air_temperature"].groupby("time.year").mean("time")
annual_ddpm = ds_ddpm["air_temperature"].groupby("time.year").mean("time")

overall_mswx = annual_mswx.mean("year")
overall_ddpm = annual_ddpm.mean("year")
overall_bias = overall_ddpm - overall_mswx

print("✅ Annual and overall means computed")

# -----------------------------
# 7. Clip Data to Shapefile
# -----------------------------
# Rename lat/lon to y/x
overall_mswx = overall_mswx.rename({"lat": "y", "lon": "x"})
overall_ddpm = overall_ddpm.rename({"lat": "y", "lon": "x"})
overall_bias = overall_bias.rename({"lat": "y", "lon": "x"})

# Set spatial dims and CRS
for da in [overall_mswx, overall_ddpm, overall_bias]:
    da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
    da.rio.write_crs("EPSG:4326", inplace=True)

# Load India shapefile
gdf = gpd.read_file(india_shp)

# Clip to India
overall_mswx_clipped = overall_mswx.rio.clip(gdf.geometry, gdf.crs)
overall_ddpm_clipped = overall_ddpm.rio.clip(gdf.geometry, gdf.crs)
overall_bias_clipped = overall_bias.rio.clip(gdf.geometry, gdf.crs)

print("✅ Data clipped to shapefile")

# ----------------------------------------------------
# 8. Calculate & Print Summary Statistics
# ----------------------------------------------------
print("\n" + "="*70)
print("     SUMMARY: Mean Spatial Statistics for Test Period")
print(f"     (Period: 2009-08-07 to 2014-12-31, n_days={len(f_mswx)})")
print("="*70 + "\n")

# --- MSWX (Target) Statistics ---
stats_mswx_mean = float(overall_mswx_clipped.mean().values)
stats_mswx_min = float(overall_mswx_clipped.min().values)
stats_mswx_max = float(overall_mswx_clipped.max().values)

print("--- MSWX (Target) ---")
print(f"Overall Spatial Mean: {stats_mswx_mean:.4f} °C")
print(f"Overall Spatial Min:  {stats_mswx_min:.4f} °C")
print(f"Overall Spatial Max:  {stats_mswx_max:.4f} °C")
print("\n")

# --- DDPM (Prediction) Statistics ---
stats_ddpm_mean = float(overall_ddpm_clipped.mean().values)
stats_ddpm_min = float(overall_ddpm_clipped.min().values)
stats_ddpm_max = float(overall_ddpm_clipped.max().values)

print("--- DDPM (Prediction) ---")
print(f"Overall Spatial Mean: {stats_ddpm_mean:.4f} °C")
print(f"Overall Spatial Min:  {stats_ddpm_min:.4f} °C")
print(f"Overall Spatial Max:  {stats_ddpm_max:.4f} °PosC")
print("\n")

# --- BIAS (DDPM - MSWX) Statistics ---
bias_values_clipped = overall_bias_clipped.values 
stats_bias_mean = float(np.nanmean(bias_values_clipped))
stats_mab = float(np.nanmean(np.abs(bias_values_clipped)))
stats_rmse = float(np.sqrt(np.nanmean(np.square(bias_values_clipped))))
stats_bias_min = float(np.nanmin(bias_values_clipped))
stats_bias_max = float(np.nanmax(bias_values_clipped))

print("--- BIAS (DDPM - MSWX) ---")
print(f"Mean Spatial Bias (MB):   {stats_bias_mean:.4f} °C")
print(f"Mean Absolute Bias (MAB): {stats_mab:.4f} °C")
print(f"Spatial RMSE:             {stats_rmse:.4f} °C")
print(f"Min Bias (Coldest):       {stats_bias_min:.4f} °C")
print(f"Max Bias (Warmest):       {stats_bias_max:.4f} °C")
print("\n")

# -----------------------------
# 9. Define Plotting Parameters
# -----------------------------
vmin_temp = float(min(stats_mswx_min, stats_ddpm_min))
vmax_temp = float(max(stats_mswx_max, stats_ddpm_max))
# Define normalization for the bias plot (centered at 0)
bias_norm = colors.TwoSlopeNorm(vcenter=0, vmin=-0.5, vmax=0.5)

# Define a sequential/perceptual colormap for temperature
temp_cmap = "Spectral_r" 
# Define a diverging colormap for bias
bias_cmap = "RdBu_r" 

print("--- Plotting Parameters ---")
print(f"Temp Plot vmin/vmax: {vmin_temp:.2f} / {vmax_temp:.2f} °C")
print(f"Temp Cmap: {temp_cmap}")
print("Bias Plot normalization: Centered at 0, vmin=-0.5, vmax=0.5")
print(f"Bias Cmap: {bias_cmap}")
print("="*70 + "\n")

# -----------------------------
# 10. Create Plots
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(20, 6)) 
plt.subplots_adjust(wspace=0.1) 

# --- Improved plotting function ---
def plot_da(ax, da, cmap, panel_label, vmin=None, vmax=None, norm=None, title=""):
    """
    Plots a DataArray with publication-quality settings.
    """
    im = da.plot(ax=ax, cmap=cmap, vmin=vmin, vmax=vmax, norm=norm, add_colorbar=False)
    gdf.boundary.plot(ax=ax, edgecolor="black", linewidth=0.5) 
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title(title, fontsize=16) 
    
    ax.text(0.95, 0.95, panel_label, 
            transform=ax.transAxes, 
            fontsize=20, 
            fontweight='bold', 
            va='top', 
            ha='right',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
    
    return im

# MSWX Mean
im1 = plot_da(axes[0], overall_mswx_clipped, 
              cmap=temp_cmap, 
              vmin=vmin_temp, 
              vmax=vmax_temp, 
              title="MSWX Mean Test Period", 
              panel_label="(a)")

# DDPM Mean
im2 = plot_da(axes[1], overall_ddpm_clipped, 
              cmap=temp_cmap, 
              vmin=vmin_temp, 
              vmax=vmax_temp, 
              title="DDPM Mean Test Period", 
              panel_label="(b)")

# Bias
im3 = plot_da(axes[2], overall_bias_clipped, 
              cmap=bias_cmap, 
              norm=bias_norm, 
              title="Bias (DDPM − MSWX)", 
              panel_label="(c)")

# -----------------------------
# 11. Add Final Colorbars
# -----------------------------
# Horizontal for mean maps (lowered AND shifted left)
cbar_ax_h = fig.add_axes([0.05, 0.02, 0.5, 0.03])  # <--- Shifted left (0.25 to 0.23)
cbar_h = fig.colorbar(im2, cax=cbar_ax_h, orientation='horizontal', extend='both') # <--- Added extend
cbar_h.set_label("Temperature (°C)", fontsize=18) 
cbar_h.ax.tick_params(labelsize=12) 

# Vertical for bias
cbar_ax_v = fig.add_axes([0.89, 0.25, 0.015, 0.5]) 
cbar_v = fig.colorbar(im3, cax=cbar_ax_v, orientation='vertical', extend='both') # <--- Added extend
cbar_v.set_label("Bias (°C)", fontsize=18) 
cbar_v.ax.tick_params(labelsize=12) 

# Adjust layout to prevent overlap
plt.tight_layout(rect=[0, 0.05, 0.88, 1]) 

# -----------------------------
# 12. Save and Show
# -----------------------------
plt.show()

# Save the figure with high DPI
save_path = "<SET YOUR PATH HERE>"
fig.savefig(save_path, dpi=300, bbox_inches='tight')

print(f"✅ Plotting complete. Figure saved to {save_path}")

In [ ]:
#!/usr/bin/env python3
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import rioxarray
import geopandas as gpd
import matplotlib.colors as colors

# -----------------------------
# 1. Paths
# -----------------------------
hr_mswx = "<SET YOUR PATH HERE>"
hr_ddpm = "<SET YOUR PATH HERE>"
india_shp = "<SET YOUR PATH HERE>"
save_path = "<SET YOUR PATH HERE>"

# -----------------------------
# 2. Helper functions
# -----------------------------
def get_file_date(fname):
    """Extracts YYYYDOY from filename."""
    base = os.path.basename(fname)
    base = base.replace("_ddpm", "").replace(".nc", "")
    return int(base)

def doy_to_datetime(yydoy):
    """Converts YYYYDOY to datetime object."""
    year = int(str(yydoy)[:4])
    doy = int(str(yydoy)[4:])
    return pd.to_datetime(f"{year}-{doy:03d}", format="%Y-%j")

def filter_period(files, start=2009219, end=2014365):  # Testing period
    """Filters a list of files to be within the start/end YYYYDOY range."""
    valid = []
    for f in files:
        try:
            d = get_file_date(f)
            if start <= d <= end:
                valid.append(f)
        except Exception:
            continue
    return valid

# -----------------------------
# 3. Load files
# -----------------------------
f_mswx = sorted(glob.glob(os.path.join(hr_mswx, "*.nc")))
f_ddpm = sorted(glob.glob(os.path.join(hr_ddpm, "*_ddpm.nc")))

f_mswx = filter_period(f_mswx)
f_ddpm = filter_period(f_ddpm)

print(f"Found {len(f_mswx)} MSWX files and {len(f_ddpm)} DDPM files for the test period.")

# -----------------------------
# 4. Open datasets
# -----------------------------
ds_mswx = xr.open_mfdataset(
    f_mswx, combine="nested", concat_dim="time", engine="netcdf4"
)
ds_ddpm = xr.open_mfdataset(
    f_ddpm, combine="nested", concat_dim="time", engine="netcdf4"
)

# -----------------------------
# 5. Assign time coordinate
# -----------------------------
if "time" not in ds_mswx.coords:
    doy_list = [get_file_date(f) for f in f_mswx]
    times = [doy_to_datetime(d) for d in doy_list]
    ds_mswx = ds_mswx.assign_coords(time=("time", times))
    ds_ddpm = ds_ddpm.assign_coords(time=("time", times))

# -----------------------------
# 6. Calculate Annual & Overall Mean
# -----------------------------
da_m = ds_mswx["air_temperature"]
da_d = ds_ddpm["air_temperature"]

annual_mswx = da_m.groupby("time.year").mean("time")
annual_ddpm = da_d.groupby("time.year").mean("time")

overall_mswx = annual_mswx.mean("year")
overall_ddpm = annual_ddpm.mean("year")
overall_bias = overall_ddpm - overall_mswx

print("✅ Annual and overall means computed")

# -----------------------------
# 7. Clip Data to Shapefile
# -----------------------------
# Rename lat/lon to y/x
overall_mswx = overall_mswx.rename({"lat": "y", "lon": "x"})
overall_ddpm = overall_ddpm.rename({"lat": "y", "lon": "x"})
overall_bias = overall_bias.rename({"lat": "y", "lon": "x"})

# Set spatial dims and CRS
for da in [overall_mswx, overall_ddpm, overall_bias]:
    da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
    da.rio.write_crs("EPSG:4326", inplace=True)

# Load India shapefile
gdf = gpd.read_file(india_shp)

# Clip to India
overall_mswx_clipped = overall_mswx.rio.clip(gdf.geometry, gdf.crs)
overall_ddpm_clipped = overall_ddpm.rio.clip(gdf.geometry, gdf.crs)
overall_bias_clipped = overall_bias.rio.clip(gdf.geometry, gdf.crs)

print("✅ Data clipped to shapefile")

# -----------------------------
# 8. Compute time-based metrics: RMSE, MAE, correlation
# -----------------------------
print("✅ Computing RMSE, MAE, and correlation maps over the test period")

da_m_full = ds_mswx["air_temperature"]
da_d_full = ds_ddpm["air_temperature"]

# Bias time series
bias_ts = da_d_full - da_m_full

# RMSE and MAE over time (per grid cell)
rmse_map = np.sqrt((bias_ts ** 2).mean("time", skipna=True))
mae_map = np.abs(bias_ts).mean("time", skipna=True)

# Spatial (per-grid) Pearson correlation over time
corr_map = xr.corr(da_d_full, da_m_full, dim="time")

# Clip these maps to India as well
rmse_map = rmse_map.rename({"lat": "y", "lon": "x"})
rmse_map.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
rmse_map.rio.write_crs("EPSG:4326", inplace=True)
rmse_clipped = rmse_map.rio.clip(gdf.geometry, gdf.crs)

mae_map = mae_map.rename({"lat": "y", "lon": "x"})
mae_map.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
mae_map.rio.write_crs("EPSG:4326", inplace=True)
mae_clipped = mae_map.rio.clip(gdf.geometry, gdf.crs)

corr_map = corr_map.rename({"lat": "y", "lon": "x"})
corr_map.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
corr_map.rio.write_crs("EPSG:4326", inplace=True)
corr_clipped = corr_map.rio.clip(gdf.geometry, gdf.crs)

# ----------------------------------------------------
# 9. Calculate & Print Summary Statistics
# ----------------------------------------------------
print("\n" + "=" * 70)
print("     SUMMARY: Mean Spatial Statistics for Test Period")
print(f"     (Period: 2009-08-07 to 2014-12-31, n_days={len(f_mswx)})")
print("=" * 70 + "\n")

# --- MSWX (Target) Statistics ---
stats_mswx_mean = float(overall_mswx_clipped.mean().values)
stats_mswx_min = float(overall_mswx_clipped.min().values)
stats_mswx_max = float(overall_mswx_clipped.max().values)

print("--- MSWX (Target) ---")
print(f"Overall Spatial Mean: {stats_mswx_mean:.4f} °C")
print(f"Overall Spatial Min:  {stats_mswx_min:.4f} °C")
print(f"Overall Spatial Max:  {stats_mswx_max:.4f} °C")
print("\n")

# --- DDPM (Prediction) Statistics ---
stats_ddpm_mean = float(overall_ddpm_clipped.mean().values)
stats_ddpm_min = float(overall_ddpm_clipped.min().values)
stats_ddpm_max = float(overall_ddpm_clipped.max().values)

print("--- DDPM (Prediction) ---")
print(f"Overall Spatial Mean: {stats_ddpm_mean:.4f} °C")
print(f"Overall Spatial Min:  {stats_ddpm_min:.4f} °C")
print(f"Overall Spatial Max:  {stats_ddpm_max:.4f} °C")
print("\n")

# --- BIAS (DDPM - MSWX) Statistics ---
bias_values_clipped = overall_bias_clipped.values
stats_bias_mean = float(np.nanmean(bias_values_clipped))
stats_mab = float(np.nanmean(np.abs(bias_values_clipped)))
stats_rmse = float(np.sqrt(np.nanmean(np.square(bias_values_clipped))))
stats_bias_min = float(np.nanmin(bias_values_clipped))
stats_bias_max = float(np.nanmax(bias_values_clipped))

print("--- BIAS (DDPM - MSWX) ---")
print(f"Mean Spatial Bias (MB):   {stats_bias_mean:.4f} °C")
print(f"Mean Absolute Bias (MAB): {stats_mab:.4f} °C")
print(f"Spatial RMSE (mean field):{stats_rmse:.4f} °C")
print(f"Min Bias (Coldest):       {stats_bias_min:.4f} °C")
print(f"Max Bias (Warmest):       {stats_bias_max:.4f} °C")
print("\n")

# -----------------------------
# 10. Define Plotting Parameters
# -----------------------------
# Temperature range
vmin_temp = float(min(stats_mswx_min, stats_ddpm_min))
vmax_temp = float(max(stats_mswx_max, stats_ddpm_max))

# Bias normalization (centered at 0)
bias_norm = colors.TwoSlopeNorm(vcenter=0, vmin=-0.5, vmax=0.5)

# RMSE/MAE normalization (Using absolute MIN and MAX)
rmse_vals = rmse_clipped.values.flatten()
rmse_vals = rmse_vals[np.isfinite(rmse_vals)]
mae_vals = mae_clipped.values.flatten()
mae_vals = mae_vals[np.isfinite(mae_vals)]
err_vals_all = np.concatenate([rmse_vals, mae_vals]) if mae_vals.size > 0 else rmse_vals

if err_vals_all.size > 0:
    vmin_err = float(np.nanmin(err_vals_all))
    vmax_err = float(np.nanmax(err_vals_all))
else:
    vmin_err, vmax_err = 0.0, 1.0

# Correlation range based on data (Absolute MIN and MAX)
corr_vals = corr_clipped.values.flatten()
corr_vals = corr_vals[np.isfinite(corr_vals)]
if corr_vals.size > 0:
    vmin_corr = float(np.nanmin(corr_vals))
    vmax_corr = float(np.nanmax(corr_vals))
else:
    vmin_corr, vmax_corr = -1.0, 1.0

# Colormaps
temp_cmap = "Spectral_r"
bias_cmap = "RdBu_r"
err_cmap = "YlOrRd"
corr_cmap = "viridis"   # non-divergent for correlation

print("--- Plotting Parameters ---")
print(f"Temp Plot vmin/vmax: {vmin_temp:.2f} / {vmax_temp:.2f} °C")
print(f"Temp Cmap: {temp_cmap}")
print("Bias Plot normalization: Centered at 0, vmin=-0.5, vmax=0.5")
print(f"Bias Cmap: {bias_cmap}")
print(f"Error (RMSE/MAE) vmin/vmax: {vmin_err:.2f} / {vmax_err:.2f} °C")
print(f"Correlation vmin/vmax (Min-Max): {vmin_corr:.2f} / {vmax_corr:.2f}")
print("=" * 70 + "\n")

# -----------------------------
# 11. Create 2×3 Figure
# -----------------------------
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
plt.subplots_adjust(wspace=0.1, hspace=0.15)

def plot_da(ax, da, cmap, panel_label, vmin=None, vmax=None, norm=None):
    """
    Plots a DataArray with publication-quality settings (NO titles at all).
    """
    im = da.plot(
        ax=ax,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        norm=norm,
        add_colorbar=False,
        add_labels=False  # ✅ prevents xarray from adding any auto text
    )

    # ✅ HARD RESET of any leftover title
    ax.set_title("")

    # India boundary
    gdf.boundary.plot(ax=ax, edgecolor="black", linewidth=0.5)

    # Clean axes
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    # Panel label only (a, b, c...)
    ax.text(
        0.95,
        0.95,
        panel_label,
        transform=ax.transAxes,
        fontsize=28,  # Increased from 20 for legibility
        fontweight="bold",
        va="top",
        ha="right",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7),
    )

    return im

def annotate_mean(ax, text):
    """
    Place a mean-value box in the bottom-right corner of the panel.
    """
    ax.text(
        0.98,
        0.02,
        text,
        transform=ax.transAxes,
        fontsize=18,  # Increased from 11 for legibility
        fontweight="bold",
        va="bottom",
        ha="right",
        bbox=dict(boxstyle="round,pad=0.25", fc="white", alpha=0.85, edgecolor="black"),
    )

# Row 1: Means & Bias
im1 = plot_da(
    axes[0, 0],
    overall_mswx_clipped,
    cmap=temp_cmap,
    vmin=vmin_temp,
    vmax=vmax_temp,
    panel_label="(a)",
)

im2 = plot_da(
    axes[0, 1],
    overall_ddpm_clipped,
    cmap=temp_cmap,
    vmin=vmin_temp,
    vmax=vmax_temp,
    panel_label="(b)",
)

im3 = plot_da(
    axes[0, 2],
    overall_bias_clipped,
    cmap=bias_cmap,
    norm=bias_norm,
    panel_label="(c)",
)

# Row 2: RMSE, MAE, Correlation
im4 = plot_da(
    axes[1, 0],
    rmse_clipped,
    cmap=err_cmap,
    vmin=vmin_err,
    vmax=vmax_err,
    panel_label="(d)",
)

im5 = plot_da(
    axes[1, 1],
    mae_clipped,
    cmap=err_cmap,
    vmin=vmin_err,
    vmax=vmax_err,
    panel_label="(e)",
)

im6 = plot_da(
    axes[1, 2],
    corr_clipped,
    cmap=corr_cmap,
    vmin=vmin_corr,
    vmax=vmax_corr,
    panel_label="(f)",
)

# -----------------------------
# 12. Annotate mean values in each panel
# -----------------------------
mean_mswx = float(np.nanmean(overall_mswx_clipped.values))
mean_ddpm = float(np.nanmean(overall_ddpm_clipped.values))
mean_bias = float(np.nanmean(overall_bias_clipped.values))
mean_rmse_map = float(np.nanmean(rmse_clipped.values))
mean_mae_map = float(np.nanmean(mae_clipped.values))
mean_corr_map = float(np.nanmean(corr_clipped.values))

annotate_mean(axes[0, 0], f"Mean: {mean_mswx:.3f} °C")
annotate_mean(axes[0, 1], f"Mean: {mean_ddpm:.3f} °C")
annotate_mean(axes[0, 2], f"Mean Bias: {mean_bias:+.3f} °C")
annotate_mean(axes[1, 0], f"Mean RMSE: {mean_rmse_map:.3f} °C")
annotate_mean(axes[1, 1], f"Mean MAE: {mean_mae_map:.3f} °C")
annotate_mean(axes[1, 2], f"Mean r: {mean_corr_map:.3f}")

# First do tight_layout so axes positions are final before placing colorbars
plt.tight_layout(rect=[0.02, 0.07, 0.91, 0.98])

# -----------------------------
# 13. Colorbars with adjusted positions
# -----------------------------
# 13.1 Vertical temperature colorbar to the LEFT of MSWX (panel a)
box1 = axes[0, 0].get_position()   # MSWX panel position

cbar_width = 0.015
cbar_height = box1.y1 - box1.y0

# Place it slightly to the LEFT of MSWX
cbar_ax_temp = fig.add_axes(
    [box1.x0 - 0.035, box1.y0, cbar_width, cbar_height]
)

cbar_temp = fig.colorbar(im2, cax=cbar_ax_temp, orientation="vertical", extend="both")

# ✅ MOVE LABEL TO THE *LEFT SIDE* OF COLORBAR
# Increased fontsize from 16 to 24 for legibility
cbar_temp.set_label("Temperature (°C)", fontsize=24, labelpad=14)
cbar_temp.ax.yaxis.set_label_position("left")
cbar_temp.ax.yaxis.tick_left()

# Increased tick labelsize from 12 to 20 for legibility
cbar_temp.ax.tick_params(labelsize=20)

# 13.2 Horizontal error colorbar UNDER RMSE panel (bottom-left)
box_rmse = axes[1, 0].get_position()
cbar_height_err = 0.02
cbar_ax_err = fig.add_axes(
    [box_rmse.x0, box_rmse.y0 - 0.06, box_rmse.width, cbar_height_err]
)
cbar_err = fig.colorbar(im4, cax=cbar_ax_err, orientation="horizontal", extend="both")
# Increased fontsize from 16 to 24
cbar_err.set_label("Error (°C) [RMSE / MAE]", fontsize=24)
# Increased tick labelsize from 12 to 20
cbar_err.ax.tick_params(labelsize=20)

# 13.3 Vertical colorbar for Bias (right side, upper half)
cbar_ax_bias = fig.add_axes([0.93, 0.55, 0.015, 0.35])
cbar_bias = fig.colorbar(im3, cax=cbar_ax_bias, orientation="vertical", extend="both")
# Increased fontsize from 16 to 24
cbar_bias.set_label("Bias (°C)", fontsize=24)
# Increased tick labelsize from 12 to 20
cbar_bias.ax.tick_params(labelsize=20)

# 13.4 Vertical colorbar for Correlation (right side, lower half), non-divergent cmap
cbar_ax_corr = fig.add_axes([0.93, 0.10, 0.015, 0.35])
cbar_corr = fig.colorbar(im6, cax=cbar_ax_corr, orientation="vertical")
# Increased fontsize from 16 to 24
cbar_corr.set_label("Correlation (r)", fontsize=24)
# Increased tick labelsize from 12 to 20
cbar_corr.ax.tick_params(labelsize=20)

# -----------------------------
# 14. Save and Show
# -----------------------------
plt.show()
fig.savefig(save_path, dpi=300, bbox_inches="tight")
print(f"✅ Plotting complete. Figure saved to {save_path}")

In [ ]:
#!/usr/bin/env python3
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import rioxarray
import geopandas as gpd
import matplotlib.colors as colors

# -----------------------------
# 1. Paths
# -----------------------------
hr_mswx = "<SET YOUR PATH HERE>"
hr_ddpm = "<SET YOUR PATH HERE>"
india_shp = "<SET YOUR PATH HERE>"
save_path = "<SET YOUR PATH HERE>"

# -----------------------------
# 2. Helper functions
# -----------------------------
def get_file_date(fname):
    """Extracts YYYYDOY from filename."""
    base = os.path.basename(fname)
    # Removing the specific suffix to isolate the date
    base = base.replace("_ddpm_artifact_free", "").replace(".nc", "")
    return int(base)

def doy_to_datetime(yydoy):
    """Converts YYYYDOY to datetime object."""
    year = int(str(yydoy)[:4])
    doy = int(str(yydoy)[4:])
    return pd.to_datetime(f"{year}-{doy:03d}", format="%Y-%j")

def filter_period(files, start=2009219, end=2014365):  # Testing period
    """Filters a list of files to be within the start/end YYYYDOY range."""
    valid = []
    for f in files:
        try:
            d = get_file_date(f)
            if start <= d <= end:
                valid.append(f)
        except Exception:
            continue
    return valid

# -----------------------------
# 3. Load files
# -----------------------------
f_mswx = sorted(glob.glob(os.path.join(hr_mswx, "*.nc")))

# UPDATED: Changed glob pattern to match "*_ddpm_artifact_free.nc"
f_ddpm = sorted(glob.glob(os.path.join(hr_ddpm, "*_ddpm_artifact_free.nc")))

f_mswx = filter_period(f_mswx)
f_ddpm = filter_period(f_ddpm)

print(f"Found {len(f_mswx)} MSWX files and {len(f_ddpm)} DDPM files for the test period.")

# -----------------------------
# 4. Open datasets
# -----------------------------
ds_mswx = xr.open_mfdataset(
    f_mswx, combine="nested", concat_dim="time", engine="netcdf4"
)
ds_ddpm = xr.open_mfdataset(
    f_ddpm, combine="nested", concat_dim="time", engine="netcdf4"
)

# -----------------------------
# 5. Assign time coordinate
# -----------------------------
if "time" not in ds_mswx.coords:
    doy_list = [get_file_date(f) for f in f_mswx]
    times = [doy_to_datetime(d) for d in doy_list]
    ds_mswx = ds_mswx.assign_coords(time=("time", times))
    ds_ddpm = ds_ddpm.assign_coords(time=("time", times))

# -----------------------------
# 6. Calculate Annual & Overall Mean
# -----------------------------
da_m = ds_mswx["air_temperature"]
da_d = ds_ddpm["air_temperature"]

annual_mswx = da_m.groupby("time.year").mean("time")
annual_ddpm = da_d.groupby("time.year").mean("time")

overall_mswx = annual_mswx.mean("year")
overall_ddpm = annual_ddpm.mean("year")
overall_bias = overall_ddpm - overall_mswx

print("✅ Annual and overall means computed")

# -----------------------------
# 7. Clip Data to Shapefile
# -----------------------------
# Rename lat/lon to y/x
overall_mswx = overall_mswx.rename({"lat": "y", "lon": "x"})
overall_ddpm = overall_ddpm.rename({"lat": "y", "lon": "x"})
overall_bias = overall_bias.rename({"lat": "y", "lon": "x"})

# Set spatial dims and CRS
for da in [overall_mswx, overall_ddpm, overall_bias]:
    da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
    da.rio.write_crs("EPSG:4326", inplace=True)

# Load India shapefile
gdf = gpd.read_file(india_shp)

# Clip to India
overall_mswx_clipped = overall_mswx.rio.clip(gdf.geometry, gdf.crs)
overall_ddpm_clipped = overall_ddpm.rio.clip(gdf.geometry, gdf.crs)
overall_bias_clipped = overall_bias.rio.clip(gdf.geometry, gdf.crs)

print("✅ Data clipped to shapefile")

# -----------------------------
# 8. Compute time-based metrics: RMSE, MAE, correlation
# -----------------------------
print("✅ Computing RMSE, MAE, and correlation maps over the test period")

da_m_full = ds_mswx["air_temperature"]
da_d_full = ds_ddpm["air_temperature"]

# Bias time series
bias_ts = da_d_full - da_m_full

# RMSE and MAE over time (per grid cell)
rmse_map = np.sqrt((bias_ts ** 2).mean("time", skipna=True))
mae_map = np.abs(bias_ts).mean("time", skipna=True)

# Spatial (per-grid) Pearson correlation over time
corr_map = xr.corr(da_d_full, da_m_full, dim="time")

# Clip these maps to India as well
rmse_map = rmse_map.rename({"lat": "y", "lon": "x"})
rmse_map.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
rmse_map.rio.write_crs("EPSG:4326", inplace=True)
rmse_clipped = rmse_map.rio.clip(gdf.geometry, gdf.crs)

mae_map = mae_map.rename({"lat": "y", "lon": "x"})
mae_map.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
mae_map.rio.write_crs("EPSG:4326", inplace=True)
mae_clipped = mae_map.rio.clip(gdf.geometry, gdf.crs)

corr_map = corr_map.rename({"lat": "y", "lon": "x"})
corr_map.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
corr_map.rio.write_crs("EPSG:4326", inplace=True)
corr_clipped = corr_map.rio.clip(gdf.geometry, gdf.crs)

# ----------------------------------------------------
# 9. Calculate & Print Summary Statistics
# ----------------------------------------------------
print("\n" + "=" * 70)
print("     SUMMARY: Mean Spatial Statistics for Test Period")
print(f"     (Period: 2009-08-07 to 2014-12-31, n_days={len(f_mswx)})")
print("=" * 70 + "\n")

# --- MSWX (Target) Statistics ---
stats_mswx_mean = float(overall_mswx_clipped.mean().values)
stats_mswx_min = float(overall_mswx_clipped.min().values)
stats_mswx_max = float(overall_mswx_clipped.max().values)

print("--- MSWX (Target) ---")
print(f"Overall Spatial Mean: {stats_mswx_mean:.4f} °C")
print(f"Overall Spatial Min:  {stats_mswx_min:.4f} °C")
print(f"Overall Spatial Max:  {stats_mswx_max:.4f} °C")
print("\n")

# --- DDPM (Prediction) Statistics ---
stats_ddpm_mean = float(overall_ddpm_clipped.mean().values)
stats_ddpm_min = float(overall_ddpm_clipped.min().values)
stats_ddpm_max = float(overall_ddpm_clipped.max().values)

print("--- DDPM (Prediction) ---")
print(f"Overall Spatial Mean: {stats_ddpm_mean:.4f} °C")
print(f"Overall Spatial Min:  {stats_ddpm_min:.4f} °C")
print(f"Overall Spatial Max:  {stats_ddpm_max:.4f} °C")
print("\n")

# --- BIAS (DDPM - MSWX) Statistics ---
bias_values_clipped = overall_bias_clipped.values
stats_bias_mean = float(np.nanmean(bias_values_clipped))
stats_mab = float(np.nanmean(np.abs(bias_values_clipped)))
stats_rmse = float(np.sqrt(np.nanmean(np.square(bias_values_clipped))))
stats_bias_min = float(np.nanmin(bias_values_clipped))
stats_bias_max = float(np.nanmax(bias_values_clipped))

print("--- BIAS (DDPM - MSWX) ---")
print(f"Mean Spatial Bias (MB):   {stats_bias_mean:.4f} °C")
print(f"Mean Absolute Bias (MAB): {stats_mab:.4f} °C")
print(f"Spatial RMSE (mean field):{stats_rmse:.4f} °C")
print(f"Min Bias (Coldest):       {stats_bias_min:.4f} °C")
print(f"Max Bias (Warmest):       {stats_bias_max:.4f} °C")
print("\n")

# -----------------------------
# 10. Define Plotting Parameters
# -----------------------------
# Temperature range
vmin_temp = float(min(stats_mswx_min, stats_ddpm_min))
vmax_temp = float(max(stats_mswx_max, stats_ddpm_max))

# Bias normalization (centered at 0)
bias_norm = colors.TwoSlopeNorm(vcenter=0, vmin=-0.5, vmax=0.5)

# RMSE/MAE normalization
rmse_vals = rmse_clipped.values.flatten()
rmse_vals = rmse_vals[np.isfinite(rmse_vals)]
mae_vals = mae_clipped.values.flatten()
mae_vals = mae_vals[np.isfinite(mae_vals)]
err_vals_all = np.concatenate([rmse_vals, mae_vals]) if mae_vals.size > 0 else rmse_vals

if err_vals_all.size > 0:
    vmin_err = 0.0
    vmax_err = float(np.percentile(err_vals_all, 98.0))
    if vmax_err <= vmin_err:
        vmax_err = float(np.nanmax(err_vals_all))
else:
    vmin_err, vmax_err = 0.0, 1.0

# Correlation range based on data (1st–99th percentile)
corr_vals = corr_clipped.values.flatten()
corr_vals = corr_vals[np.isfinite(corr_vals)]
if corr_vals.size > 0:
    vmin_corr = float(np.percentile(corr_vals, 1.0))
    vmax_corr = float(np.percentile(corr_vals, 99.0))
    if vmax_corr <= vmin_corr:
        vmin_corr, vmax_corr = -1.0, 1.0
else:
    vmin_corr, vmax_corr = -1.0, 1.0

# Colormaps
temp_cmap = "Spectral_r"
bias_cmap = "RdBu_r"
err_cmap = "YlOrRd"
corr_cmap = "viridis"   # non-divergent for correlation

print("--- Plotting Parameters ---")
print(f"Temp Plot vmin/vmax: {vmin_temp:.2f} / {vmax_temp:.2f} °C")
print(f"Temp Cmap: {temp_cmap}")
print("Bias Plot normalization: Centered at 0, vmin=-0.5, vmax=0.5")
print(f"Bias Cmap: {bias_cmap}")
print(f"Error (RMSE/MAE) vmin/vmax: {vmin_err:.2f} / {vmax_err:.2f} °C")
print(f"Correlation vmin/vmax (1–99%): {vmin_corr:.2f} / {vmax_corr:.2f}")
print("=" * 70 + "\n")

# -----------------------------
# 11. Create 2×3 Figure
# -----------------------------
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
plt.subplots_adjust(wspace=0.1, hspace=0.15)

def plot_da(ax, da, cmap, panel_label, vmin=None, vmax=None, norm=None):
    """
    Plots a DataArray with publication-quality settings (NO titles at all).
    """
    im = da.plot(
        ax=ax,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        norm=norm,
        add_colorbar=False,
        add_labels=False  # ✅ prevents xarray from adding any auto text
    )

    # ✅ HARD RESET of any leftover title
    ax.set_title("")

    # India boundary
    gdf.boundary.plot(ax=ax, edgecolor="black", linewidth=0.5)

    # Clean axes
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    # Panel label only (a, b, c...)
    ax.text(
        0.95,
        0.95,
        panel_label,
        transform=ax.transAxes,
        fontsize=20,
        fontweight="bold",
        va="top",
        ha="right",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7),
    )

    return im

def annotate_mean(ax, text):
    """
    Place a mean-value box in the bottom-right corner of the panel.
    """
    ax.text(
        0.98,
        0.02,
        text,
        transform=ax.transAxes,
        fontsize=11,
        fontweight="bold",
        va="bottom",
        ha="right",
        bbox=dict(boxstyle="round,pad=0.25", fc="white", alpha=0.85, edgecolor="black"),
    )

# Row 1: Means & Bias
im1 = plot_da(
    axes[0, 0],
    overall_mswx_clipped,
    cmap=temp_cmap,
    vmin=vmin_temp,
    vmax=vmax_temp,
    panel_label="(a)",
)

im2 = plot_da(
    axes[0, 1],
    overall_ddpm_clipped,
    cmap=temp_cmap,
    vmin=vmin_temp,
    vmax=vmax_temp,
    panel_label="(b)",
)

im3 = plot_da(
    axes[0, 2],
    overall_bias_clipped,
    cmap=bias_cmap,
    norm=bias_norm,
    panel_label="(c)",
)

# Row 2: RMSE, MAE, Correlation
im4 = plot_da(
    axes[1, 0],
    rmse_clipped,
    cmap=err_cmap,
    vmin=vmin_err,
    vmax=vmax_err,
    panel_label="(d)",
)

im5 = plot_da(
    axes[1, 1],
    mae_clipped,
    cmap=err_cmap,
    vmin=vmin_err,
    vmax=vmax_err,
    panel_label="(e)",
)

im6 = plot_da(
    axes[1, 2],
    corr_clipped,
    cmap=corr_cmap,
    vmin=vmin_corr,
    vmax=vmax_corr,
    panel_label="(f)",
)

# -----------------------------
# 12. Annotate mean values in each panel
# -----------------------------
mean_mswx = float(np.nanmean(overall_mswx_clipped.values))
mean_ddpm = float(np.nanmean(overall_ddpm_clipped.values))
mean_bias = float(np.nanmean(overall_bias_clipped.values))
mean_rmse_map = float(np.nanmean(rmse_clipped.values))
mean_mae_map = float(np.nanmean(mae_clipped.values))
mean_corr_map = float(np.nanmean(corr_clipped.values))

annotate_mean(axes[0, 0], f"Mean: {mean_mswx:.3f} °C")
annotate_mean(axes[0, 1], f"Mean: {mean_ddpm:.3f} °C")
annotate_mean(axes[0, 2], f"Mean Bias: {mean_bias:+.3f} °C")
annotate_mean(axes[1, 0], f"Mean RMSE: {mean_rmse_map:.3f} °C")
annotate_mean(axes[1, 1], f"Mean MAE: {mean_mae_map:.3f} °C")
annotate_mean(axes[1, 2], f"Mean r: {mean_corr_map:.3f}")

# First do tight_layout so axes positions are final before placing colorbars
plt.tight_layout(rect=[0.02, 0.07, 0.91, 0.98])

# -----------------------------
# 13. Colorbars with adjusted positions
# -----------------------------
# 13.1 Vertical temperature colorbar to the LEFT of MSWX (panel a)
box1 = axes[0, 0].get_position()   # MSWX panel position

cbar_width = 0.015
cbar_height = box1.y1 - box1.y0

# Place it slightly to the LEFT of MSWX
cbar_ax_temp = fig.add_axes(
    [box1.x0 - 0.035, box1.y0, cbar_width, cbar_height]
)

cbar_temp = fig.colorbar(im2, cax=cbar_ax_temp, orientation="vertical", extend="both")

# ✅ MOVE LABEL TO THE *LEFT SIDE* OF COLORBAR
cbar_temp.set_label("Temperature (°C)", fontsize=16, labelpad=14)
cbar_temp.ax.yaxis.set_label_position("left")
cbar_temp.ax.yaxis.tick_left()

cbar_temp.ax.tick_params(labelsize=12)

# 13.2 Horizontal error colorbar UNDER RMSE panel (bottom-left)
box_rmse = axes[1, 0].get_position()
cbar_height_err = 0.02
cbar_ax_err = fig.add_axes(
    [box_rmse.x0, box_rmse.y0 - 0.06, box_rmse.width, cbar_height_err]
)
cbar_err = fig.colorbar(im4, cax=cbar_ax_err, orientation="horizontal", extend="both")
cbar_err.set_label("Error (°C) [RMSE / MAE]", fontsize=16)
cbar_err.ax.tick_params(labelsize=12)

# 13.3 Vertical colorbar for Bias (right side, upper half)
cbar_ax_bias = fig.add_axes([0.93, 0.55, 0.015, 0.35])
cbar_bias = fig.colorbar(im3, cax=cbar_ax_bias, orientation="vertical", extend="both")
cbar_bias.set_label("Bias (°C)", fontsize=16)
cbar_bias.ax.tick_params(labelsize=12)

# 13.4 Vertical colorbar for Correlation (right side, lower half), non-divergent cmap
cbar_ax_corr = fig.add_axes([0.93, 0.10, 0.015, 0.35])
cbar_corr = fig.colorbar(im6, cax=cbar_ax_corr, orientation="vertical")
cbar_corr.set_label("Correlation (r)", fontsize=16)
cbar_corr.ax.tick_params(labelsize=12)

# -----------------------------
# 14. Save and Show
# -----------------------------
plt.show()
# fig.savefig(save_path, dpi=300, bbox_inches="tight")
# print(f"✅ Plotting complete. Figure saved to {save_path}")

In [ ]:
#!/usr/bin/env python3
"""
Pointwise Correlation Scatter (Same Test Period)
Location: Lon = 77, Lat = 10
Period: 2009-08-07 to 2014-12-31
"""

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
from scipy.stats import pearsonr

# ==========================
# PATHS
# ==========================
HR_MSWX = "<SET YOUR PATH HERE>"
HR_DDPM = "<SET YOUR PATH HERE>"
OUT_PNG = "<SET YOUR PATH HERE>"

TARGET_LON = 77.0
TARGET_LAT = 10.0

TEST_START = 2009219   # YYYYDOY
TEST_END   = 2014365

# ==========================
# HELPERS
# ==========================
def get_file_date(fname):
    base = os.path.basename(fname)
    base = base.replace("_ddpm", "").replace(".nc", "")
    return int(base)

def doy_to_datetime(yydoy):
    year = int(str(yydoy)[:4])
    doy = int(str(yydoy)[4:])
    return pd.to_datetime(f"{year}-{doy:03d}", format="%Y-%j")

def filter_period(files, start, end):
    valid = []
    for f in files:
        try:
            d = get_file_date(f)
            if start <= d <= end:
                valid.append(f)
        except:
            continue
    return valid

# ==========================
# LOAD & FILTER FILES
# ==========================
mswx_files = sorted(glob.glob(os.path.join(HR_MSWX, "*.nc")))
ddpm_files = sorted(glob.glob(os.path.join(HR_DDPM, "*_ddpm.nc")))

mswx_files = filter_period(mswx_files, TEST_START, TEST_END)
ddpm_files = filter_period(ddpm_files, TEST_START, TEST_END)

if len(mswx_files) == 0 or len(ddpm_files) == 0:
    raise RuntimeError("No files found for the test period.")

print(f"Using {len(mswx_files)} days from 2009–2014")

# ==========================
# OPEN DATASETS
# ==========================
ds_mswx = xr.open_mfdataset(mswx_files, combine="nested", concat_dim="time", engine="netcdf4")
ds_ddpm = xr.open_mfdataset(ddpm_files, combine="nested", concat_dim="time", engine="netcdf4")

# ==========================
# ASSIGN TIME COORDINATE
# ==========================
times = [doy_to_datetime(get_file_date(f)) for f in mswx_files]
ds_mswx = ds_mswx.assign_coords(time=("time", times))
ds_ddpm = ds_ddpm.assign_coords(time=("time", times))

# ==========================
# VARIABLE DETECTION
# ==========================
var_m = "air_temperature" if "air_temperature" in ds_mswx.data_vars else list(ds_mswx.data_vars)[0]
var_d = "air_temperature" if "air_temperature" in ds_ddpm.data_vars else list(ds_ddpm.data_vars)[0]

# ==========================
# EXTRACT POINT TIMESERIES
# ==========================
ts_mswx = ds_mswx[var_m].sel(lon=TARGET_LON, lat=TARGET_LAT, method="nearest")
ts_ddpm = ds_ddpm[var_d].sel(lon=TARGET_LON, lat=TARGET_LAT, method="nearest")

# Remove NaNs
mask = np.isfinite(ts_mswx.values) & np.isfinite(ts_ddpm.values)
mswx_vals = ts_mswx.values[mask]
ddpm_vals = ts_ddpm.values[mask]

# ==========================
# CORRELATION
# ==========================
r, p = pearsonr(mswx_vals, ddpm_vals)

print("\n================= POINTWISE CORRELATION (TEST PERIOD) =================")
print(f"Location : Lon = {TARGET_LON} , Lat = {TARGET_LAT}")
print("Period   : 2009-08-07 to 2014-12-31")
print(f"Samples  : {len(mswx_vals)}")
print(f"Pearson r: {r:.5f}")
print(f"P-value  : {p:.3e}")
print("=======================================================================")

# ==========================
# SCATTER PLOT
# ==========================
plt.figure(figsize=(7, 7))

plt.scatter(mswx_vals, ddpm_vals, s=15, alpha=0.6, edgecolor="k")

# 1:1 line
min_val = min(mswx_vals.min(), ddpm_vals.min())
max_val = max(mswx_vals.max(), ddpm_vals.max())
plt.plot([min_val, max_val], [min_val, max_val], "r--", lw=2, label="1:1 Reference")

# Labels
plt.xlabel("MSWX Temperature (°C)", fontsize=14, fontweight="bold")
plt.ylabel("DDPM Temperature (°C)", fontsize=14, fontweight="bold")

# Correlation annotation
plt.text(
    0.02, 0.98,
    f"r = {r:.4f}\np = {p:.2e}\nN = {len(mswx_vals)}",
    transform=plt.gca().transAxes,
    fontsize=13,
    ha="left",
    va="top",
    bbox=dict(facecolor="white", edgecolor="black", alpha=0.9)
)

plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(OUT_PNG, dpi=300, bbox_inches="tight")
plt.show()

print(f"\n✅ Scatter saved to: {OUT_PNG}")

In [ ]:
import xarray as xr

def describe(ds, name="Dataset"):
    print("\n" + "="*70)
    print(f"🔍 {name}")
    print("="*70)

    # Basic summary
    print("\n📌 Summary:")
    print(ds)

    # Dimensions
    print("\n📏 Dimensions:")
    for dim, size in ds.dims.items():
        print(f"  - {dim}: {size}")

    # Coordinates
    print("\n🧭 Coordinates:")
    for c in ds.coords:
        print(f"  - {c}: shape={ds[c].shape}, dtype={ds[c].dtype}")

    # Variables
    print("\n📂 Data Variables:")
    for v in ds.data_vars:
        var = ds[v]
        print(f"  - {v}: shape={var.shape}, dtype={var.dtype}")

    # Attributes (optional – only if present)
    if ds.attrs:
        print("\n📝 Global Attributes:")
        for k, v in ds.attrs.items():
            print(f"  - {k}: {v}")
    else:
        print("\n📝 Global Attributes: None")

    print("\n")

ds_resunet = xr.open_dataset("<SET YOUR PATH HERE>")
ds_ddpm    = xr.open_dataset("<SET YOUR PATH HERE>")

describe(ds_resunet, name="ResUNet Downscaled Output")
describe(ds_ddpm, name="DDPM Downscaled Output")

In [ ]:
#!/usr/bin/env python3
"""
DDPM plotting — shared color ranges + two horizontal colorbars beneath the last (Himalayan) row.

 - Single uniform temperature colormap (cmcrameri.cm.lapaz) and single shared Normalize across all cases.
 - Single shared error Normalize across all cases.
 - Only two colorbars (both horizontal) placed beneath the Himalayan Transect (last row):
     * temperature cbar beneath the Coarsened Input plot (axes[-1,1])
     * error cbar beneath the Absolute Error plot (axes[-1,3])
 - Contours at 4°C intervals for every case (A/B/C).
 - Metrics:
    * Col 3 (DDPM): SPC (Bottom-Left), SSIM (Bottom-Right).
    * Col 4 (Error): RMSE & MAE (Vertical text on the right side).
"""
import os
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle
import matplotlib.ticker as mticker
import cmcrameri.cm as cmc

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.mpl.geoaxes

from scipy.stats import pearsonr
from scipy.interpolate import griddata
from skimage.metrics import structural_similarity as ssim
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# -------------------------
# USER CONFIG
# -------------------------
hr_mswx_dir = "<SET YOUR PATH HERE>"
hr_ddpm_dir = "<SET YOUR PATH HERE>"
coarse_input_dir = "<SET YOUR PATH HERE>"
india_shp = "<SET YOUR PATH HERE>"
output_file = "<SET YOUR PATH HERE>"
metrics_csv = "<SET YOUR PATH HERE>"

VARNAMES = {"MSWX": "air_temperature", "COARSE": "air_temperature", "DDPM": "air_temperature"}

# single uniform temperature colormap for all cases (cmcrameri)
TEMP_CMAP = cmc.vik       # change if you prefer another cmc.* map
CMAP_ERR = 'Reds'           # absolute error colormap

PROJECTION = ccrs.PlateCarree()

# UPDATED CASE STUDIES
CASE_STUDIES = [
    {'name': 'Hot Day',             'type': 'day',  'date': '2010-06-20', 'extent': [76, 88, 20, 30], 'inset_extent': [83, 84, 28, 29], 'clim': (30, 50)},
    {'name': 'Cold Day',            'type': 'day',  'date': '2008-01-21', 'extent': [76, 88, 20, 30], 'inset_extent': [83, 84, 28, 29], 'clim': (0, 25)},
    {'name': 'Monsoon (July Avg)', 'type': 'mean', 'year': 2014, 'month': 7, 'extent': [76, 88, 20, 30], 'inset_extent': [83, 84, 28, 29], 'clim': (20, 35)},
    {'name': 'Himalayan Transect', 'type': 'day',  'date': '2014-07-15', 'extent': [76, 88, 28, 36], 'inset_extent': [83, 84, 28, 29], 'clim': (0, 30)}
]

# -------------------------
# Helpers
# -------------------------
def ensure_lonlat_names(da):
    try:
        if hasattr(da, 'coords'):
            if 'x' in da.coords and 'lon' not in da.coords:
                da = da.rename({'x': 'lon'})
            if 'y' in da.coords and 'lat' not in da.coords:
                da = da.rename({'y': 'lat'})
            if 'longitude' in da.coords and 'lon' not in da.coords:
                da = da.rename({'longitude': 'lon'})
            if 'latitude' in da.coords and 'lat' not in da.coords:
                da = da.rename({'latitude': 'lat'})
    except Exception:
        pass
    return da

def safe_print_coords(da, name):
    try:
        lon = None; lat = None
        if 'lon' in da.coords:
            lon = da['lon']
        elif 'x' in da.coords:
            lon = da['x']
        if 'lat' in da.coords:
            lat = da['lat']
        elif 'y' in da.coords:
            lat = da['y']
        if lon is not None and lat is not None:
            print(f"  {name} lon min/max: {float(lon.min()):.4f} / {float(lon.max()):.4f}; lat min/max: {float(lat.min()):.4f} / {float(lat.max()):.4f}")
    except Exception:
        pass

def month_end(year, month):
    start = pd.Timestamp(year=year, month=month, day=1)
    return start + pd.offsets.MonthEnd(0)

def maybe_to_celsius(da):
    try:
        meanv = float(da.mean().values)
        if meanv > 200:
            return da - 273.15
    except Exception:
        pass
    return da

def sel_lonlat_slice(da, lon_min, lon_max, lat_min, lat_max):
    da = ensure_lonlat_names(da)
    try:
        lon = da['lon'] if 'lon' in da.coords else (da['x'] if 'x' in da.coords else None)
        lat = da['lat'] if 'lat' in da.coords else (da['y'] if 'y' in da.coords else None)
        if lon is not None and lat is not None and getattr(lon, 'ndim', 1) == 1 and getattr(lat, 'ndim', 1) == 1:
            try:
                # handle descending lat
                if float(lat[0]) > float(lat[-1]):
                    lat_slice = slice(lat_max, lat_min)
                else:
                    lat_slice = slice(lat_min, lat_max)
                return da.sel(lon=slice(lon_min, lon_max), lat=lat_slice)
            except Exception:
                return da.sel(lon=slice(lon_min, lon_max), lat=slice(lat_min, lat_max))
        else:
            try:
                lon2 = da['lon']; lat2 = da['lat']
                mask = (lon2 >= lon_min) & (lon2 <= lon_max) & (lat2 >= lat_min) & (lat2 <= lat_max)
                return da.where(mask, drop=True)
            except Exception:
                return da
    except Exception:
        return da

def round_down_4(x):
    return np.floor(x / 4.0) * 4.0

def round_up_4(x):
    return np.ceil(x / 4.0) * 4.0

# metrics (Updated to include MAE)
def calculate_metrics(da_true, da_pred, extent, debug=False):
    try:
        true_clip = sel_lonlat_slice(da_true, extent[0], extent[1], extent[2], extent[3])
        pred_clip = sel_lonlat_slice(da_pred, extent[0], extent[1], extent[2], extent[3])
        if getattr(true_clip, 'size', 0) == 0 or getattr(pred_clip, 'size', 0) == 0:
            if debug:
                print(f"  DEBUG: empty clip (true {getattr(true_clip,'size',0)}, pred {getattr(pred_clip,'size',0)})")
            return np.nan, np.nan, np.nan, np.nan

        if 'time' in true_clip.dims:
            true_2d = true_clip.mean(dim='time', skipna=True)
        else:
            true_2d = true_clip
        if 'time' in pred_clip.dims:
            pred_2d = pred_clip.mean(dim='time', skipna=True)
        else:
            pred_2d = pred_clip

        true_2d = xr.DataArray(true_2d) if not isinstance(true_2d, xr.DataArray) else true_2d
        pred_2d = xr.DataArray(pred_2d) if not isinstance(pred_2d, xr.DataArray) else pred_2d

        try:
            pred_interp = pred_2d.interp_like(true_2d, method='linear')
        except Exception:
            pred_interp = pred_2d

        if pred_interp.shape != true_2d.shape:
             try:
                if 'lon' in true_2d.coords and 'lat' in true_2d.coords:
                    lon_true = true_2d['lon'].values; lat_true = true_2d['lat'].values
                elif 'x' in true_2d.coords and 'y' in true_2d.coords:
                    lon_true = true_2d['x'].values; lat_true = true_2d['y'].values
                else:
                    lon_true = lat_true = None
                if lon_true is not None and lat_true is not None:
                    try:
                        if 'lon' in pred_2d.coords: pred_interp = pred_2d.interp(lon=lon_true, lat=lat_true, method='linear')
                        elif 'x' in pred_2d.coords: pred_interp = pred_2d.interp(x=lon_true, y=lat_true, method='linear')
                    except: pass
             except: pass
             
             if pred_interp.shape != true_2d.shape:
                 try:
                     pass
                 except:
                     pred_interp = pred_2d

        tv = np.array(true_2d.values, dtype=float)
        pv = np.array(pred_interp.values, dtype=float)

        if tv.shape != pv.shape:
            min0 = min(tv.shape[0], pv.shape[0])
            min1 = min(tv.shape[1], pv.shape[1])
            tv = tv[:min0, :min1]
            pv = pv[:min0, :min1]

        mask = np.isfinite(tv) & np.isfinite(pv)
        if mask.sum() < 4:
            return np.nan, np.nan, np.nan, np.nan

        tvals = tv[mask]; pvals = pv[mask]
        
        # --- METRICS CALCULATION ---
        rmse = float(np.sqrt(np.mean((tvals - pvals) ** 2)))
        mae = float(np.mean(np.abs(tvals - pvals))) 

        spc = np.nan
        try:
            if tvals.size >= 2 and np.nanstd(tvals) > 0 and np.nanstd(pvals) > 0:
                spc, _ = pearsonr(tvals, pvals)
        except Exception:
            spc = np.nan

        tv_filled = np.where(np.isfinite(tv), tv, np.nanmean(tv))
        pv_filled = np.where(np.isfinite(pv), pv, np.nanmean(pv))
        ssim_score = np.nan
        if tv_filled.ndim == 2 and pv_filled.ndim == 2 and tv_filled.shape[0] >= 2 and tv_filled.shape[1] >= 2:
            minv = float(min(np.nanmin(tv_filled), np.nanmin(pv_filled)))
            maxv = float(max(np.nanmax(tv_filled), np.nanmax(pv_filled)))
            data_range = maxv - minv
            if data_range <= 0:
                ssim_score = 1.0 if np.allclose(tv_filled, pv_filled, equal_nan=True) else 0.0
            else:
                try:
                    ssim_score = float(ssim(tv_filled, pv_filled, data_range=data_range))
                except Exception:
                    ssim_score = np.nan

        return rmse, spc, ssim_score, mae

    except Exception as e:
        if debug:
            print("DEBUG calculate_metrics exception:", e)
        return np.nan, np.nan, np.nan, np.nan

# plotting helper
def plot_panel(ax, data, case, cmap, cmap_norm, is_error=False, metrics=None, gdf_india=None):
    extent = case['extent']
    ax.set_extent(extent, crs=PROJECTION)
    data = ensure_lonlat_names(data)

    im = None
    try:
        cmap_obj = cmap if not isinstance(cmap, str) else mpl.cm.get_cmap(cmap)
        im = data.plot(ax=ax, cmap=cmap_obj, norm=cmap_norm, transform=PROJECTION,
                       add_colorbar=False, add_labels=False, extend='both')
        try:
            ax.set_title('')
        except Exception:
            pass
    except Exception:
        try:
            arr = data.values
            if 'lon' in data.coords and 'lat' in data.coords and arr.ndim == 2:
                X, Y = np.meshgrid(data['lon'].values, data['lat'].values)
                cmap_obj = cmap if not isinstance(cmap, str) else mpl.cm.get_cmap(cmap)
                im = ax.pcolormesh(X, Y, arr, cmap=cmap_obj, norm=cmap_norm, transform=PROJECTION)
            try:
                ax.set_title('')
            except Exception:
                pass
        except Exception:
            print("  --- WARNING: couldn't draw panel")

    ax.set_aspect('equal')

    inset_ex = case.get('inset_extent', None)
    if inset_ex is not None and not is_error:
        try:
            inset_ax = inset_axes(ax, width="30%", height="30%", loc='upper right',
                                  axes_class=cartopy.mpl.geoaxes.GeoAxes,
                                  axes_kwargs={'projection': PROJECTION})
            inset_ax.set_extent(inset_ex, crs=PROJECTION)
            try:
                cmap_obj = cmap if not isinstance(cmap, str) else mpl.cm.get_cmap(cmap)
                data.plot(ax=inset_ax, cmap=cmap_obj, norm=cmap_norm, transform=PROJECTION, add_colorbar=False, add_labels=False)
            except Exception:
                arr = data.values
                if 'lon' in data.coords and 'lat' in data.coords and arr.ndim == 2:
                    X, Y = np.meshgrid(data['lon'].values, data['lat'].values)
                    cmap_obj = cmap if not isinstance(cmap, str) else mpl.cm.get_cmap(cmap)
                    inset_ax.pcolormesh(X, Y, arr, cmap=cmap_obj, norm=cmap_norm, transform=PROJECTION)
            inset_ax.add_feature(cfeature.BORDERS.with_scale('10m'), edgecolor='gray', linestyle='--', linewidth=0.5)
            try:
                inset_ax.set_xticks([]); inset_ax.set_yticks([])
                inset_ax.set_xlabel(""); inset_ax.set_ylabel("")
                inset_ax.xaxis.set_ticklabels([]); inset_ax.yaxis.set_ticklabels([])
                inset_ax.xaxis.set_visible(False); inset_ax.yaxis.set_visible(False)
            except Exception:
                pass
        except Exception:
            pass

    return im

# -------------------------
# MAIN
# -------------------------
def main():
    print("Starting shared-range plotting with two horizontal colorbars underneath Himalayan transect...")
    nrows = len(CASE_STUDIES)
    
    # -------------------------------------------------------------
    # LAYOUT ADJUSTMENT: Reduced Height to close vertical gaps
    # -------------------------------------------------------------
    # Reduced height from 18 to 12. Cartopy/GeoAxes enforce aspect ratio;
    # if the figure is too tall, it inserts white space between rows.
    # Reducing height forces them closer.
    
    # CHANGED: bottom=0.13 to allow space for colorbars
    fig, axes = plt.subplots(nrows, 4, figsize=(16, 14), subplot_kw={'projection': PROJECTION})
    
    # Tight spacing settings
    # hspace=0.01 significantly reduces gap between rows
    # wspace=0.05 keeps columns tight
    fig.subplots_adjust(left=0.13, right=0.92, bottom=0.13, top=0.96, hspace=0.02, wspace=0.05)

    # load India shapefile
    try:
        gdf_india = gpd.read_file(india_shp)
    except Exception:
        gdf_india = None

    # ---------- FIRST PASS: load all arrays (lightweight) and compute global color limits ----------
    loaded = []
    all_temp_vals = []
    all_err_vals = []
    for case in CASE_STUDIES:
        da_mswx = da_coarse = da_ddpm = None
        try:
            def standardize(da):
                return ensure_lonlat_names(da)

            if case['type'] == 'day':
                dt = pd.to_datetime(case['date'])
                yydoy = int(f"{dt.year}{dt.dayofyear:03d}")
                f_mswx = os.path.join(hr_mswx_dir, f"{yydoy}.nc")
                f_coarse = os.path.join(coarse_input_dir, f"{yydoy}_1deg.nc")
                f_ddpm = os.path.join(hr_ddpm_dir, f"{yydoy}_ddpm.nc")
                missing = [p for p in (f_mswx, f_coarse, f_ddpm) if not os.path.exists(p)]
                if missing:
                    print(f"  --- ERROR: missing file(s) for {case['name']}: {missing}")
                    loaded.append({'case': case, 'da_mswx': None, 'da_coarse': None, 'da_ddpm': None, 'da_error': None})
                    continue
                ds_m = xr.open_dataset(f_mswx); ds_c = xr.open_dataset(f_coarse); ds_d = xr.open_dataset(f_ddpm)
                var_m = VARNAMES.get('MSWX') or list(ds_m.data_vars)[0]
                var_c = VARNAMES.get('COARSE') or list(ds_c.data_vars)[0]
                var_d = VARNAMES.get('DDPM') or list(ds_d.data_vars)[0]
                da_mswx = standardize(ds_m[var_m])
                da_coarse = standardize(ds_c[var_c])
                da_ddpm = standardize(ds_d[var_d])

            elif case['type'] == 'mean':
                year = case['year']; month = case['month']
                start = pd.Timestamp(year=year, month=month, day=1)
                end = month_end(year, month)
                days = pd.date_range(start=start, end=end, freq='D')
                doys = [int(f"{d.year}{d.dayofyear:03d}") for d in days]
                f_mswx_files = [os.path.join(hr_mswx_dir, f"{d}.nc") for d in doys if os.path.exists(os.path.join(hr_mswx_dir, f"{d}.nc"))]
                f_coarse_files = [os.path.join(coarse_input_dir, f"{d}_1deg.nc") for d in doys if os.path.exists(os.path.join(coarse_input_dir, f"{d}_1deg.nc"))]
                f_ddpm_files = [os.path.join(hr_ddpm_dir, f"{d}_ddpm.nc") for d in doys if os.path.exists(os.path.join(hr_ddpm_dir, f"{d}_ddpm.nc"))]
                if not (f_mswx_files and f_coarse_files and f_ddpm_files):
                    print(f"  --- ERROR: missing files for month mean {case['name']}. Found: MSWX {len(f_mswx_files)}, coarse {len(f_coarse_files)}, ddpm {len(f_ddpm_files)}")
                    loaded.append({'case': case, 'da_mswx': None, 'da_coarse': None, 'da_ddpm': None, 'da_error': None})
                    continue
                ds_m = xr.open_mfdataset(f_mswx_files, combine='by_coords'); ds_c = xr.open_mfdataset(f_coarse_files, combine='by_coords'); ds_d = xr.open_mfdataset(f_ddpm_files, combine='by_coords')
                var_m = VARNAMES.get('MSWX') or list(ds_m.data_vars)[0]
                var_c = VARNAMES.get('COARSE') or list(ds_c.data_vars)[0]
                var_d = VARNAMES.get('DDPM') or list(ds_d.data_vars)[0]
                da_mswx = standardize(ds_m[var_m].mean(dim='time', skipna=True)).compute()
                da_coarse = standardize(ds_c[var_c].mean(dim='time', skipna=True)).compute()
                da_ddpm = standardize(ds_d[var_d].mean(dim='time', skipna=True)).compute()

            if da_mswx is None or da_coarse is None or da_ddpm is None:
                loaded.append({'case': case, 'da_mswx': None, 'da_coarse': None, 'da_ddpm': None, 'da_error': None})
                continue

            da_mswx = maybe_to_celsius(da_mswx)
            da_coarse = maybe_to_celsius(da_coarse)
            da_ddpm = maybe_to_celsius(da_ddpm)
            da_error = np.abs(da_mswx - da_ddpm)

            # gather values for global color limits
            try:
                all_temp_vals.append(da_mswx.values.ravel())
                all_temp_vals.append(da_coarse.values.ravel())
                all_temp_vals.append(da_ddpm.values.ravel())
            except Exception:
                pass
            try:
                all_err_vals.append(da_error.values.ravel())
            except Exception:
                pass

            loaded.append({'case': case, 'da_mswx': da_mswx, 'da_coarse': da_coarse, 'da_ddpm': da_ddpm, 'da_error': da_error})
            safe_print_coords(da_mswx, case['name'] + " MSWX")
        except Exception as e:
            print(f"  --- ERROR loading data for {case['name']} in first pass: {e}")
            loaded.append({'case': case, 'da_mswx': None, 'da_coarse': None, 'da_ddpm': None, 'da_error': None})

    # concatenate and compute robust global limits
    if len(all_temp_vals) > 0:
        tmp = np.concatenate([v.ravel() for v in all_temp_vals if v is not None])
        tmp = tmp[np.isfinite(tmp)]
        if tmp.size > 0:
            global_temp_vmin = float(np.nanpercentile(tmp, 2))
            global_temp_vmax = float(np.nanpercentile(tmp, 98))
            if global_temp_vmin >= global_temp_vmax:
                global_temp_vmin, global_temp_vmax = float(np.nanmin(tmp)), float(np.nanmax(tmp))
        else:
            global_temp_vmin, global_temp_vmax = -10.0, 40.0
    else:
        global_temp_vmin, global_temp_vmax = -10.0, 40.0

    if len(all_err_vals) > 0:
        err = np.concatenate([v.ravel() for v in all_err_vals if v is not None])
        err = err[np.isfinite(err)]
        if err.size > 0:
            global_err_vmax = float(np.nanpercentile(err, 98))
            if global_err_vmax <= 0:
                global_err_vmax = float(np.nanmax(err)) if err.size>0 else 1.0
        else:
            global_err_vmax = 1.0
    else:
        global_err_vmax = 1.0

    # build shared norms
    norm_temp_shared = mcolors.Normalize(vmin=global_temp_vmin, vmax=global_temp_vmax)
    norm_err_shared = mcolors.Normalize(vmin=0.0, vmax=global_err_vmax)

    # ---------- SECOND PASS: plotting using loaded content and shared norms ----------
    metrics_rows = []
    for i, entry in enumerate(loaded):
        case = entry['case']
        da_mswx = entry['da_mswx']; da_coarse = entry['da_coarse']; da_ddpm = entry['da_ddpm']; da_error = entry['da_error']

        if case['type'] == 'day':
            date_label = pd.to_datetime(case['date']).strftime("%Y-%m-%d")
        else:
            date_label = f"{case.get('year')}-{case.get('month'):02d}"

        if da_mswx is None or da_coarse is None or da_ddpm is None:
            for j in range(4):
                axes[i, j].set_visible(False)
            metrics_rows.append({'case': case['name'], 'date': date_label, 'spc': np.nan, 'rmse': np.nan, 'ssim': np.nan, 'mae': np.nan})
            continue

        # calculate metrics for this case (on clipped extent)
        rmse, spc, ssim_score, mae = calculate_metrics(da_mswx, da_ddpm, case['extent'])
        metrics_rows.append({'case': case['name'], 'date': date_label, 'spc': spc, 'rmse': rmse, 'ssim': ssim_score, 'mae': mae})

        # plot panels with shared colormap/norm
        plot_panel(axes[i, 0], da_mswx, case, TEMP_CMAP, norm_temp_shared, is_error=False, metrics=None, gdf_india=gdf_india)
        plot_panel(axes[i, 1], da_coarse, case, TEMP_CMAP, norm_temp_shared, is_error=False, metrics=None, gdf_india=gdf_india)
        
        # ADD SMALL WHITE BOX IN LOW-RES COLUMN (INDEX 1)
        if 'inset_extent' in case:
            box = case['inset_extent'] # [lon_min, lon_max, lat_min, lat_max]
            # Create a Rectangle patch: (x, y), width, height
            rect = Rectangle((box[0], box[2]), box[1]-box[0], box[3]-box[2],
                             linewidth=1.2, edgecolor='white', facecolor='none',
                             transform=PROJECTION, zorder=50)
            axes[i, 1].add_patch(rect)

        plot_panel(axes[i, 2], da_ddpm, case, TEMP_CMAP, norm_temp_shared, is_error=False, metrics=None, gdf_india=gdf_india)
        plot_panel(axes[i, 3], da_error, case, CMAP_ERR, norm_err_shared, is_error=True, metrics=None, gdf_india=gdf_india)

        # ticks: first column both lat & lon; others only lon (bottom)
        try:
            lon_min, lon_max, lat_min, lat_max = case['extent'][0], case['extent'][1], case['extent'][2], case['extent'][3]
            xticks = np.linspace(lon_min, lon_max, 5)
            yticks = np.linspace(lat_min, lat_max, 5)
            
            # Determine if this is the last row
            is_last_row = (i == nrows - 1)

            for col in range(4):
                ax = axes[i, col]
                try:
                    ax.set_xticks(xticks, crs=PROJECTION)
                except Exception:
                    try:
                        ax.set_xticks(xticks)
                    except Exception:
                        pass
                if col == 0:
                    try:
                        ax.set_yticks(yticks, crs=PROJECTION)
                    except Exception:
                        try:
                            ax.set_yticks(yticks)
                        except Exception:
                            pass
                    # Only show bottom labels if it is the last row
                    ax.tick_params(which='both', axis='x', rotation=90, top=False, bottom=True, labeltop=False, labelbottom=is_last_row)
                    ax.tick_params(which='both', axis='y', left=True, right=False, labelleft=True, labelright=False)
                else:
                    try:
                        # Only show bottom labels if it is the last row
                        ax.tick_params(which='both', axis='x', rotation=90, top=False, bottom=True, labeltop=False, labelbottom=is_last_row)
                        ax.tick_params(which='both', axis='y', left=False, right=False, labelleft=False, labelright=False)
                    except Exception:
                        pass
                try:
                    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
                    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
                except Exception:
                    pass
        except Exception as e:
            print(f"  --- WARNING: could not set ticks for row {i}: {e}")

        # Draw contours at 4°C intervals for A/B/C using the shared clim
        try:
            vmin_viz = global_temp_vmin; vmax_viz = global_temp_vmax
            lvl_min = round_down_4(vmin_viz)
            lvl_max = round_up_4(vmax_viz)
            if lvl_min == lvl_max:
                lvl_min -= 4.0; lvl_max += 4.0
            levels = np.arange(lvl_min, lvl_max + 1e-6, 4.0)
            lw = 0.6

            for col_idx, da_tmp in enumerate([da_mswx, da_coarse, da_ddpm]):
                ax = axes[i, col_idx]
                da_tmp = ensure_lonlat_names(da_tmp)
                if da_tmp is None:
                    continue
                if 'time' in da_tmp.dims:
                    dat2 = da_tmp.mean(dim='time', skipna=True)
                else:
                    dat2 = da_tmp
                lon_name = 'lon' if 'lon' in dat2.coords else ('x' if 'x' in dat2.coords else None)
                lat_name = 'lat' if 'lat' in dat2.coords else ('y' if 'y' in dat2.coords else None)
                if lon_name and lat_name:
                    lon = np.array(dat2[lon_name].values)
                    lat = np.array(dat2[lat_name].values)
                    Z = np.array(dat2.values)
                    if Z.ndim == 2 and lon.size > 1 and lat.size > 1:
                        Lon, Lat = np.meshgrid(lon, lat)
                        mean_val = np.nanmean(Z[np.isfinite(Z)]) if np.isfinite(np.nanmean(Z)) else 0.0
                        contour_color = 'black' if mean_val >= (vmin_viz + (vmax_viz - vmin_viz)/4.0) else 'white'
                        ax.contour(Lon, Lat, Z, levels=levels, colors=contour_color, linewidths=lw,
                                   linestyles='-', transform=PROJECTION, zorder=50)
                else:
                    if 'x' in dat2.coords and 'y' in dat2.coords:
                        lon = np.array(dat2['x'].values)
                        lat = np.array(dat2['y'].values)
                        Z = np.array(dat2.values)
                        if Z.ndim == 2 and lon.size > 1 and lat.size > 1:
                            Lon, Lat = np.meshgrid(lon, lat)
                            contour_color = 'black'
                            ax.contour(Lon, Lat, Z, levels=levels, colors=contour_color, linewidths=lw,
                                     linestyles='-', transform=PROJECTION, zorder=50)
        except Exception as e:
            print(f"  --- WARNING: Could not draw contours for {case['name']}: {e}")

        # Metric boxes on top of DDPM (axes[i,2])
        try:
            axm = axes[i, 2]
            if not axm.get_visible():
                pass
            else:
                # SPC box
                spc_box = Rectangle((0.005, 0.005), 0.38, 0.08, transform=axm.transAxes,
                                    facecolor='white', edgecolor='none', zorder=400, clip_on=False)
                axm.add_patch(spc_box)
                spc_text = "spc: N/A" if np.isnan(spc) else f"SPC: {spc:.4f}"
                axm.text(0.015, 0.02, spc_text, transform=axm.transAxes, fontsize=12, va='bottom', ha='left', zorder=410, clip_on=False)

                # SSIM box
                ssim_box = Rectangle((0.60, 0.005), 0.395, 0.08, transform=axm.transAxes,
                                     facecolor='white', edgecolor='none', zorder=400, clip_on=False)
                axm.add_patch(ssim_box)
                ssim_text = "SSIM: N/A" if np.isnan(ssim_score) else f"SSIM: {ssim_score:.4f}"
                axm.text(0.98, 0.02, ssim_text, transform=axm.transAxes, fontsize=12, va='bottom', ha='right', zorder=410, clip_on=False)

        except Exception as e:
            print(f"  --- WARNING: Could not draw metric boxes for {case['name']}: {e}")
        
        # Vertical RMSE and MAE text on right of Error plot
        try:
            ax_err = axes[i, 3]
            if ax_err.get_visible():
                if np.isnan(rmse):
                    stats_str = "RMSE: N/A\nMAE: N/A"
                else:
                    stats_str = f"RMSE: {rmse:.2f} °C\nMAE: {mae:.2f} °C"

                ax_err.text(1.12, 0.5, stats_str, transform=ax_err.transAxes,
                            rotation=270, va='center', ha='left', fontsize=10, weight='bold')
        except Exception as e:
            print(f"  --- WARNING: Could not draw vertical RMSE/MAE text for {case['name']}: {e}")

    # ---------- AFTER PLOTTING: add two HORIZONTAL colorbars beneath the last (Himalayan) row ----------
    try:
        last_row = nrows - 1
        bbox_coarse = axes[last_row, 1].get_position()
        bbox_error = axes[last_row, 3].get_position()

        # temperature colorbar
        cbar_h_height = 0.018
        # INCREASED GAP FROM 0.035 to 0.08
        cbar_gap = 0.08
        # Ensure we don't drop below canvas; might need tuning if bottom=0.08
        cbar_temp_y = max(0.01, bbox_coarse.y0 - cbar_gap)
        cax_temp = fig.add_axes([bbox_coarse.x0, cbar_temp_y, bbox_coarse.width, cbar_h_height], zorder=250)
        sm_temp = mpl.cm.ScalarMappable(norm=norm_temp_shared, cmap=TEMP_CMAP)
        sm_temp.set_array([])
        
        # REMOVED integer=True constraint, allow floats
        locator_temp = mticker.MaxNLocator(nbins=7, integer=False, min_n_ticks=3)
        ticks_temp = locator_temp.tick_values(norm_temp_shared.vmin, norm_temp_shared.vmax)
        ticks_temp = [t for t in ticks_temp if t >= norm_temp_shared.vmin and t <= norm_temp_shared.vmax]
        
        cbar_temp = fig.colorbar(sm_temp, cax=cax_temp, orientation='horizontal', ticks=ticks_temp)
        cbar_temp.set_label("Temperature (°C)", fontsize=12)
        cbar_temp.ax.tick_params(labelsize=12)
        try:
            # Format as float with 1 decimal place
            cbar_temp.ax.set_xticklabels([f"{t:.1f}" for t in cbar_temp.get_ticks()], fontsize=12)
        except Exception:
            pass

        # error colorbar
        cbar_err_y = max(0.01, bbox_error.y0 - cbar_gap)
        cax_err = fig.add_axes([bbox_error.x0, cbar_err_y, bbox_error.width, cbar_h_height], zorder=250)
        sm_err = mpl.cm.ScalarMappable(norm=norm_err_shared, cmap=CMAP_ERR)
        sm_err.set_array([])
        
        # REMOVED integer=True constraint, allow floats
        locator_err = mticker.MaxNLocator(nbins=5, integer=False, min_n_ticks=3)
        ticks_err = locator_err.tick_values(norm_err_shared.vmin, norm_err_shared.vmax)
        ticks_err = [t for t in ticks_err if t >= norm_err_shared.vmin and t <= norm_err_shared.vmax]
        
        cbar_err = fig.colorbar(sm_err, cax=cax_err, orientation='horizontal', ticks=ticks_err)
        cbar_err.set_label("Absolute Error (°C)", fontsize=12)
        cbar_err.ax.tick_params(labelsize=12)
        try:
            # Format as float with 1 decimal place
            cbar_err.ax.set_xticklabels([f"{t:.1f}" for t in cbar_err.get_ticks()], fontsize=12)
        except Exception:
            pass

    except Exception as e:
        print("  --- WARNING: Could not create final shared horizontal colorbars:", e)

    # finalize: draw, add column headers and left labels
    fig.canvas.draw()

    col_titles = ["MSWX (0.1°)", "Low-Res MSWX (1.0°)", "DDPM MSWX (0.1°)", "Absolute Error"]
    for j, title in enumerate(col_titles):
        try:
            bbox = axes[0, j].get_position()
            x_center = bbox.x0 + bbox.width / 2.0
            y_top = bbox.y1
            fig.text(x_center, y_top + 0.012, title, fontsize=12, weight='bold', ha='center', va='bottom')
        except Exception:
            pass

    for i, case in enumerate(CASE_STUDIES):
        try:
            bbox = axes[i, 0].get_position()
            y_center = bbox.y0 + bbox.height / 2.0
            x_pos = 0.06 
            
            if case['type'] == 'day':
                date_label = pd.to_datetime(case['date']).strftime("%Y-%m-%d")
            else:
                date_label = f"{case.get('year')}-{case.get('month'):02d}"
            
            mean_str = "Mean: N/A"
            try:
                da_full = loaded[i]['da_mswx']
                if da_full is not None:
                    ext = case['extent']
                    da_clipped = sel_lonlat_slice(da_full, ext[0], ext[1], ext[2], ext[3])
                    val = float(da_clipped.mean().item())
                    mean_str = f"Mean: {val:.1f} °C"
            except Exception:
                pass

            left_label = f"{case['name']}\n{date_label}\n{mean_str}"
            fig.text(x_pos, y_center, left_label, fontsize=10, va='center', ha='center', rotation=90, weight='bold')
        except Exception:
            pass

    # save metrics CSV
    try:
        dfm = pd.DataFrame(metrics_rows)
        os.makedirs(os.path.dirname(metrics_csv), exist_ok=True)
        dfm.to_csv(metrics_csv, index=False)
        print(f"Saved per-case metrics to {metrics_csv}")
    except Exception as e:
        print("Could not save metrics CSV:", e)

    # save figure
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"\n✅ Plotting complete. Figure saved to {output_file}")
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
DDPM Case Study Visualisation — High-Legibility Final Version
============================================================
Updates:
- Significantly increased font sizes for Lat/Lon, Metrics, and Headers.
- Unified India Extent for all rows.
- Silver Isotherms (4°C) with improved visibility.
- Masked analysis and plotting using official India_Boundary.shp.
"""

import os, warnings
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import regionmask
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
from matplotlib.patches import Rectangle
import cmcrameri.cm as cmc
import cartopy.crs as ccrs
from scipy.stats import pearsonr
from skimage.metrics import structural_similarity as ssim

warnings.filterwarnings("ignore")

# ── PATHS ────────────────────────────────────────────────────────────────────
HR_MSWX_DIR   = "<SET YOUR PATH HERE>"
HR_DDPM_DIR   = "<SET YOUR PATH HERE>"
LR_INPUT_DIR  = "<SET YOUR PATH HERE>"
INDIA_SHP     = "<SET YOUR PATH HERE>"
OUT_PNG       = "DDPM_case_studies_high_legibility.png"

# ── DISPLAY SETTINGS (Increased Font Sizes) ──────────────────────────────────
TEMP_CMAP, ERR_CMAP = cmc.vik, "viridis"
PROJ = ccrs.PlateCarree()
DPI  = 300

# Font Size Config for high legibility
FS_HEADER   = 16   # Column titles
FS_ROW      = 14   # Row titles (left side)
FS_METRIC   = 12   # SPC, SSIM, RMSE, MAE (inside white boxes)
FS_CRITERIA = 12   # Selection justification (right side)
FS_TICKS    = 11   # Lat/Lon coordinate values
FS_CBAR     = 13   # Colorbar labels

INDIA_EXT = [67.0, 97.5, 7.0, 37.5]

# ── CASE STUDIES ─────────────────────────────────────────────────────────────
CASE_STUDIES = [
    {"label": "Extreme Heat", "token": "2012152", "date_str":"31 May 2012", 
     "criteria": "AMJ Season:\nHighest Domain-Mean"},
    {"label": "Extreme Cold", "token": "2011006", "date_str":"06 Jan 2011", 
     "criteria": "DJF Season:\nLowest Domain-Mean"},
    {"label": "Monsoon Complexity", "token": "2012156", "date_str":"04 Jun 2012", 
     "criteria": "JJAS Season:\nMax Spatial Std (σ)"},
    {"label": "Representative Winter", "token": "2013348", "date_str":"14 Dec 2013", 
     "criteria": "DJF Season:\nRepresentative Winter"},
    {"label": "Max Spatial Gradient", "token": "2010082", "date_str":"23 Mar 2010", 
     "criteria": "All Seasons:\nMax Laplacian Norm"},
]

# ── HELPERS ──────────────────────────────────────────────────────────────────
def fix_coords(da):
    da = da.squeeze()
    if "time" in da.coords: da = da.drop_vars("time")
    rn = {o: n for o, n in [("x","lon"),("y","lat"),("longitude","lon"),("latitude","lat")] 
          if o in da.coords and n not in da.coords}
    return da.rename(rn) if rn else da

def to_celsius(da):
    return da - 273.15 if float(da.mean()) > 200 else da

# ── PROCESSING ───────────────────────────────────────────────────────────────
india_gdf = gpd.read_file(INDIA_SHP)
cache, t_vals, e_vals = [], [], []

print("Processing spatial masking and metrics...")

for case in CASE_STUDIES:
    f_hr = os.path.join(HR_MSWX_DIR, f"{case['token']}.nc")
    f_lr = os.path.join(LR_INPUT_DIR, f"{case['token']}_1deg.nc")
    f_dp = os.path.join(HR_DDPM_DIR, f"{case['token']}_ddpm.nc")
    
    if not all(os.path.exists(p) for p in [f_hr, f_lr, f_dp]): continue
    
    hr  = to_celsius(fix_coords(xr.open_dataset(f_hr)["air_temperature"]))
    lr  = to_celsius(fix_coords(xr.open_dataset(f_lr)["air_temperature"]))
    ddp = to_celsius(fix_coords(xr.open_dataset(f_dp)["air_temperature"]))
    
    mask_hr = regionmask.mask_3D_geopandas(india_gdf, hr.lon, hr.lat).any("region")
    mask_lr = regionmask.mask_3D_geopandas(india_gdf, lr.lon, lr.lat).any("region")
    
    hr_m, lr_m, ddp_m = hr.where(mask_hr), lr.where(mask_lr), ddp.where(mask_hr)
    ddp_res = ddp_m.interp_like(hr_m)
    err_m = np.abs(hr_m - ddp_res)
    
    def clip(da):
        l_sl = slice(INDIA_EXT[3], INDIA_EXT[2]) if da.lat[0] > da.lat[-1] else slice(INDIA_EXT[2], INDIA_EXT[3])
        return da.sel(lon=slice(INDIA_EXT[0], INDIA_EXT[1]), lat=l_sl)

    cache.append({"hr": clip(hr_m), "lr": clip(lr_m), "ddp": clip(ddp_m), "err": clip(err_m), "crit": case["criteria"]})
    t_vals.extend([hr_m.values.flatten(), ddp_m.values.flatten()])
    e_vals.append(err_m.values.flatten())

norm_T = mcolors.Normalize(vmin=np.percentile(np.concatenate([v[np.isfinite(v)] for v in t_vals]), 1), 
                           vmax=np.percentile(np.concatenate([v[np.isfinite(v)] for v in t_vals]), 99))
norm_E = mcolors.Normalize(vmin=0, vmax=np.percentile(np.concatenate([v[np.isfinite(v)] for v in e_vals]), 98))

# ── PLOTTING ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(len(CASE_STUDIES), 4, figsize=(22, 22), subplot_kw={"projection": PROJ})
plt.subplots_adjust(left=0.10, right=0.85, top=0.96, bottom=0.07, hspace=0.12, wspace=0.02)

for i, entry in enumerate(cache):
    fields = [entry["hr"], entry["lr"], entry["ddp"], entry["err"]]
    cmaps, norms = [TEMP_CMAP]*3 + [ERR_CMAP], [norm_T]*3 + [norm_E]
    
    for j in range(4):
        ax = axes[i, j]
        ax.set_extent(INDIA_EXT, crs=PROJ)
        fields[j].plot(ax=ax, cmap=cmaps[j], norm=norms[j], transform=PROJ, add_colorbar=False, add_labels=False, rasterized=True)
        india_gdf.boundary.plot(ax=ax, color="black", linewidth=0.9, transform=PROJ, zorder=15)
        
        # Bold Silver Contours
        if j < 3:
            ax.contour(fields[j].lon, fields[j].lat, fields[j].values, levels=np.arange(-20, 50, 4), 
                       colors="yellow", linewidths=0.8, alpha=0.9, transform=PROJ, zorder=12)

        # High-Legibility Lat/Lon Ticks
        gl = ax.gridlines(crs=PROJ, draw_labels=True, linewidth=0.3, color='gray', alpha=0.3, linestyle='--')
        gl.top_labels = gl.right_labels = False
        gl.left_labels = (j == 0); gl.bottom_labels = (i == len(cache)-1)
        gl.xlabel_style = gl.ylabel_style = {'size': FS_TICKS, 'weight': 'bold'}

    # Metrics Computation
    t_v, p_v = fields[0].values, fields[2].values
    m = np.isfinite(t_v) & np.isfinite(p_v)
    rmse, mae = np.sqrt(np.mean((t_v[m]-p_v[m])**2)), np.mean(np.abs(t_v[m]-p_v[m]))
    spc = pearsonr(t_v[m], p_v[m])[0]
    
    t_2d = np.where(np.isfinite(t_v), t_v, np.nanmean(t_v))
    p_2d = np.where(np.isfinite(p_v), p_v, np.nanmean(p_v))
    ssim_v = ssim(t_2d, p_2d, data_range=np.max(t_2d)-np.min(t_2d)) if (np.max(t_2d)-np.min(t_2d)) > 0 else 0

    # Large Metric Boxes
    axes[i, 2].text(0.96, 0.04, f"SPC: {spc:.3f}\nSSIM: {ssim_v:.3f}", transform=axes[i,2].transAxes, 
                    ha="right", fontweight="bold", fontsize=FS_METRIC, bbox=dict(fc="white", alpha=0.9, ec="#888", boxstyle="round,pad=0.3"))
    axes[i, 3].text(0.96, 0.04, f"RMSE: {rmse:.3f}\nMAE: {mae:.3f}", transform=axes[i,3].transAxes, 
                    ha="right", fontweight="bold", fontsize=FS_METRIC, bbox=dict(fc="white", alpha=0.9, ec="#888", boxstyle="round,pad=0.3"))

    # Selection Justification
    axes[i, 3].text(1.1, 0.5, entry["crit"], transform=axes[i, 3].transAxes, va="center", ha="left", 
                    fontweight="bold", fontsize=FS_CRITERIA, bbox=dict(boxstyle="round,pad=0.5", fc="#f4f4f4", ec="#aaa"))

    # Large Row Labels
    axes[i, 0].text(-0.4, 0.5, f"{CASE_STUDIES[i]['label']}\n{CASE_STUDIES[i]['date_str']}", transform=axes[i,0].transAxes, 
                    rotation=90, va="center", ha="center", fontweight="bold", fontsize=FS_ROW)

# Large Headers
headers = ["Reference (MSWX)", "Coarse Input (1°)", "DDPM Output", "Absolute Bias"]
for j, h in enumerate(headers): axes[0, j].set_title(h, fontweight="bold", fontsize=FS_HEADER, pad=15)

# Large Colorbars
cax_t = fig.add_axes([0.15, 0.04, 0.25, 0.015])
cb1 = fig.colorbar(mpl.cm.ScalarMappable(norm=norm_T, cmap=TEMP_CMAP), cax=cax_t, orientation="horizontal")
cb1.set_label("Temperature (°C)", fontsize=FS_CBAR, fontweight='bold')

cax_e = fig.add_axes([0.55, 0.04, 0.2, 0.015])
cb2 = fig.colorbar(mpl.cm.ScalarMappable(norm=norm_E, cmap=ERR_CMAP), cax=cax_e, orientation="horizontal")
cb2.set_label("Bias Magnitude (°C)", fontsize=FS_CBAR, fontweight='bold')

# plt.savefig(OUT_PNG, dpi=DPI, bbox_inches="tight")
plt.show()

In [ ]:
#!/usr/bin/env python3
"""
DDPM Case Study Visualisation — High-Legibility Final Version
============================================================
Updates:
- Event labels (Col 1) moved to Bottom-Right corner.
- KGE metric (Col 3) moved to Top-Right corner.
- SSIM (Col 3) maintained in Bottom-Right for balance.
- Font sizes significantly increased for visibility from distance.
"""

import os, warnings
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import regionmask
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
from scipy.stats import pearsonr
from skimage.metrics import structural_similarity as ssim
import cmcrameri.cm as cmc
import cartopy.crs as ccrs

warnings.filterwarnings("ignore")

# ── PATHS ─────────────────────────────────────────────────────────────────────
HR_MSWX_DIR   = "<SET YOUR PATH HERE>"
HR_DDPM_DIR   = "<SET YOUR PATH HERE>"
LR_INPUT_DIR  = "<SET YOUR PATH HERE>"
INDIA_SHP     = "<SET YOUR PATH HERE>"
OUT_PNG       = "<SET YOUR PATH HERE>"
OUT_CSV       = "<SET YOUR PATH HERE>"

# ── DISPLAY SETTINGS (Sized for Distance Legibility) ──────────────────────────
TEMP_CMAP = cmc.vik
ERR_CMAP   = "cividis"
PROJ       = ccrs.PlateCarree()
DPI        = 600

# Scaled up font configuration
FS_HEADER = 18   # Column titles
FS_ROW    = 14   # Event/Date inset text
FS_METRIC = 14   # Performance metrics
FS_TICKS  = 12   # Lat/Lon coordinate values
FS_CBAR   = 15   # Colorbar labels

INDIA_EXT = [67.0, 97.5, 7.0, 37.5]

# ── CASE STUDIES ──────────────────────────────────────────────────────────────
CASE_STUDIES = [
    {"label": "Extreme Heat",          "token": "2012152", "date_str": "31 May 2012"},
    {"label": "Extreme Cold",          "token": "2011006", "date_str": "06 Jan 2011"},
    {"label": "Monsoon Complexity",    "token": "2012156", "date_str": "04 Jun 2012"},
    {"label": "Representative Winter", "token": "2013348", "date_str": "14 Dec 2013"},
]

# ── METRIC FUNCTIONS ──────────────────────────────────────────────────────────
def kge(obs, sim):
    r     = float(pearsonr(obs, sim)[0])
    alpha = float(np.std(sim)  / np.std(obs))  if np.std(obs)  > 0 else np.nan
    beta  = float(np.mean(sim) / np.mean(obs)) if np.mean(obs) != 0 else np.nan
    return 1.0 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2) if not np.isnan(alpha) else np.nan

def variance_ratio(obs, sim):
    return float(np.std(sim) / np.std(obs)) if np.std(obs) > 0 else np.nan

def compute_all_metrics(t_2d, p_2d):
    mask = np.isfinite(t_2d) & np.isfinite(p_2d)
    tv, pv = t_2d[mask], p_2d[mask]
    rmse_v = float(np.sqrt(np.mean((tv - pv)**2)))
    mae_v  = float(np.mean(np.abs(tv - pv)))
    tf = np.where(np.isfinite(t_2d), t_2d, np.nanmean(t_2d))
    pf = np.where(np.isfinite(p_2d), p_2d, np.nanmean(p_2d))
    dr = float(np.nanmax(tf) - np.nanmin(tf))
    ssim_v = float(ssim(tf, pf, data_range=dr)) if dr > 0 else np.nan
    return {"rmse": rmse_v, "mae": mae_v, "kge": kge(tv, pv), "ssim_v": ssim_v, "vr": variance_ratio(tv, pv)}

# ── HELPERS ───────────────────────────────────────────────────────────────────
def fix_coords(da):
    da = da.squeeze()
    if "time" in da.coords: da = da.drop_vars("time")
    rn = {o: n for o, n in [("x","lon"),("y","lat"),("longitude","lon"),("latitude","lat")] if o in da.coords and n not in da.coords}
    return da.rename(rn) if rn else da

def to_celsius(da):
    return da - 273.15 if float(da.mean()) > 200 else da

def clip_india(da):
    lat_sl = (slice(INDIA_EXT[3], INDIA_EXT[2]) if float(da.lat[0]) > float(da.lat[-1]) else slice(INDIA_EXT[2], INDIA_EXT[3]))
    return da.sel(lon=slice(INDIA_EXT[0], INDIA_EXT[1]), lat=lat_sl)

def metric_box(ax, text, x, y, ha="right", va="bottom", fontsize=FS_METRIC):
    ax.text(x, y, text, transform=ax.transAxes, ha=ha, va=va, fontweight="bold", fontsize=fontsize,
            bbox=dict(fc="white", alpha=0.95, ec="#444444", boxstyle="round,pad=0.4", lw=1.0), zorder=30)

# ── PROCESSING ────────────────────────────────────────────────────────────────
print("Pass 1 — loading and masking ...")
india_gdf = gpd.read_file(INDIA_SHP)
cache, t_vals, e_vals = [], [], []

for case in CASE_STUDIES:
    paths = [os.path.join(d, f"{case['token']}" + s) for d, s in zip([HR_MSWX_DIR, LR_INPUT_DIR, HR_DDPM_DIR], [".nc", "_1deg.nc", "_ddpm.nc"])]
    if not all(os.path.exists(p) for p in paths): continue

    hr, lr, ddp = [to_celsius(fix_coords(xr.open_dataset(p)["air_temperature"])) for p in paths]
    m_hr = regionmask.mask_3D_geopandas(india_gdf, hr.lon, hr.lat).any("region")
    m_lr = regionmask.mask_3D_geopandas(india_gdf, lr.lon, lr.lat).any("region")

    hr_m, lr_m, ddp_m = hr.where(m_hr), lr.where(m_lr), ddp.where(m_hr)
    ddp_r = ddp_m.interp_like(hr_m, method="linear")
    err_m = np.abs(hr_m - ddp_r)

    cache.append({"hr": clip_india(hr_m), "lr": clip_india(lr_m), "ddp": clip_india(ddp_m), "ddp_r": clip_india(ddp_r), "err": clip_india(err_m)})
    for da in (hr_m, ddp_m):
        v = da.values.ravel(); t_vals.append(v[np.isfinite(v)])
    v = err_m.values.ravel(); e_vals.append(v[np.isfinite(v)])

all_t  = np.concatenate(t_vals)
norm_T = mcolors.Normalize(vmin=np.percentile(all_t, 1), vmax=np.percentile(all_t, 99))
norm_E = mcolors.Normalize(vmin=0, vmax=np.percentile(np.concatenate(e_vals), 98))

# ── FIGURE ────────────────────────────────────────────────────────────────────
NROWS = len(CASE_STUDIES)
fig, axes = plt.subplots(NROWS, 4, figsize=(22, NROWS * 5.0), subplot_kw={"projection": PROJ})
fig.patch.set_facecolor("white")
# Increased bottom to ensure colorbars and labels have breathing room
fig.subplots_adjust(left=0.05, right=0.95, top=0.94, bottom=0.10, hspace=0.12, wspace=0.04)

# ── PLOTTING ──────────────────────────────────────────────────────────────────
metrics_rows = []
for i, (case, entry) in enumerate(zip(CASE_STUDIES, cache)):
    is_last = (i == NROWS - 1)
    fields = [entry["hr"], entry["lr"], entry["ddp"], entry["err"]]
    cmaps, norms = [TEMP_CMAP]*3 + [ERR_CMAP], [norm_T]*3 + [norm_E]

    for j in range(4):
        ax = axes[i, j]
        ax.set_extent(INDIA_EXT, crs=PROJ)
        fields[j].plot(ax=ax, cmap=cmaps[j], norm=norms[j], transform=PROJ, add_colorbar=False, add_labels=False, rasterized=True)
        india_gdf.boundary.plot(ax=ax, color="black", linewidth=1.0, transform=PROJ, zorder=15)
        
        gl = ax.gridlines(crs=PROJ, draw_labels=True, linewidth=0.3, color="#aaaaaa", alpha=0.4, linestyle="--")
        gl.top_labels = gl.right_labels = False
        gl.left_labels, gl.bottom_labels = (j == 0), is_last
        gl.xlabel_style = gl.ylabel_style = {"size": FS_TICKS, "weight": "bold"}

        # Column 1: Event/Day label moved to Bottom-Right
        if j == 0:
            label_text = f"{case['label']}\n{case['date_str']}"
            metric_box(ax, label_text, x=0.96, y=0.04, ha="right", va="bottom", fontsize=FS_ROW)

    m = compute_all_metrics(entry["hr"].values.astype(float), entry["ddp_r"].values.astype(float))
    metrics_rows.append({"case": case["label"], "date": case["date_str"], **m})
    
    # Column 3 (DDPM): KGE -> Top-Right | SSIM -> Bottom-Right
    metric_box(axes[i, 2], f"KGE: {m['kge']:.3f}", x=0.96, y=0.96, ha="right", va="top", fontsize=FS_METRIC)
    metric_box(axes[i, 2], f"SSIM: {m['ssim_v']:.3f}", x=0.96, y=0.04, ha="right", va="bottom", fontsize=FS_METRIC)
    
    # Column 4 (Error): RMSE/MAE/VR -> Bottom-Right
    metric_box(axes[i, 3], f"RMSE: {m['rmse']:.3f} C\nMAE: {m['mae']:.3f} C\nVR: {m['vr']:.3f}", 
               x=0.96, y=0.04, ha="right", va="bottom", fontsize=FS_METRIC)

# ── COLORBARS ─────────────────────────────────────────────────────────────────
bbox_l, bbox_r = axes[-1, 0].get_position(), axes[-1, 2].get_position()
cax_t = fig.add_axes([bbox_l.x0, 0.03, (bbox_r.x1 - bbox_l.x0), 0.02])
fig.colorbar(mpl.cm.ScalarMappable(norm=norm_T, cmap=TEMP_CMAP), cax=cax_t, orientation="horizontal", extend="both").set_label("Temperature (°C)", fontsize=FS_CBAR, fontweight="bold")

bbox_e = axes[-1, 3].get_position()
cax_e = fig.add_axes([bbox_e.x0, 0.03, bbox_e.width, 0.02])
fig.colorbar(mpl.cm.ScalarMappable(norm=norm_E, cmap=ERR_CMAP), cax=cax_e, orientation="horizontal", extend="max").set_label("Absolute Error (°C)", fontsize=FS_CBAR, fontweight="bold")

# ── ANNOTATIONS ───────────────────────────────────────────────────────────────
fig.canvas.draw()
titles = ["Reference MSWX", "Coarse Input (1.0°)", "DDPM Output", "Absolute Error"]
for j, title in enumerate(titles):
    bbox = axes[0, j].get_position()
    fig.text(bbox.x0 + bbox.width / 2, bbox.y1 + 0.015, title, fontsize=FS_HEADER, fontweight="bold", ha="center")

plt.savefig(OUT_PNG, dpi=DPI, bbox_inches="tight")
print(f"Success! Saved to {OUT_PNG}")
plt.show()

In [ ]:
#!/usr/bin/env python3
"""
Grid Information Detector
Purpose: Reads a sample NetCDF file and extracts the key grid parameters
         needed for the PSD analysis script.
"""

import xarray as xr
import numpy as np
import sys
import warnings

# --- CONFIGURATION ---
# Change this if your main climate variable is not 'air_temperature'
VARIABLE_NAME = "air_temperature"
# Change this if your coordinate names are not 'lat' and 'lon'
LAT_NAME = "lat"
LON_NAME = "lon"
# ---------------------

def analyze_grid(file_path):
    """
    Loads a NetCDF file and calculates grid parameters.
    """
    print(f"--- Analyzing file: {file_path} ---")
    
    try:
        with xr.open_dataset(file_path) as ds:
            
            # --- 1. Get Domain Pixel Size (N_PIXELS) ---
            
            # Check for main variable
            if VARIABLE_NAME not in ds.variables:
                print(f"Error: Variable '{VARIABLE_NAME}' not found.")
                print(f"Available variables: {list(ds.variables.keys())}")
                return
                
            # Check for coordinates
            if LAT_NAME not in ds.coords or LON_NAME not in ds.coords:
                print(f"Error: Coordinates '{LAT_NAME}' or '{LON_NAME}' not found.")
                print(f"Available coordinates: {list(ds.coords.keys())}")
                return

            lat_coords = ds[LAT_NAME]
            lon_coords = ds[LON_NAME]
            
            n_lat = len(lat_coords)
            n_lon = len(lon_coords)
            
            print(f"Grid dimensions (Lat, Lon): ({n_lat}, {n_lon})")
            
            # For the Q1 script, N_PIXELS refers to the first dimension (latitude)
            your_domain_n_pixels = n_lat
            
            if n_lat != n_lon:
                warnings.warn(
                    f"Warning: Grid is not square ({n_lat}x{n_lon}). "
                    "The PSD analysis assumes a roughly isotropic grid. "
                    f"Using N_PIXELS = {n_lat} (from latitude)."
                )

            # --- 2. Get Grid Spacing (GRID_SPACING_KM) ---
            
            # Calculate spacing in degrees
            # Use .diff() and take the mean for robustness, just in case
            lat_spacing_deg = np.abs(np.mean(np.diff(lat_coords)))
            lon_spacing_deg = np.abs(np.mean(np.diff(lon_coords)))
            
            print(f"Grid spacing (deg): (Lat: {lat_spacing_deg:.4f}, Lon: {lon_spacing_deg:.4f})")
            
            # Find domain center latitude for longitude correction
            center_lat_deg = lat_coords.mean().item()
            
            # Convert degrees to kilometers
            DEG_TO_KM_LAT = 111.13  # Average km per degree latitude
            
            lat_spacing_km = lat_spacing_deg * DEG_TO_KM_LAT
            lon_spacing_km = lon_spacing_deg * DEG_TO_KM_LAT * np.cos(np.radians(center_lat_deg))
            
            print(f"Grid spacing (km): (Lat: {lat_spacing_km:.3f}, Lon: {lon_spacing_km:.3f})")

            # Use the average spacing for the isotropic assumption
            your_grid_spacing_km = (lat_spacing_km + lon_spacing_km) / 2.0
            
            print("\n" + "="*50)
            print("--- COPY THESE VALUES INTO YOUR Q1 SCRIPT ---")
            print(f"\nYOUR_GRID_SPACING_KM = {your_grid_spacing_km:.4f}")
            print(f"YOUR_DOMAIN_N_PIXELS = {your_domain_n_pixels}\n")
            print("="*50)
            
    except Exception as e:
        print(f"\nAn error occurred:")
        print(e)

# --- MAIN EXECUTION ---
# Hardcoded the file path you provided:
file_path_to_analyze = "<SET YOUR PATH HERE>"

# Run the analysis
analyze_grid(file_path_to_analyze)

In [ ]:
#!/usr/bin/env python3
"""
Power Spectral Density Analysis (2003–2014 Mean) - Q1 JOURNAL QUALITY

Notes:
- Only the main PSD comparison panel (panel 1) is produced.
- Kolmogorov and mesoscale reference lines removed.
- Ensemble stats (mean, 95% CI) saved to CSV; metrics saved to a text file.
- Title removed and fonts increased for legibility.
"""

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings
from scipy.ndimage import uniform_filter1d

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =====================================================================
# CONFIGURATION & CONSTANTS
# =====================================================================

# Grid Parameters (from your domain)
YOUR_GRID_SPACING_KM = 10.5924
YOUR_DOMAIN_N_PIXELS = 500

# Physical Constants & References
RESOLUTION_LIMIT_KM = 111.0  # 1° at equator (degrees to km conversion)

# Paths (update if needed)
HR_DIR = "<SET YOUR PATH HERE>"
DDPM_DIR = "<SET YOUR PATH HERE>"

# Output files (written in current working dir)
PLOT_FILE_MAIN = "psd_analysis_main_no_bilin_no_refs_largefont.png"
DATA_FILE = "psd_analysis_final_data_no_bilin_no_refs.csv"
STATS_FILE = "psd_analysis_statistics_no_bilin_no_refs.txt"

# Font sizes for improved legibility
LABEL_FS = 16
TICK_FS = 14
LEGEND_FS = 14
ANNOT_FS = 13
FIGSIZE = (12, 8)  # slightly larger figure

# =====================================================================
# RADIAL PSD FUNCTION (IMPROVED)
# =====================================================================

def radial_psd(field, grid_spacing_km, convert_to_1d=True):
    """
    Compute isotropic 1D PSD from 2D field with proper normalization.
    Returns radial (1D) PSD (or 2D-averaged radial values if convert_to_1d=False).
    """
    f = field - np.nanmean(field)

    if f.ndim != 2:
        raise ValueError(f"Input field must be 2D, got shape {field.shape}")

    ny, nx = f.shape
    domain_width_km = nx * grid_spacing_km
    domain_height_km = ny * grid_spacing_km

    # 2D FFT with proper normalization
    F = np.fft.fftshift(np.fft.fft2(f)) / (nx * ny)
    psd2d = (np.abs(F) ** 2) * (domain_width_km * domain_height_km)

    # Radial binning for isotropic PSD
    y, x = np.indices((ny, nx))
    cx, cy = nx // 2, ny // 2

    r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
    r = r.astype(int)

    tbin = np.bincount(r.ravel(), psd2d.ravel())
    nr = np.bincount(r.ravel())
    rad = tbin / np.maximum(nr, 1)

    # Convert 2D radial spectrum to 1D by integrating over annulus
    if convert_to_1d:
        k_indices = np.arange(len(rad)).astype(float)
        # avoid divide-by-zero for annulus factor at k=0
        annulus_factor = np.zeros_like(k_indices)
        nonzero = k_indices > 0
        annulus_factor[nonzero] = 2 * np.pi * k_indices[nonzero] / domain_width_km
        rad = rad * annulus_factor

    return rad


def compute_spectral_slope(wavenumber, psd, k_min, k_max):
    """
    Compute spectral slope in log-log space using least-squares fit.
    Returns (slope, r_squared).
    """
    mask = (wavenumber >= k_min) & (wavenumber <= k_max) & (psd > 0) & (wavenumber > 0)

    if mask.sum() < 3:
        return np.nan, np.nan

    log_k = np.log10(wavenumber[mask])
    log_psd = np.log10(psd[mask])

    coeffs = np.polyfit(log_k, log_psd, 1)
    slope = coeffs[0]

    predicted = np.polyval(coeffs, log_k)
    ss_res = np.sum((log_psd - predicted) ** 2)
    ss_tot = np.sum((log_psd - np.mean(log_psd)) ** 2)
    r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan

    return slope, r_squared


def compute_spectrum_metrics(wavenumber, psd_array, method_name):
    """
    Compute comprehensive metrics for spectral comparison.
    Accepts psd_array with shape (n_samples, n_k) or (n_k,)
    """
    if psd_array.ndim == 1:
        psd_array = psd_array[np.newaxis, :]

    mean_psd = np.nanmean(psd_array, axis=0)

    # Spectral slopes in different regimes (values chosen as in original)
    k_syn_min, k_syn_max = 0.001, 0.01
    slope_syn, r2_syn = compute_spectral_slope(wavenumber, mean_psd, k_syn_min, k_syn_max)

    k_meso_min, k_meso_max = 0.01, 0.1
    slope_meso, r2_meso = compute_spectral_slope(wavenumber, mean_psd, k_meso_min, k_meso_max)

    # Total integrated energy (approx via discrete sum)
    valid_mask = (wavenumber > 0) & (mean_psd > 0)
    dk = wavenumber[1] - wavenumber[0] if len(wavenumber) > 1 else 1.0
    total_energy = np.nansum(mean_psd[valid_mask] * dk)

    # Peak wavenumber
    peak_idx = np.nanargmax(mean_psd)
    peak_k = wavenumber[peak_idx] if peak_idx < len(wavenumber) else np.nan
    peak_psd = mean_psd[peak_idx] if peak_idx < len(mean_psd) else np.nan

    metrics = {
        "method": method_name,
        "mean_psd": mean_psd,
        "std_psd": np.nanstd(psd_array, axis=0),
        "slope_synoptic": slope_syn,
        "r2_synoptic": r2_syn,
        "slope_mesoscale": slope_meso,
        "r2_mesoscale": r2_meso,
        "total_energy": total_energy,
        "peak_wavenumber": peak_k,
        "peak_psd": peak_psd,
    }

    return metrics


# =====================================================================
# MAIN EXECUTION
# =====================================================================

def main():
    years = range(2003, 2015)

    psd_HR_all = []
    psd_DDPM_all = []

    print("=" * 70)
    print("PSD ANALYSIS: Loading and processing climate data (2003-2014)")
    print("=" * 70)

    # -------------------------------------------------------
    # DATA LOADING
    # -------------------------------------------------------
    for yr in years:
        print(f"  Processing year {yr}...")
        year_count = 0
        for doy in range(1, 367):
            fname_hr = f"{yr}{doy:03d}.nc"
            f_hr = os.path.join(HR_DIR, fname_hr)
            f_ddpm = os.path.join(DDPM_DIR, f"{yr}{doy:03d}_ddpm.nc")

            if not (os.path.exists(f_hr) and os.path.exists(f_ddpm)):
                continue

            try:
                ds_hr = xr.open_dataset(f_hr)
                ds_ddpm = xr.open_dataset(f_ddpm)
            except Exception as e:
                print(f"    Skipping {fname_hr}, load error: {e}")
                try:
                    ds_hr.close()
                except Exception:
                    pass
                try:
                    ds_ddpm.close()
                except Exception:
                    pass
                continue

            try:
                var_hr = "air_temperature" if "air_temperature" in ds_hr.data_vars else list(ds_hr.data_vars)[0]
                var_ddpm = "air_temperature" if "air_temperature" in ds_ddpm.data_vars else list(ds_ddpm.data_vars)[0]

                da_hr = np.squeeze(ds_hr[var_hr].values)
                da_ddpm = np.squeeze(ds_ddpm[var_ddpm].values)
            except Exception as e:
                print(f"    Skipping {fname_hr}, extraction error: {e}")
                ds_hr.close()
                ds_ddpm.close()
                continue
            finally:
                ds_hr.close()
                ds_ddpm.close()

            # Validate shape
            if da_hr.ndim != 2 or da_hr.shape[0] != YOUR_DOMAIN_N_PIXELS or da_hr.shape[1] != YOUR_DOMAIN_N_PIXELS:
                continue

            # Compute radial PSDs
            try:
                psd_hr = radial_psd(da_hr, YOUR_GRID_SPACING_KM)
                psd_ddpm = radial_psd(da_ddpm, YOUR_GRID_SPACING_KM)
            except Exception as e:
                print(f"    Skipping {fname_hr}, PSD error: {e}")
                continue

            psd_HR_all.append(psd_hr)
            psd_DDPM_all.append(psd_ddpm)
            year_count += 1

        print(f"    ✓ Loaded {year_count} days for {yr}")

    # -------------------------------------------------------
    # VALIDATION & ENSEMBLE STATISTICS
    # -------------------------------------------------------
    if not psd_HR_all:
        print("\n✗ ERROR: No valid data found. Check file paths and contents.")
        return

    n_samples = len(psd_HR_all)
    print(f"\n✓ Total samples loaded: {n_samples} days")

    # Convert to arrays (pad to same length if necessary)
    max_len = max(max(len(p) for p in psd_HR_all), max(len(p) for p in psd_DDPM_all))

    def pad_array_list(lst, target_len):
        arr = np.full((len(lst), target_len), np.nan, dtype=float)
        for i, v in enumerate(lst):
            L = len(v)
            arr[i, :L] = v
        return arr

    psd_HR_array = pad_array_list(psd_HR_all, max_len)
    psd_DDPM_array = pad_array_list(psd_DDPM_all, max_len)

    print("  Computing ensemble statistics...")

    # Mean and confidence intervals (95% CI)
    psd_HR_mean = np.nanmean(psd_HR_array, axis=0)
    psd_HR_std = np.nanstd(psd_HR_array, axis=0)
    psd_HR_p05 = np.nanpercentile(psd_HR_array, 5, axis=0)
    psd_HR_p95 = np.nanpercentile(psd_HR_array, 95, axis=0)

    psd_DDPM_mean = np.nanmean(psd_DDPM_array, axis=0)
    psd_DDPM_std = np.nanstd(psd_DDPM_array, axis=0)
    psd_DDPM_p05 = np.nanpercentile(psd_DDPM_array, 5, axis=0)
    psd_DDPM_p95 = np.nanpercentile(psd_DDPM_array, 95, axis=0)

    # -------------------------------------------------------
    # PHYSICAL AXES CALCULATION
    # -------------------------------------------------------
    k_index = np.arange(len(psd_HR_mean))
    valid_k = k_index > 0

    domain_width_km = YOUR_DOMAIN_N_PIXELS * YOUR_GRID_SPACING_KM
    wavenumber_km = k_index[valid_k] / domain_width_km

    nyquist_wavenumber = 1.0 / (2.0 * YOUR_GRID_SPACING_KM)
    res_limit_wavenum = 1.0 / RESOLUTION_LIMIT_KM

    # Truncate all data at Nyquist
    valid_nyquist = wavenumber_km <= nyquist_wavenumber
    wavenumber_km = wavenumber_km[valid_nyquist]

    psd_HR_mean = psd_HR_mean[valid_k][valid_nyquist]
    psd_HR_std = psd_HR_std[valid_k][valid_nyquist]
    psd_HR_p05 = psd_HR_p05[valid_k][valid_nyquist]
    psd_HR_p95 = psd_HR_p95[valid_k][valid_nyquist]

    psd_DDPM_mean = psd_DDPM_mean[valid_k][valid_nyquist]
    psd_DDPM_std = psd_DDPM_std[valid_k][valid_nyquist]
    psd_DDPM_p05 = psd_DDPM_p05[valid_k][valid_nyquist]
    psd_DDPM_p95 = psd_DDPM_p95[valid_k][valid_nyquist]

    # -------------------------------------------------------
    # SPECTRAL METRICS & STATISTICS
    # -------------------------------------------------------
    print("  Computing spectral metrics...")

    metrics_HR = compute_spectrum_metrics(wavenumber_km, psd_HR_array[:, valid_k][:, valid_nyquist], "True MSWX")
    metrics_DDPM = compute_spectrum_metrics(wavenumber_km, psd_DDPM_array[:, valid_k][:, valid_nyquist], "DDPM")

    # -------------------------------------------------------
    # SAVE DATA TO CSV
    # -------------------------------------------------------
    print(f"\nSaving data to {DATA_FILE}...")
    try:
        header = (
            "Wavenumber_km,"
            "PSD_True_mean,PSD_True_p05,PSD_True_p95,"
            "PSD_DDPM_mean,PSD_DDPM_p05,PSD_DDPM_p95"
        )
        data_to_save = np.column_stack(
            (
                wavenumber_km,
                psd_HR_mean,
                psd_HR_p05,
                psd_HR_p95,
                psd_DDPM_mean,
                psd_DDPM_p05,
                psd_DDPM_p95,
            )
        )
        np.savetxt(DATA_FILE, data_to_save, delimiter=",", header=header, comments="")
        print(f"✓ Data saved to {DATA_FILE}")
    except Exception as e:
        print(f"✗ Error saving data: {e}")

    # -------------------------------------------------------
    # SAVE STATISTICS TO TEXT FILE
    # -------------------------------------------------------
    print(f"Saving statistics to {STATS_FILE}...")
    try:
        with open(STATS_FILE, "w") as f:
            f.write("=" * 80 + "\n")
            f.write("POWER SPECTRAL DENSITY ANALYSIS - STATISTICAL SUMMARY\n")
            f.write("=" * 80 + "\n\n")

            f.write(f"Analysis Period: 2003-2014 ({n_samples} days)\n")
            f.write(f"Domain: {YOUR_DOMAIN_N_PIXELS}×{YOUR_DOMAIN_N_PIXELS} pixels\n")
            f.write(f"Grid spacing: {YOUR_GRID_SPACING_KM} km\n")
            f.write(f"Domain size: {domain_width_km:.1f} km\n")
            f.write(f"Nyquist wavenumber: {nyquist_wavenumber:.6f} km⁻¹\n")
            f.write(f"Resolution limit (1°): {res_limit_wavenum:.6f} km⁻¹\n\n")

            for metrics in [metrics_HR, metrics_DDPM]:
                f.write(f"\n{'-' * 80}\n")
                f.write(f"METHOD: {metrics['method']}\n")
                f.write(f"{'-' * 80}\n")
                f.write(f"Total integrated energy: {metrics['total_energy']:.4e} K²·km\n")
                if metrics["peak_wavenumber"] and metrics["peak_wavenumber"] > 0:
                    f.write(
                        f"Peak wavenumber: {metrics['peak_wavenumber']:.6f} km⁻¹ "
                        f"(wavelength: {1/metrics['peak_wavenumber']:.2f} km)\n"
                    )
                else:
                    f.write("Peak wavenumber: N/A\n")
                f.write(f"Peak spectral density: {metrics['peak_psd']:.4e} K²·km\n\n")
                f.write(f"Synoptic-scale regime (0.001-0.01 km⁻¹):\n")
                f.write(f"  Slope: {metrics['slope_synoptic']:.3f} (R²={metrics['r2_synoptic']:.4f})\n")
                f.write(f"Mesoscale regime (0.01-0.1 km⁻¹):\n")
                f.write(f"  Slope: {metrics['slope_mesoscale']:.3f} (R²={metrics['r2_mesoscale']:.4f})\n")

        print(f"✓ Statistics saved to {STATS_FILE}")
    except Exception as e:
        print(f"✗ Error saving statistics: {e}")

    # =====================================================================
    # MAIN PUBLICATION-QUALITY PLOT (only panel 1, no reference slopes)
    # =====================================================================
    print(f"\nGenerating publication-quality figure (main panel only, no refs)...")

    fig, ax = plt.subplots(figsize=FIGSIZE)

    ax.loglog(wavenumber_km, psd_HR_mean, "k-", linewidth=2.5, label="True MSWX (0.1°)", zorder=5)

    # Keep the CI for DDPM to show model stability
    ax.fill_between(wavenumber_km, psd_DDPM_p05, psd_DDPM_p95, color="dodgerblue", alpha=0.18, linewidth=0)
    ax.loglog(wavenumber_km, psd_DDPM_mean, "b-", linewidth=2.5, label="DDPM Downscaling", zorder=3)

    # Annotations
    y_bottom, y_top = ax.get_ylim()

    # Resolution limit (1°)
    ax.axvline(res_limit_wavenum, color="red", linestyle="--", alpha=0.6, linewidth=1.8)
    ax.text(res_limit_wavenum * 1.2, y_bottom * 2.0, "1° Limit", color="red", alpha=0.9, ha="left",
            fontsize=ANNOT_FS, weight="bold")

    # Nyquist limit
    nyquist_wavenumber = 1.0 / (2.0 * YOUR_GRID_SPACING_KM)
    ax.axvline(nyquist_wavenumber, color="purple", linestyle="--", alpha=0.6, linewidth=1.8)
    ax.text(nyquist_wavenumber * 0.7, y_top * 0.4, "Nyquist", color="purple", alpha=0.9, ha="right",
            fontsize=ANNOT_FS, weight="bold")

    # Scale regimes labels
    ax.text(res_limit_wavenum * 0.03, y_top * 0.15, "Synoptic Scales", color="black", alpha=0.8,
            fontsize=ANNOT_FS, weight="bold")
    ax.text(res_limit_wavenum * 1.5, y_top * 0.15, "Mesoscales", color="black", alpha=0.8,
            fontsize=ANNOT_FS, weight="bold")

    # Formatting: increased font sizes, no main title
    ax.set_xlabel(r"Wavenumber ($\mathrm{km^{-1}}$)", fontsize=LABEL_FS, weight="bold")
    ax.set_ylabel(r"PSD ($\mathrm{K^2 \cdot km}$)", fontsize=LABEL_FS, weight="bold")

    ax.legend(loc="lower left", fontsize=LEGEND_FS, framealpha=0.95, edgecolor="black")
    ax.grid(True, which="both", linestyle="-", linewidth=0.6, alpha=0.35)
    ax.set_axisbelow(True)

    # Secondary axis (Wavelength) with larger label
    def k_to_wav(k):
        return np.where(k > 0, 1.0 / k, np.inf)

    def wav_to_k(wav):
        return np.where(wav > 0, 1.0 / wav, np.inf)

    ax_sec = ax.secondary_xaxis("top", functions=(k_to_wav, wav_to_k))
    ax_sec.set_xlabel(r"Wavelength ($\mathrm{km}$)", fontsize=LABEL_FS, weight="bold")

    # Improved tick formatting + larger tick labels
    ax.xaxis.set_major_locator(ticker.LogLocator(base=10, numticks=8))
    ax.xaxis.set_major_formatter(ticker.LogFormatterSciNotation())
    ax.xaxis.set_minor_locator(ticker.LogLocator(base=10, subs=np.arange(2, 10) * 0.1, numticks=12))

    ax.yaxis.set_major_locator(ticker.LogLocator(base=10, numticks=8))
    ax.yaxis.set_major_formatter(ticker.LogFormatterSciNotation())

    ax.tick_params(axis="both", which="major", labelsize=TICK_FS)
    ax.tick_params(axis="both", which="minor", labelsize=TICK_FS - 2)

    # Set x-axis limits
    try:
        ax.set_xlim(wavenumber_km.min(), nyquist_wavenumber)
    except Exception:
        pass

    plt.tight_layout()
    plt.savefig(PLOT_FILE_MAIN, dpi=300, bbox_inches="tight", facecolor="white")  # PNG, 300 dpi
    print(f"✓ Main plot saved to {PLOT_FILE_MAIN}")

    plt.show()

    print("\n" + "=" * 70)
    print("✓ ANALYSIS COMPLETE")
    print("=" * 70)
    print(f"\nOutput files:")
    print(f"  - {PLOT_FILE_MAIN} (PNG, 300 dpi)")
    print(f"  - {DATA_FILE} (Numerical data)")
    print(f"  - {STATS_FILE} (Statistical summary)")


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Bias Distribution & Q-Q Plot Analysis (publication quality)

- Keeps only two panels: (a) spatial bias distribution (KDE + normal fit + histogram)
  and (b) Quantile-Quantile plot (with 1:1 and fitted line).
- No overall title/heading printed on the figure.
- Increased font sizes for legibility.
- Units changed to degrees Celsius (°C).
- Saves the distribution figure as PNG (300 dpi).
"""

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import glob
import os
import rioxarray
import geopandas as gpd
import statsmodels.api as sm
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.ndimage import uniform_filter1d
import warnings

warnings.filterwarnings('ignore')

# =====================================================================
# CONFIGURATION
# =====================================================================

# Paths (update if needed)
HR_MSWX = "<SET YOUR PATH HERE>"
HR_DDPM = "<SET YOUR PATH HERE>"
INDIA_SHP = "<SET YOUR PATH HERE>"

# Analysis parameters (keep as in your pipeline)
TEST_START = 2009219  # 2009-08-07 (YYYYDOY)
TEST_END = 2014365

# Grid spacing (km)
GRID_SPACING_KM = 10.5924

# Output files
PLOT_DIST_FILE = "bias_distribution_and_qq.png"   # PNG at 300 dpi
STATS_FILE = "bias_analysis_statistics.txt"
DATA_FILE = "bias_analysis_data.csv"

# Font sizes for legibility
LABEL_FS = 16
TICK_FS = 14
LEGEND_FS = 13
ANNOT_FS = 13
FIGSIZE = (14, 7)

# =====================================================================
# Helper functions (unchanged behavior, minor name/unit adjustments)
# =====================================================================

def get_file_date(fname):
    base = os.path.basename(fname)
    base = base.replace("_ddpm", "").replace(".nc", "")
    try:
        return int(base)
    except Exception:
        # fallback: extract 7/8/10-digit date-like token
        import re
        m = re.search(r"(\d{7,8})", base)
        return int(m.group(1)) if m else None

def doy_to_datetime(yydoy):
    year = int(str(yydoy)[:4])
    doy = int(str(yydoy)[4:])
    return pd.to_datetime(f"{year}-{doy:03d}", format="%Y-%j")

def filter_period(files, start=2009219, end=2014365):
    valid = []
    for f in files:
        try:
            d = get_file_date(f)
            if d is None:
                continue
            if start <= d <= end:
                valid.append(f)
        except:
            continue
    return valid

def compute_effective_dof(field_2d, grid_spacing_km, decorr_scale_km=300):
    # simplified robust implementation (returns effective DOF estimate)
    if field_2d is None or np.isnan(field_2d).all():
        return {'total_dof':0,'effective_dof':0,'reduction_factor':1.0,'decorr_scale_km':decorr_scale_km}
    ny, nx = np.array(field_2d.shape)
    total = ny * nx
    domain_w = nx * grid_spacing_km
    domain_h = ny * grid_spacing_km
    n_indep_x = max(1, domain_w / decorr_scale_km)
    n_indep_y = max(1, domain_h / decorr_scale_km)
    eff = max(1, int(n_indep_x * n_indep_y))
    reduction = total / eff if eff>0 else 1.0
    return {'total_dof': total, 'effective_dof': eff, 'reduction_factor': reduction, 'decorr_scale_km': decorr_scale_km}

def test_normality(data, name=""):
    """
    Robust wrapper for normality tests. Returns dict with Shapiro p, Anderson stat,
    KS vs fitted normal, Jarque-Bera, and boolean is_normal flag (conservative).
    """
    arr = np.asarray(data).ravel()
    arr = arr[np.isfinite(arr)]
    res = {'name': name, 'n_samples': len(arr)}
    if len(arr) == 0:
        res.update({'shapiro_stat': np.nan, 'shapiro_p': np.nan, 'anderson_stat': np.nan,
                    'anderson_crit': [], 'ks_stat': np.nan, 'ks_p': np.nan,
                    'jb_stat': np.nan, 'jb_p': np.nan, 'is_normal': False})
        return res

    # Shapiro (only for n <= 5000; otherwise return nan)
    try:
        if len(arr) <= 5000:
            sw_stat, sw_p = stats.shapiro(arr)
        else:
            sw_stat, sw_p = (np.nan, np.nan)
    except Exception:
        sw_stat, sw_p = (np.nan, np.nan)

    # Anderson-Darling
    try:
        ad = stats.anderson(arr, dist='norm')
        ad_stat = ad.statistic
        ad_crit = list(ad.critical_values)
    except Exception:
        ad_stat = np.nan
        ad_crit = []

    # KS against fitted normal
    try:
        mu, sigma = np.mean(arr), np.std(arr, ddof=0)
        ks_stat, ks_p = stats.kstest(arr, 'norm', args=(mu, sigma))
    except Exception:
        ks_stat, ks_p = (np.nan, np.nan)

    # Jarque-Bera
    try:
        jb_stat, jb_p = stats.jarque_bera(arr)
    except Exception:
        jb_stat, jb_p = (np.nan, np.nan)

    # Conservative is_normal: require both (if available) Shapiro p>0.05 or KS p>0.05
    is_normal = False
    if not np.isnan(sw_p):
        is_normal = sw_p > 0.05
    elif not np.isnan(ks_p):
        is_normal = ks_p > 0.05

    res.update({
        'shapiro_stat': sw_stat, 'shapiro_p': sw_p,
        'anderson_stat': ad_stat, 'anderson_crit': ad_crit,
        'ks_stat': ks_stat, 'ks_p': ks_p,
        'jb_stat': jb_stat, 'jb_p': jb_p,
        'is_normal': bool(is_normal),
    })
    return res

def fit_distributions(data):
    from scipy.stats import norm, lognorm, gamma
    data = np.asarray(data); data = data[np.isfinite(data)]
    results = {}
    if len(data)<10:
        return results
    try:
        mu, sigma = norm.fit(data)
        ks_norm = stats.kstest(data, 'norm', args=(mu, sigma))[0]
        results['normal'] = {'params':{'mu':mu,'sigma':sigma}, 'ks_stat':ks_norm}
    except:
        pass
    try:
        s, loc, scale = lognorm.fit(data)
        ks_lognorm = stats.kstest(data, 'lognorm', args=(s, loc, scale))[0]
        results['lognormal'] = {'params':{'s':s,'loc':loc,'scale':scale}, 'ks_stat':ks_lognorm}
    except:
        pass
    try:
        a, loc, scale = gamma.fit(data)
        ks_gamma = stats.kstest(data, 'gamma', args=(a, loc, scale))[0]
        results['gamma'] = {'params':{'a':a,'loc':loc,'scale':scale}, 'ks_stat':ks_gamma}
    except:
        pass
    if results:
        best = min(results.items(), key=lambda x: x[1]['ks_stat'])[0]
        results['best_fit'] = best
    return results

def compute_bias_metrics(mswx_vals, ddpm_vals, bias_vals, effective_dof=None):
    """
    Compute comprehensive bias and error metrics (units: °C).
    - mswx_vals, ddpm_vals, bias_vals: 1D numpy arrays (finite values)
    - effective_dof: int or None (if provided, used to compute SE)
    Returns dict of metrics.
    """
    # Ensure numpy arrays and drop NaNs consistently
    mswx_vals = np.asarray(mswx_vals).ravel()
    ddpm_vals = np.asarray(ddpm_vals).ravel()
    bias_vals = np.asarray(bias_vals).ravel()

    mask = np.isfinite(mswx_vals) & np.isfinite(ddpm_vals) & np.isfinite(bias_vals)
    mswx = mswx_vals[mask]
    ddpm = ddpm_vals[mask]
    bias = bias_vals[mask]

    n = len(bias)
    if n == 0:
        return {k: np.nan for k in [
            'mean_bias','median_bias','std_bias','mae','rmse','se','ci_95',
            'correlation','min_bias','max_bias','bias_by_quantile'
        ]}

    # Basic statistics
    mean_bias = float(np.mean(bias))
    median_bias = float(np.median(bias))
    std_bias = float(np.std(bias, ddof=0))   # population std (matches earlier code)
    mae = float(np.mean(np.abs(bias)))
    rmse = float(np.sqrt(np.mean(bias**2)))

    # Effective sample size for SE
    if effective_dof is None or not np.isfinite(effective_dof) or effective_dof < 1:
        n_eff = n
    else:
        n_eff = max(1, int(effective_dof))

    se = float(std_bias / np.sqrt(n_eff))
    ci_lower = mean_bias - 1.96 * se
    ci_upper = mean_bias + 1.96 * se

    # Pearson correlation (safe)
    try:
        if len(mswx) < 2 or np.nanstd(mswx) == 0 or np.nanstd(ddpm) == 0:
            corr = np.nan
        else:
            corr = float(np.corrcoef(mswx, ddpm)[0, 1])
    except Exception:
        corr = np.nan

    # Quantile biases
    quantiles = [0.05, 0.25, 0.5, 0.75, 0.95]
    bias_by_quantile = {}
    for q in quantiles:
        q_mswx = np.percentile(mswx, q * 100)
        q_ddpm = np.percentile(ddpm, q * 100)
        bias_by_quantile[q] = float(q_ddpm - q_mswx)

    metrics = {
        'mean_bias': mean_bias,
        'median_bias': median_bias,
        'std_bias': std_bias,
        'mae': mae,
        'rmse': rmse,
        'se': se,
        'ci_95': (ci_lower, ci_upper),
        'correlation': corr,
        'min_bias': float(np.min(bias)),
        'max_bias': float(np.max(bias)),
        'bias_by_quantile': bias_by_quantile,
        'n_samples': n,
        'n_eff': n_eff,
    }
    return metrics

def compute_qq_statistics(mswx_vals, ddpm_vals, effective_dof=None):
    """
    Compute Q-Q statistics between mswx_vals and ddpm_vals.
    Returns (stats_dict, q_mswx, q_ddpm).
    stats_dict contains slope, intercept, correlation, r_squared, rmse_qq, p5_bias, p95_bias, slope_ci.
    - slope_ci is (lower, upper) or None if not computable.
    """
    mswx = np.asarray(mswx_vals).ravel()
    ddpm = np.asarray(ddpm_vals).ravel()
    mask = np.isfinite(mswx) & np.isfinite(ddpm)
    mswx = mswx[mask]
    ddpm = ddpm[mask]

    if len(mswx) < 4:
        return ({'slope':np.nan,'intercept':np.nan,'correlation':np.nan,'r_squared':np.nan,
                 'rmse_qq':np.nan,'p5_bias':np.nan,'p95_bias':np.nan,'slope_ci':None}, np.array([]), np.array([]))

    percentiles = np.arange(1, 100)  # 1..99
    q_m = np.percentile(mswx, q=percentiles)
    q_d = np.percentile(ddpm, q=percentiles)

    # OLS fit q_d = slope * q_m + intercept
    slope, intercept = np.polyfit(q_m, q_d, 1)

    # predicted and residuals
    yhat = slope * q_m + intercept
    residuals = q_d - yhat

    # RMSE of QQ
    rmse_qq = float(np.sqrt(np.mean(residuals**2)))

    # correlation between quantiles
    try:
        corr = float(np.corrcoef(q_m, q_d)[0, 1])
    except Exception:
        corr = np.nan

    # R-squared (standard)
    ss_res = np.sum((q_d - yhat)**2)
    ss_tot = np.sum((q_d - np.mean(q_d))**2)
    r_squared = float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan

    # extreme quantile biases
    p5_bias = float(np.percentile(ddpm, 5) - np.percentile(mswx, 5))
    p95_bias = float(np.percentile(ddpm, 95) - np.percentile(mswx, 95))

    # slope CI using standard formula from linear regression:
    # se_slope = sqrt( MSE / sum((x - xbar)^2) )
    n_points = len(q_m)  # typically 99
    mse = np.sum(residuals**2) / max(1, (n_points - 2))
    sxx = np.sum((q_m - np.mean(q_m))**2)
    slope_ci = None
    if sxx > 0 and n_points > 2:
        se_slope = np.sqrt(mse / sxx)
        # degrees of freedom: prefer effective_dof - 2 if provided & reasonable, else use n_points - 2
        if effective_dof is not None and effective_dof > 3:
            dof = max(1, int(effective_dof) - 2)
        else:
            dof = n_points - 2
        try:
            tval = stats.t.ppf(0.975, dof)
            slope_ci = (slope - tval * se_slope, slope + tval * se_slope)
        except Exception:
            slope_ci = None

    stats_qq = {
        'slope': float(slope),
        'intercept': float(intercept),
        'correlation': corr,
        'r_squared': r_squared,
        'rmse_qq': rmse_qq,
        'p5_bias': p5_bias,
        'p95_bias': p95_bias,
        'slope_ci': slope_ci,
        'n_points': n_points,
    }
    return stats_qq, q_m, q_d

# =====================================================================
# MAIN EXECUTION
# =====================================================================

def main():
    # 1. load file lists and filter by period
    f_mswx = sorted(glob.glob(os.path.join(HR_MSWX, "*.nc")))
    f_ddpm = sorted(glob.glob(os.path.join(HR_DDPM, "*_ddpm.nc")))
    f_mswx = filter_period(f_mswx, TEST_START, TEST_END)
    f_ddpm = filter_period(f_ddpm, TEST_START, TEST_END)

    if not f_mswx or not f_ddpm:
        raise RuntimeError("No files found for the requested period. Check paths and TEST_START/END.")

    # 2. open datasets (nested; assume files ordered)
    ds_mswx = xr.open_mfdataset(f_mswx, combine="nested", concat_dim="time", engine="netcdf4", parallel=False)
    ds_ddpm = xr.open_mfdataset(f_ddpm, combine="nested", concat_dim="time", engine="netcdf4", parallel=False)

    # assign time if missing (from filenames)
    if "time" not in ds_mswx.coords:
        times = [doy_to_datetime(get_file_date(f)) for f in f_mswx]
        ds_mswx = ds_mswx.assign_coords(time=("time", times))
    if "time" not in ds_ddpm.coords:
        times = [doy_to_datetime(get_file_date(f)) for f in f_ddpm]
        ds_ddpm = ds_ddpm.assign_coords(time=("time", times))

    # 3. daily bias and some temporal metrics (kept for logging)
    bias_daily = ds_ddpm["air_temperature"] - ds_mswx["air_temperature"]
    annual_rmse = bias_daily.groupby("time.year").apply(lambda x: np.sqrt(np.mean(x**2)))
    seasonal_rmse = bias_daily.groupby("time.season").apply(lambda x: np.sqrt(np.mean(x**2)))

    # 4. spatial annual mean fields averaged across years and then mean-of-years
    overall_mswx = ds_mswx["air_temperature"].groupby("time.year").mean("time").mean("year")
    overall_ddpm = ds_ddpm["air_temperature"].groupby("time.year").mean("time").mean("year")
    overall_bias = overall_ddpm - overall_mswx

    # 5. clip to India
    overall_mswx = overall_mswx.rename({"lat":"y","lon":"x"})
    overall_ddpm = overall_ddpm.rename({"lat":"y","lon":"x"})
    overall_bias = overall_bias.rename({"lat":"y","lon":"x"})
    overall_mswx.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True); overall_mswx.rio.write_crs("EPSG:4326", inplace=True)
    overall_ddpm.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True); overall_ddpm.rio.write_crs("EPSG:4326", inplace=True)
    overall_bias.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True); overall_bias.rio.write_crs("EPSG:4326", inplace=True)

    gdf = gpd.read_file(INDIA_SHP)
    overall_mswx_clipped = overall_mswx.rio.clip(gdf.geometry, gdf.crs)
    overall_ddpm_clipped = overall_ddpm.rio.clip(gdf.geometry, gdf.crs)
    overall_bias_clipped = overall_bias.rio.clip(gdf.geometry, gdf.crs)

    mswx_vals = overall_mswx_clipped.values.flatten(); mswx_vals = mswx_vals[np.isfinite(mswx_vals)]
    ddpm_vals = overall_ddpm_clipped.values.flatten(); ddpm_vals = ddpm_vals[np.isfinite(ddpm_vals)]
    bias_vals = overall_bias_clipped.values.flatten(); bias_vals = bias_vals[np.isfinite(bias_vals)]

    # 6. spatial autocorr / effective DOF
    dof_info = compute_effective_dof(overall_bias_clipped.values, GRID_SPACING_KM)
    effective_dof = dof_info['effective_dof']

    # 7. distribution tests & fits
    norm_bias = test_normality(bias_vals, "Bias")
    norm_mswx = test_normality(mswx_vals, "MSWX")
    norm_ddpm = test_normality(ddpm_vals, "DDPM")
    dist_fits = fit_distributions(bias_vals)
    bias_metrics = compute_bias_metrics(mswx_vals, ddpm_vals, bias_vals, effective_dof)

    # 8. Q-Q statistics
    qq_stats, q_mswx, q_ddpm = compute_qq_statistics(mswx_vals, ddpm_vals, effective_dof)

    # 9. significance tests
    t_stat, t_p = stats.ttest_1samp(bias_vals, popmean=0)
    ks_stat, ks_p = stats.ks_2samp(mswx_vals, ddpm_vals)

    # 10. save CSV of quantile pairs (for external inspection)
    try:
        data_out = np.column_stack((q_mswx, q_ddpm, q_ddpm - q_mswx))
        np.savetxt(DATA_FILE, data_out, delimiter=",", header="MSWX_Quantile_C,DDPM_Quantile_C,Quantile_Bias_C", comments="")
    except Exception as e:
        print("Warning: failed to save data CSV:", e)

    # 11. save textual statistics (units in °C)
    try:
        with open(STATS_FILE, "w") as f:
            f.write("BIAS ANALYSIS (DDPM vs MSWX)\n")
            f.write("="*60 + "\n")
            f.write(f"Analysis period (files): {len(f_mswx)} MSWX files, {len(f_ddpm)} DDPM files\n")
            f.write(f"Valid spatial points (after clipping): {len(bias_vals):,}\n")
            f.write(f"Total spatial DOF: {dof_info['total_dof']:,}\n")
            f.write(f"Effective DOF: {effective_dof:,}\n")
            f.write(f"Reduction factor: {dof_info['reduction_factor']:.2f}x (decorr scale {dof_info['decorr_scale_km']} km)\n\n")
            f.write("BASIC STATISTICS (units: °C)\n")
            f.write(f"Mean bias: {bias_metrics['mean_bias']:.4f} °C\n")
            f.write(f"95% CI: [{bias_metrics['ci_95'][0]:.4f}, {bias_metrics['ci_95'][1]:.4f}] °C\n")
            f.write(f"Median bias: {bias_metrics['median_bias']:.4f} °C\n")
            f.write(f"Std dev (bias): {bias_metrics['std_bias']:.4f} °C\n")
            f.write(f"MAE: {bias_metrics['mae']:.4f} °C\n")
            f.write(f"RMSE: {bias_metrics['rmse']:.4f} °C\n")
            f.write(f"Correlation (spatial means): {bias_metrics['correlation']:.6f}\n\n")
            f.write("Q-Q STATISTICS\n")
            f.write(f"Slope: {qq_stats['slope']:.4f}\n")
            f.write(f"Intercept: {qq_stats['intercept']:.4f}\n")
            f.write(f"Correlation (quantiles): {qq_stats['correlation']:.6f}\n")
            f.write(f"R^2: {qq_stats['r_squared']:.6f}\n")
            f.write(f"5% quantile bias: {qq_stats['p5_bias']:.4f} °C\n")
            f.write(f"95% quantile bias: {qq_stats['p95_bias']:.4f} °C\n\n")
            f.write("SIGNIFICANCE TESTS\n")
            f.write(f"One-sample t-test (H0: mean bias = 0): t={t_stat:.4f}, p={t_p:.6e}\n")
            f.write(f"KS two-sample (MSWX vs DDPM): KS={ks_stat:.4f}, p={ks_p:.6e}\n")
    except Exception as e:
        print("Warning: failed to save stats file:", e)

    # =================================================================
    # PLOTTING: Only two panels (a) bias KDE+hist+normal fit and (b) Q-Q
    # =================================================================
    fig = plt.figure(figsize=FIGSIZE)
    gs = GridSpec(1, 2, figure=fig, wspace=0.35)

    # Panel (a): Bias KDE + histogram + normal fit
    ax1 = fig.add_subplot(gs[0, 0])
    x_range = np.linspace(np.min(bias_vals), np.max(bias_vals), 300)

    # KDE as a LINE (no filled area)
    try:
        kde = gaussian_kde(bias_vals)
        kde_vals = kde(x_range)
        ax1.plot(x_range, kde_vals, color='steelblue', linewidth=2.5, label='KDE (line)')
    except Exception:
        # fallback to nothing if KDE fails
        pass

    # Normal fit (pdf scaled to density) as dashed red line
    mu, sigma = stats.norm.fit(bias_vals)
    normal_pdf = stats.norm.pdf(x_range, mu, sigma)
    ax1.plot(x_range, normal_pdf, 'r--', linewidth=2, label='Normal fit')

    # Histogram as faint bars (kept for context)
    ax1.hist(bias_vals, bins=50, density=True, alpha=0.25, color='gray', edgecolor='k', label='Histogram')

    # Zero and mean lines
    ax1.axvline(0.0, color='black', linestyle='-', linewidth=1.4)
    ax1.axvline(bias_metrics['mean_bias'], color='darkblue', linestyle='--', linewidth=1.6,
                label=f"Mean = {bias_metrics['mean_bias']:.3f} °C")

    ax1.set_xlabel('Bias (DDPM − MSWX) (°C)', fontsize=LABEL_FS, fontweight='bold')
    ax1.set_ylabel('Density', fontsize=LABEL_FS, fontweight='bold')
    ax1.legend(fontsize=LEGEND_FS, loc='upper right')
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(axis='both', labelsize=TICK_FS)

    # stats_text = f"μ = {bias_metrics['mean_bias']:.4f} °C\nσ = {bias_metrics['std_bias']:.4f} °C\nRMSE = {bias_metrics['rmse']:.4f} °C"
    # ax1.text(0.02, 0.98, stats_text, transform=ax1.transAxes, fontsize=ANNOT_FS,
    #         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))


    # Panel (b): Q-Q plot
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.scatter(q_mswx, q_ddpm, alpha=0.6, s=28, color='steelblue', label='Quantile pairs')

    # 1:1 line and fitted line
    min_val = min(q_mswx.min(), q_ddpm.min())
    max_val = max(q_mswx.max(), q_ddpm.max())
    ax2.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1.6, label='1:1 reference')

    fitted_line = qq_stats['slope'] * q_mswx + qq_stats['intercept']
    ax2.plot(q_mswx, fitted_line, color='green', linewidth=2.0, label=f"Fit: y={qq_stats['slope']:.3f}x+{qq_stats['intercept']:.3f}")

    ax2.set_xlabel('MSWX Quantiles (°C)', fontsize=LABEL_FS, fontweight='bold')
    ax2.set_ylabel('DDPM Quantiles (°C)', fontsize=LABEL_FS, fontweight='bold')
    ax2.legend(fontsize=LEGEND_FS)
    ax2.grid(True, alpha=0.3)
    ax2.set_aspect('equal', adjustable='box')
    ax2.tick_params(axis='both', labelsize=TICK_FS)

    # Tight layout and save as PNG (300 dpi)
    plt.tight_layout()
    plt.savefig(PLOT_DIST_FILE, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Saved figure: {PLOT_DIST_FILE}")

    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Bias Distribution & Q-Q Plot Analysis (publication quality)

NEW: Now includes BOTH spatial and temporal Q-Q plots in a 3-panel figure
- Panel (a): Spatial bias distribution (KDE + normal fit + histogram)
- Panel (b): Spatial Q-Q plot (climatological mean field)
- Panel (c): Temporal Q-Q plot (daily domain-averaged temperatures)
"""

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import glob
import os
import rioxarray
import geopandas as gpd
import statsmodels.api as sm
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.ndimage import uniform_filter1d
import warnings

warnings.filterwarnings('ignore')

# =====================================================================
# CONFIGURATION
# =====================================================================

# Paths (update if needed)
HR_MSWX = "<SET YOUR PATH HERE>"
HR_DDPM = "<SET YOUR PATH HERE>"
INDIA_SHP = "<SET YOUR PATH HERE>"

# Analysis parameters
TEST_START = 2009219  # 2009-08-07 (YYYYDOY)
TEST_END = 2014365

# Grid spacing (km)
GRID_SPACING_KM = 10.5924

# Output files
PLOT_DIST_FILE = "bias_distribution_and_qq_3panel.png"   # PNG at 300 dpi
STATS_FILE = "bias_analysis_statistics.txt"
DATA_FILE = "bias_analysis_data.csv"

# Font sizes for legibility
LABEL_FS = 16
TICK_FS = 14
LEGEND_FS = 12
ANNOT_FS = 13
FIGSIZE = (18, 6)  # Wider for 3 panels

# =====================================================================
# Helper functions (unchanged)
# =====================================================================

def get_file_date(fname):
    base = os.path.basename(fname)
    base = base.replace("_ddpm", "").replace(".nc", "")
    try:
        return int(base)
    except Exception:
        import re
        m = re.search(r"(\d{7,8})", base)
        return int(m.group(1)) if m else None

def doy_to_datetime(yydoy):
    year = int(str(yydoy)[:4])
    doy = int(str(yydoy)[4:])
    return pd.to_datetime(f"{year}-{doy:03d}", format="%Y-%j")

def filter_period(files, start=2009219, end=2014365):
    valid = []
    for f in files:
        try:
            d = get_file_date(f)
            if d is None:
                continue
            if start <= d <= end:
                valid.append(f)
        except:
            continue
    return valid

def compute_effective_dof(field_2d, grid_spacing_km, decorr_scale_km=300):
    if field_2d is None or np.isnan(field_2d).all():
        return {'total_dof':0,'effective_dof':0,'reduction_factor':1.0,'decorr_scale_km':decorr_scale_km}
    ny, nx = np.array(field_2d.shape)
    total = ny * nx
    domain_w = nx * grid_spacing_km
    domain_h = ny * grid_spacing_km
    n_indep_x = max(1, domain_w / decorr_scale_km)
    n_indep_y = max(1, domain_h / decorr_scale_km)
    eff = max(1, int(n_indep_x * n_indep_y))
    reduction = total / eff if eff>0 else 1.0
    return {'total_dof': total, 'effective_dof': eff, 'reduction_factor': reduction, 'decorr_scale_km': decorr_scale_km}

def compute_qq_statistics(mswx_vals, ddpm_vals, effective_dof=None):
    """
    Compute Q-Q statistics between mswx_vals and ddpm_vals.
    Returns (stats_dict, q_mswx, q_ddpm).
    """
    mswx = np.asarray(mswx_vals).ravel()
    ddpm = np.asarray(ddpm_vals).ravel()
    mask = np.isfinite(mswx) & np.isfinite(ddpm)
    mswx = mswx[mask]
    ddpm = ddpm[mask]

    if len(mswx) < 4:
        return ({'slope':np.nan,'intercept':np.nan,'correlation':np.nan,'r_squared':np.nan,
                 'rmse_qq':np.nan,'p5_bias':np.nan,'p95_bias':np.nan,'slope_ci':None}, 
                np.array([]), np.array([]))

    percentiles = np.arange(1, 100)  # 1..99
    q_m = np.percentile(mswx, q=percentiles)
    q_d = np.percentile(ddpm, q=percentiles)

    # OLS fit q_d = slope * q_m + intercept
    slope, intercept = np.polyfit(q_m, q_d, 1)
    yhat = slope * q_m + intercept
    residuals = q_d - yhat
    rmse_qq = float(np.sqrt(np.mean(residuals**2)))

    # correlation between quantiles
    try:
        corr = float(np.corrcoef(q_m, q_d)[0, 1])
    except Exception:
        corr = np.nan

    # R-squared
    ss_res = np.sum((q_d - yhat)**2)
    ss_tot = np.sum((q_d - np.mean(q_d))**2)
    r_squared = float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan

    # extreme quantile biases
    p5_bias = float(np.percentile(ddpm, 5) - np.percentile(mswx, 5))
    p95_bias = float(np.percentile(ddpm, 95) - np.percentile(mswx, 95))

    stats_qq = {
        'slope': float(slope),
        'intercept': float(intercept),
        'correlation': corr,
        'r_squared': r_squared,
        'rmse_qq': rmse_qq,
        'p5_bias': p5_bias,
        'p95_bias': p95_bias,
        'n_points': len(q_m),
    }
    return stats_qq, q_m, q_d

# =====================================================================
# MAIN EXECUTION
# =====================================================================

def main():
    # 1. Load file lists and filter by period
    f_mswx = sorted(glob.glob(os.path.join(HR_MSWX, "*.nc")))
    f_ddpm = sorted(glob.glob(os.path.join(HR_DDPM, "*_ddpm.nc")))
    f_mswx = filter_period(f_mswx, TEST_START, TEST_END)
    f_ddpm = filter_period(f_ddpm, TEST_START, TEST_END)

    if not f_mswx or not f_ddpm:
        raise RuntimeError("No files found for the requested period.")

    print(f"Found {len(f_mswx)} MSWX files and {len(f_ddpm)} DDPM files")

    # 2. Open datasets
    ds_mswx = xr.open_mfdataset(f_mswx, combine="nested", concat_dim="time", 
                                 engine="netcdf4", parallel=False)
    ds_ddpm = xr.open_mfdataset(f_ddpm, combine="nested", concat_dim="time", 
                                 engine="netcdf4", parallel=False)

    # Assign time if missing
    if "time" not in ds_mswx.coords:
        times = [doy_to_datetime(get_file_date(f)) for f in f_mswx]
        ds_mswx = ds_mswx.assign_coords(time=("time", times))
    if "time" not in ds_ddpm.coords:
        times = [doy_to_datetime(get_file_date(f)) for f in f_ddpm]
        ds_ddpm = ds_ddpm.assign_coords(time=("time", times))

    print(f"Loaded datasets: {len(ds_mswx.time)} time steps")

    # =====================================================================
    # PART A: SPATIAL Q-Q (Original analysis - climatological mean field)
    # =====================================================================
    print("\n=== Computing SPATIAL Q-Q analysis ===")
    
    # Spatial annual mean fields averaged across years
    overall_mswx = ds_mswx["air_temperature"].groupby("time.year").mean("time").mean("year")
    overall_ddpm = ds_ddpm["air_temperature"].groupby("time.year").mean("time").mean("year")
    overall_bias = overall_ddpm - overall_mswx

    # Clip to India
    overall_mswx = overall_mswx.rename({"lat":"y","lon":"x"})
    overall_ddpm = overall_ddpm.rename({"lat":"y","lon":"x"})
    overall_bias = overall_bias.rename({"lat":"y","lon":"x"})
    
    overall_mswx.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
    overall_mswx.rio.write_crs("EPSG:4326", inplace=True)
    overall_ddpm.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
    overall_ddpm.rio.write_crs("EPSG:4326", inplace=True)
    overall_bias.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
    overall_bias.rio.write_crs("EPSG:4326", inplace=True)

    gdf = gpd.read_file(INDIA_SHP)
    overall_mswx_clipped = overall_mswx.rio.clip(gdf.geometry, gdf.crs)
    overall_ddpm_clipped = overall_ddpm.rio.clip(gdf.geometry, gdf.crs)
    overall_bias_clipped = overall_bias.rio.clip(gdf.geometry, gdf.crs)

    # Flatten for spatial Q-Q
    mswx_vals_spatial = overall_mswx_clipped.values.flatten()
    ddpm_vals_spatial = overall_ddpm_clipped.values.flatten()
    bias_vals_spatial = overall_bias_clipped.values.flatten()
    
    mswx_vals_spatial = mswx_vals_spatial[np.isfinite(mswx_vals_spatial)]
    ddpm_vals_spatial = ddpm_vals_spatial[np.isfinite(ddpm_vals_spatial)]
    bias_vals_spatial = bias_vals_spatial[np.isfinite(bias_vals_spatial)]

    print(f"Spatial samples: {len(bias_vals_spatial)}")

    # Compute spatial Q-Q statistics
    dof_info = compute_effective_dof(overall_bias_clipped.values, GRID_SPACING_KM)
    qq_stats_spatial, q_mswx_spatial, q_ddpm_spatial = compute_qq_statistics(
        mswx_vals_spatial, ddpm_vals_spatial, dof_info['effective_dof']
    )

    # =====================================================================
    # PART B: TEMPORAL Q-Q (NEW - daily domain averages)
    # =====================================================================
    print("\n=== Computing TEMPORAL Q-Q analysis ===")
    
    # Compute domain-averaged temperature for each day
    # This gives us the temporal distribution of daily mean temperatures
    mswx_daily_avg = ds_mswx["air_temperature"].mean(dim=['lat', 'lon']).values
    ddpm_daily_avg = ds_ddpm["air_temperature"].mean(dim=['lat', 'lon']).values
    
    # Remove NaNs
    mask_temporal = np.isfinite(mswx_daily_avg) & np.isfinite(ddpm_daily_avg)
    mswx_vals_temporal = mswx_daily_avg[mask_temporal]
    ddpm_vals_temporal = ddpm_daily_avg[mask_temporal]
    
    print(f"Temporal samples (days): {len(mswx_vals_temporal)}")
    print(f"Temperature range: {mswx_vals_temporal.min():.2f} to {mswx_vals_temporal.max():.2f}°C")

    # Compute temporal Q-Q statistics
    qq_stats_temporal, q_mswx_temporal, q_ddpm_temporal = compute_qq_statistics(
        mswx_vals_temporal, ddpm_vals_temporal
    )

    # =====================================================================
    # SAVE STATISTICS
    # =====================================================================
    try:
        with open(STATS_FILE, "w") as f:
            f.write("BIAS ANALYSIS (DDPM vs MSWX)\n")
            f.write("="*70 + "\n\n")
            
            f.write("SPATIAL Q-Q ANALYSIS (Climatological Mean Field)\n")
            f.write("-"*70 + "\n")
            f.write(f"Number of spatial points: {len(bias_vals_spatial):,}\n")
            f.write(f"Effective DOF: {dof_info['effective_dof']:,}\n")
            f.write(f"Mean bias: {np.mean(bias_vals_spatial):.4f} °C\n")
            f.write(f"Std bias: {np.std(bias_vals_spatial):.4f} °C\n")
            f.write(f"Q-Q Slope: {qq_stats_spatial['slope']:.4f}\n")
            f.write(f"Q-Q Intercept: {qq_stats_spatial['intercept']:.4f} °C\n")
            f.write(f"Q-Q R²: {qq_stats_spatial['r_squared']:.6f}\n")
            f.write(f"5th percentile bias: {qq_stats_spatial['p5_bias']:.4f} °C\n")
            f.write(f"95th percentile bias: {qq_stats_spatial['p95_bias']:.4f} °C\n\n")
            
            f.write("TEMPORAL Q-Q ANALYSIS (Daily Domain Averages)\n")
            f.write("-"*70 + "\n")
            f.write(f"Number of days: {len(mswx_vals_temporal):,}\n")
            f.write(f"Temperature range (MSWX): {mswx_vals_temporal.min():.2f} to {mswx_vals_temporal.max():.2f} °C\n")
            f.write(f"Temperature range (DDPM): {ddpm_vals_temporal.min():.2f} to {ddpm_vals_temporal.max():.2f} °C\n")
            f.write(f"Q-Q Slope: {qq_stats_temporal['slope']:.4f}\n")
            f.write(f"Q-Q Intercept: {qq_stats_temporal['intercept']:.4f} °C\n")
            f.write(f"Q-Q R²: {qq_stats_temporal['r_squared']:.6f}\n")
            f.write(f"5th percentile bias: {qq_stats_temporal['p5_bias']:.4f} °C\n")
            f.write(f"95th percentile bias: {qq_stats_temporal['p95_bias']:.4f} °C\n\n")
            
        print(f"\nStatistics saved to: {STATS_FILE}")
    except Exception as e:
        print(f"Warning: failed to save stats file: {e}")

    # =====================================================================
    # PLOTTING: THREE PANELS
    # =====================================================================
    print("\n=== Creating 3-panel figure ===")
    
    fig = plt.figure(figsize=FIGSIZE)
    gs = GridSpec(1, 3, figure=fig, wspace=0.30)

    # -------------------------------------------------------------------------
    # Panel (a): Bias Distribution (KDE + histogram + normal fit)
    # -------------------------------------------------------------------------
    ax1 = fig.add_subplot(gs[0, 0])
    x_range = np.linspace(np.min(bias_vals_spatial), np.max(bias_vals_spatial), 300)

    # KDE line
    try:
        kde = gaussian_kde(bias_vals_spatial)
        kde_vals = kde(x_range)
        ax1.plot(x_range, kde_vals, color='steelblue', linewidth=2.5, label='KDE')
    except Exception:
        pass

    # Normal fit
    mu, sigma = stats.norm.fit(bias_vals_spatial)
    normal_pdf = stats.norm.pdf(x_range, mu, sigma)
    ax1.plot(x_range, normal_pdf, 'r--', linewidth=2, label='Normal fit')

    # Histogram
    ax1.hist(bias_vals_spatial, bins=50, density=True, alpha=0.25, 
             color='gray', edgecolor='k', label='Histogram')

    # Reference lines
    ax1.axvline(0.0, color='black', linestyle='-', linewidth=1.4)
    ax1.axvline(np.mean(bias_vals_spatial), color='darkblue', linestyle='--', 
                linewidth=1.6, label=f"Mean = {np.mean(bias_vals_spatial):.3f} °C")

    ax1.set_xlabel('Spatial Bias (DDPM − MSWX) (°C)', fontsize=LABEL_FS, fontweight='bold')
    ax1.set_ylabel('Density', fontsize=LABEL_FS, fontweight='bold')
    ax1.set_title('(a) Spatial Bias Distribution', fontsize=LABEL_FS, fontweight='bold', pad=10)
    ax1.legend(fontsize=LEGEND_FS, loc='upper right')
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(axis='both', labelsize=TICK_FS)

    # -------------------------------------------------------------------------
    # Panel (b): Spatial Q-Q Plot (Climatological)
    # -------------------------------------------------------------------------
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.scatter(q_mswx_spatial, q_ddpm_spatial, alpha=0.6, s=28, 
                color='steelblue', label='Quantile pairs')

    # 1:1 line
    min_val = min(q_mswx_spatial.min(), q_ddpm_spatial.min())
    max_val = max(q_mswx_spatial.max(), q_ddpm_spatial.max())
    ax2.plot([min_val, max_val], [min_val, max_val], 'k--', 
             linewidth=1.6, label='1:1 reference')

    # Fitted line
    fitted_line = qq_stats_spatial['slope'] * q_mswx_spatial + qq_stats_spatial['intercept']
    ax2.plot(q_mswx_spatial, fitted_line, color='green', linewidth=2.0, 
             label=f"Fit: y={qq_stats_spatial['slope']:.3f}x{qq_stats_spatial['intercept']:+.2f}")

    ax2.set_xlabel('MSWX Spatial Quantiles (°C)', fontsize=LABEL_FS, fontweight='bold')
    ax2.set_ylabel('DDPM Spatial Quantiles (°C)', fontsize=LABEL_FS, fontweight='bold')
    ax2.set_title(f'(b) Spatial Q-Q (R²={qq_stats_spatial["r_squared"]:.4f})', 
                  fontsize=LABEL_FS, fontweight='bold', pad=10)
    ax2.legend(fontsize=LEGEND_FS, loc='lower right')
    ax2.grid(True, alpha=0.3)
    ax2.set_aspect('equal', adjustable='box')
    ax2.tick_params(axis='both', labelsize=TICK_FS)

    # -------------------------------------------------------------------------
    # Panel (c): Temporal Q-Q Plot (Daily Averages) - NEW
    # -------------------------------------------------------------------------
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.scatter(q_mswx_temporal, q_ddpm_temporal, alpha=0.6, s=28, 
                color='coral', label='Quantile pairs')

    # 1:1 line
    min_val_t = min(q_mswx_temporal.min(), q_ddpm_temporal.min())
    max_val_t = max(q_mswx_temporal.max(), q_ddpm_temporal.max())
    ax3.plot([min_val_t, max_val_t], [min_val_t, max_val_t], 'k--', 
             linewidth=1.6, label='1:1 reference')

    # Fitted line
    fitted_line_t = qq_stats_temporal['slope'] * q_mswx_temporal + qq_stats_temporal['intercept']
    ax3.plot(q_mswx_temporal, fitted_line_t, color='darkred', linewidth=2.0, 
             label=f"Fit: y={qq_stats_temporal['slope']:.3f}x{qq_stats_temporal['intercept']:+.2f}")

    ax3.set_xlabel('MSWX Temporal Quantiles (°C)', fontsize=LABEL_FS, fontweight='bold')
    ax3.set_ylabel('DDPM Temporal Quantiles (°C)', fontsize=LABEL_FS, fontweight='bold')
    ax3.set_title(f'(c) Temporal Q-Q (R²={qq_stats_temporal["r_squared"]:.4f})', 
                  fontsize=LABEL_FS, fontweight='bold', pad=10)
    ax3.legend(fontsize=LEGEND_FS, loc='lower right')
    ax3.grid(True, alpha=0.3)
    ax3.set_aspect('equal', adjustable='box')
    ax3.tick_params(axis='both', labelsize=TICK_FS)

    # -------------------------------------------------------------------------
    # Save figure
    # -------------------------------------------------------------------------
    plt.tight_layout()
    plt.savefig(PLOT_DIST_FILE, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Saved figure: {PLOT_DIST_FILE}")
    
    plt.show()

    print("\n=== Analysis Complete ===")
    print(f"Spatial Q-Q: R² = {qq_stats_spatial['r_squared']:.6f}, "
          f"5th/95th %ile bias = {qq_stats_spatial['p5_bias']:.3f}/{qq_stats_spatial['p95_bias']:.3f}°C")
    print(f"Temporal Q-Q: R² = {qq_stats_temporal['r_squared']:.6f}, "
          f"5th/95th %ile bias = {qq_stats_temporal['p5_bias']:.3f}/{qq_stats_temporal['p95_bias']:.3f}°C")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Distributional Fidelity Analysis — Section 3.5 (Q1-upgraded)
=============================================================
Four-panel figure:
  (a) Spatial bias KDE + normal fit + excess kurtosis annotation
  (b) Spatial Q-Q with 95% bootstrap CI ribbon
  (c) Temporal Q-Q with 95% bootstrap CI ribbon  
  (d) Q-Q residual (deviation from 1:1) for both — the diagnostic
      panel that converts a visual check into a quantitative one

FIXES vs original script:
  - India mask applied to BOTH spatial AND temporal analyses
  - Bootstrap CI ribbons on panels (b) and (c)
  - Excess kurtosis reported (leptokurtic tails = Himalayan pixels)
  - Residual panel (d) added
  - 500 quantile points instead of 99 → smoother Q-Q curves
  - P5/P95 tail bias annotated directly on residual panel
"""

import os, glob, re
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import rasterio.features
import rasterio.transform
from shapely.geometry import mapping
from scipy import stats
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings("ignore")

# ── CONFIG ───────────────────────────────────────────────────────────────────
HR_MSWX   = "<SET YOUR PATH HERE>"
HR_DDPM   = "<SET YOUR PATH HERE>"
INDIA_SHP = "<SET YOUR PATH HERE>"

TEST_START  = 2009219
TEST_END    = 2014365
N_BOOTSTRAP = 500
PERCENTILES = np.linspace(0.2, 99.8, 500)   # 500 pts → smooth curves

OUT_PNG = "distributional_fidelity_upgraded.png"
DPI     = 300

LAT_NAMES = ("lat","latitude","y")
LON_NAMES = ("lon","longitude","x")
VAR_NAMES = ("air_temperature","tas","t2m","temperature","temp")

# ── PALETTE ──────────────────────────────────────────────────────────────────
C_BLUE  = "#1a5e96"
C_RED   = "#b5311a"
C_GREEN = "#1a7a4a"
C_GRAY  = "#555555"
C_ZERO  = "#222222"

# ── HELPERS ──────────────────────────────────────────────────────────────────
def parse_token(fname):
    base = os.path.splitext(os.path.basename(fname))[0]
    for pat in [r"(\d{7})", r"(\d{8})"]:
        m = re.search(pat, base)
        if m: return int(m.group(1))
    return None

def token_to_dt(tok):
    s = str(tok)
    if len(s) == 7:
        return pd.to_datetime(f"{s[:4]}-{s[4:]}", format="%Y-%j")
    return pd.to_datetime(s, format="%Y%m%d")

def filter_files(files, start, end):
    return [f for f in sorted(files)
            if (tok := parse_token(f)) is not None and start <= tok <= end]

def find_var(ds):
    for n in VAR_NAMES:
        if n in ds.data_vars: return n
    for v in ds.data_vars:
        if np.issubdtype(ds[v].dtype, np.floating) and ds[v].ndim >= 2:
            return v
    raise RuntimeError("No temperature variable found.")

def find_coord(ds, candidates):
    for n in candidates:
        if n in ds.coords: return n
    raise RuntimeError(f"None of {candidates} in {list(ds.coords)}")

def build_india_mask(shp_path, lat, lon):
    gdf = gpd.read_file(shp_path)
    if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)
    geom = gdf.dissolve().geometry.values[0].buffer(0.05)
    nrows, ncols = len(lat), len(lon)
    lon_res = abs(lon[1]-lon[0]) if len(lon)>1 else 0.1
    lat_res = abs(lat[1]-lat[0]) if len(lat)>1 else 0.1
    tfm = rasterio.transform.from_origin(
        lon.min()-lon_res/2, lat.max()+lat_res/2, lon_res, lat_res)
    outside = rasterio.features.geometry_mask(
        [mapping(geom)], out_shape=(nrows, ncols),
        transform=tfm, invert=False, all_touched=True)
    mask = ~outside
    if lat[0] < lat[-1]:
        mask = mask[::-1, :]
    return mask

def maybe_kelvin(arr):
    return arr - 273.15 if np.nanmean(arr) > 200 else arr

def bootstrap_qq(ref, pred, percentiles, n_boot=500, seed=42, ci=98):
    """Bootstrap CI on Q-Q. Returns q_ref, q_pred, lo, hi."""
    rng = np.random.default_rng(seed)
    q_ref  = np.percentile(ref,  percentiles)
    q_pred = np.percentile(pred, percentiles)
    boot   = np.zeros((n_boot, len(percentiles)))
    for i in range(n_boot):
        idx = rng.integers(0, len(pred), size=len(pred))
        boot[i] = np.percentile(pred[idx], percentiles)
    lo = np.percentile(boot, (100 - ci) / 2,       axis=0)
    hi = np.percentile(boot, 100 - (100 - ci) / 2, axis=0)
    return q_ref, q_pred, lo, hi

def fit_qq(q_ref, q_pred):
    slope, intercept = np.polyfit(q_ref, q_pred, 1)
    yhat  = slope * q_ref + intercept
    ss_res = np.sum((q_pred - yhat)**2)
    ss_tot = np.sum((q_pred - q_pred.mean())**2)
    r2 = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan
    return slope, intercept, r2

# ── LOAD DATA ────────────────────────────────────────────────────────────────
mswx_files = filter_files(glob.glob(os.path.join(HR_MSWX,"*.nc")), TEST_START, TEST_END)
ddpm_files  = filter_files(glob.glob(os.path.join(HR_DDPM, "*.nc")), TEST_START, TEST_END)
print(f"Files: MSWX={len(mswx_files)}, DDPM={len(ddpm_files)}")

mswx_times = [token_to_dt(parse_token(f)) for f in mswx_files]
ddpm_times  = [token_to_dt(parse_token(f)) for f in ddpm_files]

ds_m = (xr.open_mfdataset(mswx_files, combine="nested", concat_dim="time",
                           engine="netcdf4", parallel=False)
          .assign_coords(time=("time", pd.to_datetime(mswx_times)))
          .sortby("time"))
ds_d = (xr.open_mfdataset(ddpm_files,  combine="nested", concat_dim="time",
                           engine="netcdf4", parallel=False)
          .assign_coords(time=("time", pd.to_datetime(ddpm_times)))
          .sortby("time"))

var_m = find_var(ds_m); var_d = find_var(ds_d)
lat = ds_m[find_coord(ds_m, LAT_NAMES)].values.astype(float)
lon = ds_m[find_coord(ds_m, LON_NAMES)].values.astype(float)

print("Building India mask …")
mask_2d = build_india_mask(INDIA_SHP, lat, lon)
print(f"  India pixels: {mask_2d.sum():,}")

print("Loading arrays …")
arr_m = maybe_kelvin(ds_m[var_m].values)   # (T, nlat, nlon)
arr_d = maybe_kelvin(ds_d[var_d].values)

times_m = pd.DatetimeIndex(ds_m.time.values)
times_d = pd.DatetimeIndex(ds_d.time.values)
common  = times_m.intersection(times_d)
arr_m   = arr_m[times_m.get_indexer(common)]
arr_d   = arr_d[times_d.get_indexer(common)]
dates   = common
T       = len(dates)
print(f"  {T} common days: {dates[0].date()} → {dates[-1].date()}")

# apply India mask
arr_m[:, ~mask_2d] = np.nan
arr_d[:, ~mask_2d] = np.nan

# ── PANEL (a): SPATIAL BIAS DISTRIBUTION ─────────────────────────────────────
clim_m = np.nanmean(arr_m, axis=0)
clim_d = np.nanmean(arr_d, axis=0)
bias_sp = (clim_d - clim_m).ravel()
bias_sp = bias_sp[np.isfinite(bias_sp)]

b_mean = float(np.mean(bias_sp))
b_sd   = float(np.std(bias_sp))
b_kurt = float(stats.kurtosis(bias_sp))   # excess kurtosis
b_skew = float(stats.skew(bias_sp))
print(f"\nSpatial bias: mean={b_mean:+.5f}°C, SD={b_sd:.4f}°C, "
      f"excess κ={b_kurt:.2f}, skew={b_skew:.3f}")

# ── PANEL (b): SPATIAL Q-Q ───────────────────────────────────────────────────
clim_m_flat = clim_m.ravel(); clim_m_flat = clim_m_flat[np.isfinite(clim_m_flat)]
clim_d_flat = clim_d.ravel(); clim_d_flat = clim_d_flat[np.isfinite(clim_d_flat)]

print(f"\nSpatial Q-Q bootstrap ({N_BOOTSTRAP} iters) …")
q_ref_sp, q_pred_sp, lo_sp, hi_sp = bootstrap_qq(
    clim_m_flat, clim_d_flat, PERCENTILES, N_BOOTSTRAP)
slope_sp, intercept_sp, r2_sp = fit_qq(q_ref_sp, q_pred_sp)
resid_sp = q_pred_sp - q_ref_sp
p5_sp  = float(np.percentile(clim_d_flat,5)  - np.percentile(clim_m_flat,5))
p95_sp = float(np.percentile(clim_d_flat,95) - np.percentile(clim_m_flat,95))
print(f"  R²={r2_sp:.6f}, slope={slope_sp:.4f}, "
      f"P5 bias={p5_sp:+.4f}°C, P95 bias={p95_sp:+.4f}°C")

# ── PANEL (c): TEMPORAL Q-Q (India-masked daily means) ───────────────────────
flat_m = arr_m.reshape(T, -1)
flat_d = arr_d.reshape(T, -1)
daily_m = np.nanmean(flat_m, axis=1)
daily_d = np.nanmean(flat_d, axis=1)
ok = np.isfinite(daily_m) & np.isfinite(daily_d)
daily_m = daily_m[ok]; daily_d = daily_d[ok]

print(f"\nTemporal Q-Q bootstrap ({N_BOOTSTRAP} iters) …")
q_ref_t, q_pred_t, lo_t, hi_t = bootstrap_qq(
    daily_m, daily_d, PERCENTILES, N_BOOTSTRAP)
slope_t, intercept_t, r2_t = fit_qq(q_ref_t, q_pred_t)
resid_t = q_pred_t - q_ref_t
p5_t  = float(np.percentile(daily_d,5)  - np.percentile(daily_m,5))
p95_t = float(np.percentile(daily_d,95) - np.percentile(daily_m,95))
print(f"  R²={r2_t:.6f}, slope={slope_t:.4f}, "
      f"P5 bias={p5_t:+.4f}°C, P95 bias={p95_t:+.4f}°C")

# ── FIGURE  2×2 ──────────────────────────────────────────────────────────────
FS_LABEL  = 14    # axis labels
FS_TICK   = 12    # tick labels
FS_LEGEND = 11    # legend text
FS_ANNOT  = 11    # stat box
FS_PANEL  = 15    # panel letter (a)(b)(c)(d)
FS_TITLE  = 11    # subplot title (R², slope)

fig = plt.figure(figsize=(14, 12))
fig.patch.set_facecolor("white")
gs = GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

# ── (a) Bias distribution  [top-left] ────────────────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
x_rng = np.linspace(bias_sp.min(), bias_sp.max(), 500)
kde_obj = gaussian_kde(bias_sp, bw_method='scott')
ax_a.hist(bias_sp, bins=80, density=True, alpha=0.18,
          color=C_BLUE, edgecolor='none')
ax_a.plot(x_rng, kde_obj(x_rng), color=C_BLUE, lw=2.4, label='KDE')
mu_n, sig_n = stats.norm.fit(bias_sp)
ax_a.plot(x_rng, stats.norm.pdf(x_rng, mu_n, sig_n),
          color=C_RED, lw=2.0, ls='--', label='Normal fit')
ax_a.axvline(0, color=C_ZERO, lw=1.4, ls='-', zorder=3)
ax_a.axvline(b_mean, color=C_BLUE, lw=1.5, ls=':',
             label=f'Mean = {b_mean:+.4f}°C')
ax_a.text(0.97, 0.97,
          f"SD = {b_sd:.4f}°C\nExcess κ = {b_kurt:.2f}\nSkewness = {b_skew:.3f}",
          transform=ax_a.transAxes, fontsize=FS_ANNOT, va='top', ha='right',
          family='monospace',
          bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#cccccc', alpha=0.92))
ax_a.set_xlabel("Spatial Bias (DDPM − MSWX, °C)", fontsize=FS_LABEL, fontweight='bold')
ax_a.set_ylabel("Density", fontsize=FS_LABEL, fontweight='bold')
ax_a.legend(fontsize=FS_LEGEND, loc='upper left', framealpha=0.9)
ax_a.spines[['top','right']].set_visible(False)
ax_a.tick_params(labelsize=FS_TICK)
ax_a.text(-0.12, 1.06, "(a)", transform=ax_a.transAxes,
          fontsize=FS_PANEL, fontweight='bold', va='top')

# ── (b) Spatial Q-Q  [top-right] ─────────────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])
ax_b.fill_between(q_ref_sp, lo_sp, hi_sp,
                  color=C_BLUE, alpha=0.18, label='98% bootstrap CI')
ax_b.scatter(q_ref_sp, q_pred_sp, s=18, color=C_BLUE, alpha=0.65,
             zorder=3, label='Quantile pairs')
diag = [min(q_ref_sp.min(), q_pred_sp.min()),
        max(q_ref_sp.max(), q_pred_sp.max())]
ax_b.plot(diag, diag, 'k--', lw=1.5, label='1:1 reference')
ax_b.plot(q_ref_sp, slope_sp*q_ref_sp+intercept_sp,
          color=C_GREEN, lw=2.0,
          label=f"Fit: y = {slope_sp:.3f}x {intercept_sp:+.2f}")
ax_b.set_xlabel("MSWX Spatial Quantiles (°C)", fontsize=FS_LABEL, fontweight='bold')
ax_b.set_ylabel("DDPM Spatial Quantiles (°C)", fontsize=FS_LABEL, fontweight='bold')
ax_b.set_title(f"R² = {r2_sp:.4f}   slope = {slope_sp:.4f}",
               fontsize=FS_TITLE, color=C_GRAY, pad=5)
ax_b.legend(fontsize=FS_LEGEND, loc='lower right', framealpha=0.9)
ax_b.set_aspect('equal', adjustable='box')
ax_b.spines[['top','right']].set_visible(False)
ax_b.tick_params(labelsize=FS_TICK)
ax_b.text(-0.14, 1.06, "(b)", transform=ax_b.transAxes,
          fontsize=FS_PANEL, fontweight='bold', va='top')

# ── (c) Temporal Q-Q  [bottom-left] ──────────────────────────────────────────
ax_c = fig.add_subplot(gs[1, 0])
ax_c.fill_between(q_ref_t, lo_t, hi_t,
                  color=C_RED, alpha=0.18, label='98% bootstrap CI')
ax_c.scatter(q_ref_t, q_pred_t, s=18, color=C_RED, alpha=0.65,
             zorder=3, label='Quantile pairs')
diag_t = [min(q_ref_t.min(), q_pred_t.min()),
          max(q_ref_t.max(), q_pred_t.max())]
ax_c.plot(diag_t, diag_t, 'k--', lw=1.5, label='1:1 reference')
ax_c.plot(q_ref_t, slope_t*q_ref_t+intercept_t,
          color=C_GREEN, lw=2.0,
          label=f"Fit: y = {slope_t:.3f}x {intercept_t:+.2f}")
ax_c.set_xlabel("MSWX Temporal Quantiles (°C)", fontsize=FS_LABEL, fontweight='bold')
ax_c.set_ylabel("DDPM Temporal Quantiles (°C)", fontsize=FS_LABEL, fontweight='bold')
ax_c.set_title(f"R² = {r2_t:.4f}   slope = {slope_t:.4f}",
               fontsize=FS_TITLE, color=C_GRAY, pad=5)
ax_c.legend(fontsize=FS_LEGEND, loc='lower right', framealpha=0.9)
ax_c.set_aspect('equal', adjustable='box')
ax_c.spines[['top','right']].set_visible(False)
ax_c.tick_params(labelsize=FS_TICK)
ax_c.text(-0.14, 1.06, "(c)", transform=ax_c.transAxes,
          fontsize=FS_PANEL, fontweight='bold', va='top')

# ── (d) Q-Q residuals  [bottom-right] ────────────────────────────────────────
ax_d = fig.add_subplot(gs[1, 1])

pct_mask = (PERCENTILES >= 5) & (PERCENTILES <= 95)
P_vis    = PERCENTILES[pct_mask]

# zero reference
ax_d.axhline(0, color='#888888', lw=1.0, ls='--', alpha=0.7, zorder=1)

# residual lines — no CI bands
ax_d.plot(P_vis, resid_sp[pct_mask], color=C_BLUE, lw=2.4,
          label='Spatial', zorder=3)
ax_d.plot(P_vis, resid_t[pct_mask],  color=C_RED,  lw=2.4,
          label='Temporal', zorder=3)

# ── tight y-axis so the flat interior is the focus ───────────────────────────
_yscale = max(np.abs(resid_sp[pct_mask]).max(),
              np.abs(resid_t[pct_mask]).max()) * 6
ax_d.set_ylim(-_yscale, _yscale)

# ── P5 / P95 tail deviation callouts ─────────────────────────────────────────
# The 4 values are scientifically meaningful — they quantify tail error exactly.
# Layout: P5 labels offset LEFT of their dots (x~8); P95 labels offset RIGHT (x~88).
# Spatial annotations sit slightly above zero; Temporal slightly below.
# This keeps all 4 labels visible and non-overlapping.

_idx5  = int(np.searchsorted(PERCENTILES, 5))
_idx95 = int(np.searchsorted(PERCENTILES, 95))

_callout_kw = dict(
    fontsize=10, fontweight='bold', va='center',
    bbox=dict(boxstyle='round,pad=0.3', fc='white',
              ec='#cccccc', alpha=0.97, lw=0.8),
    arrowprops=dict(arrowstyle='->', lw=1.2, color='#888888'),
    zorder=7
)

# --- P5 spatial (blue, above zero line) ---
ax_d.annotate(f"P5 spatial\n{resid_sp[_idx5]:+.4f}°C",
              xy=(PERCENTILES[_idx5], resid_sp[_idx5]),
              xytext=(14, _yscale * 0.55),
              ha='center', color=C_BLUE, **_callout_kw)

# --- P5 temporal (red, below zero line) ---
ax_d.annotate(f"P5 temporal\n{resid_t[_idx5]:+.4f}°C",
              xy=(PERCENTILES[_idx5], resid_t[_idx5]),
              xytext=(14, -_yscale * 0.55),
              ha='center', color=C_RED, **_callout_kw)

# --- P95 spatial (blue, above) ---
ax_d.annotate(f"P95 spatial\n{resid_sp[_idx95]:+.4f}°C",
              xy=(PERCENTILES[_idx95], resid_sp[_idx95]),
              xytext=(84, _yscale * 0.55),
              ha='center', color=C_BLUE, **_callout_kw)

# --- P95 temporal (red, below) ---
ax_d.annotate(f"P95 temporal\n{resid_t[_idx95]:+.4f}°C",
              xy=(PERCENTILES[_idx95], resid_t[_idx95]),
              xytext=(84, -_yscale * 0.55),
              ha='center', color=C_RED, **_callout_kw)

# dot markers on the lines at P5 / P95
for pct_idx in [_idx5, _idx95]:
    for resid, color in [(resid_sp, C_BLUE), (resid_t, C_RED)]:
        ax_d.plot(PERCENTILES[pct_idx], resid[pct_idx], 'o',
                  color=color, ms=7, zorder=5,
                  markeredgecolor='white', markeredgewidth=1.4)

ax_d.set_xlabel("Percentile", fontsize=FS_LABEL, fontweight='bold')
ax_d.set_ylabel("DDPM − MSWX Quantile Deviation (°C)",
                fontsize=FS_LABEL, fontweight='bold')
ax_d.set_xlim(5, 95)
ax_d.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
ax_d.legend(fontsize=FS_LEGEND, loc='lower center',
            framealpha=0.9, ncol=2)
ax_d.spines[['top','right']].set_visible(False)
ax_d.tick_params(labelsize=FS_TICK)
ax_d.text(-0.14, 1.06, "(d)", transform=ax_d.transAxes,
          fontsize=FS_PANEL, fontweight='bold', va='top')

fig.suptitle(
    "Distributional Fidelity Assessment  (India boundary mask, 2009–2014)",
    fontsize=13, fontweight='bold', y=1.01)

plt.savefig(OUT_PNG, dpi=DPI, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print(f"\nSaved → {OUT_PNG}")

# ── MANUSCRIPT NUMBERS ────────────────────────────────────────────────────────
print("\n── Stats for manuscript ──")
print(f"Spatial bias : mean={b_mean:+.6f}°C  SD={b_sd:.5f}°C  "
      f"excess κ={b_kurt:.3f}  skew={b_skew:.4f}")
print(f"Spatial Q-Q  : R²={r2_sp:.6f}  slope={slope_sp:.5f}  "
      f"P5 bias={p5_sp:+.5f}°C  P95 bias={p95_sp:+.5f}°C")
print(f"Temporal Q-Q : R²={r2_t:.6f}  slope={slope_t:.5f}  "
      f"P5 bias={p5_t:+.5f}°C  P95 bias={p95_t:+.5f}°C")
print(f"Bias range   : [{bias_sp.min():+.5f}, {bias_sp.max():+.5f}]°C")

In [ ]:
#!/usr/bin/env python3
"""
Comprehensive Bias & Q-Q Plot Analysis (publication quality)
Optimized for High-Performance Computing (80 Cores)

NEW FEATURES:
- 3-panel main figure: (a) bias distribution, (b) spatial Q-Q, (c) temporal Q-Q
- 4-panel supplementary figure: Per-grid-cell temporal Q-Q statistics maps
  showing R², slope, 5th percentile bias, and 95th percentile bias
- Parallel processing for per-grid-cell analysis
"""

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import matplotlib.colors as mcolors
import glob
import os
import rioxarray
import geopandas as gpd
import statsmodels.api as sm
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.ndimage import uniform_filter1d
import warnings
from tqdm import tqdm
import multiprocessing
from functools import partial

warnings.filterwarnings('ignore')

# =====================================================================
# CONFIGURATION
# =====================================================================

# Paths
HR_MSWX = "<SET YOUR PATH HERE>"
HR_DDPM = "<SET YOUR PATH HERE>"
INDIA_SHP = "<SET YOUR PATH HERE>"

# Analysis parameters
TEST_START = 2009219  # 2009-08-07 (YYYYDOY)
TEST_END = 2014365
NUM_CORES = 80  # Number of CPU workers

# Grid spacing (km)
GRID_SPACING_KM = 10.5924

# Output files
PLOT_MAIN_FILE = "bias_distribution_and_qq_3panel.png"
PLOT_GRIDCELL_FILE = "per_gridcell_temporal_qq_maps.png"
STATS_FILE = "bias_analysis_statistics.txt"
DATA_FILE = "bias_analysis_data.csv"
GRIDCELL_STATS_FILE = "per_gridcell_temporal_qq_stats.nc"  # NetCDF for spatial stats

# Font sizes
LABEL_FS = 16
TICK_FS = 14
LEGEND_FS = 12
ANNOT_FS = 13
TITLE_FS = 18
FIGSIZE_MAIN = (18, 6)
FIGSIZE_MAPS = (20, 16)

# =====================================================================
# Helper functions
# =====================================================================

def get_file_date(fname):
    base = os.path.basename(fname)
    base = base.replace("_ddpm", "").replace(".nc", "")
    try:
        return int(base)
    except Exception:
        import re
        m = re.search(r"(\d{7,8})", base)
        return int(m.group(1)) if m else None

def doy_to_datetime(yydoy):
    year = int(str(yydoy)[:4])
    doy = int(str(yydoy)[4:])
    return pd.to_datetime(f"{year}-{doy:03d}", format="%Y-%j")

def filter_period(files, start=2009219, end=2014365):
    valid = []
    for f in files:
        try:
            d = get_file_date(f)
            if d is None:
                continue
            if start <= d <= end:
                valid.append(f)
        except:
            continue
    return valid

def compute_effective_dof(field_2d, grid_spacing_km, decorr_scale_km=300):
    if field_2d is None or np.isnan(field_2d).all():
        return {'total_dof':0,'effective_dof':0,'reduction_factor':1.0,'decorr_scale_km':decorr_scale_km}
    ny, nx = np.array(field_2d.shape)
    total = ny * nx
    domain_w = nx * grid_spacing_km
    domain_h = ny * grid_spacing_km
    n_indep_x = max(1, domain_w / decorr_scale_km)
    n_indep_y = max(1, domain_h / decorr_scale_km)
    eff = max(1, int(n_indep_x * n_indep_y))
    reduction = total / eff if eff>0 else 1.0
    return {'total_dof': total, 'effective_dof': eff, 'reduction_factor': reduction, 'decorr_scale_km': decorr_scale_km}

def compute_qq_statistics(mswx_vals, ddpm_vals, effective_dof=None):
    """
    Compute Q-Q statistics between mswx_vals and ddpm_vals.
    Returns (stats_dict, q_mswx, q_ddpm).
    """
    mswx = np.asarray(mswx_vals).ravel()
    ddpm = np.asarray(ddpm_vals).ravel()
    mask = np.isfinite(mswx) & np.isfinite(ddpm)
    mswx = mswx[mask]
    ddpm = ddpm[mask]

    if len(mswx) < 10:  # Need at least 10 points for meaningful Q-Q
        return ({'slope':np.nan,'intercept':np.nan,'correlation':np.nan,'r_squared':np.nan,
                 'rmse_qq':np.nan,'p5_bias':np.nan,'p95_bias':np.nan,'slope_ci':None}, 
                np.array([]), np.array([]))

    percentiles = np.arange(1, 100)  # 1..99
    q_m = np.percentile(mswx, q=percentiles)
    q_d = np.percentile(ddpm, q=percentiles)

    # OLS fit q_d = slope * q_m + intercept
    slope, intercept = np.polyfit(q_m, q_d, 1)
    yhat = slope * q_m + intercept
    residuals = q_d - yhat
    rmse_qq = float(np.sqrt(np.mean(residuals**2)))

    # correlation between quantiles
    try:
        corr = float(np.corrcoef(q_m, q_d)[0, 1])
    except Exception:
        corr = np.nan

    # R-squared
    ss_res = np.sum((q_d - yhat)**2)
    ss_tot = np.sum((q_d - np.mean(q_d))**2)
    r_squared = float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan

    # extreme quantile biases
    p5_bias = float(np.percentile(ddpm, 5) - np.percentile(mswx, 5))
    p95_bias = float(np.percentile(ddpm, 95) - np.percentile(mswx, 95))

    stats_qq = {
        'slope': float(slope),
        'intercept': float(intercept),
        'correlation': corr,
        'r_squared': r_squared,
        'rmse_qq': rmse_qq,
        'p5_bias': p5_bias,
        'p95_bias': p95_bias,
        'n_points': len(q_m),
    }
    return stats_qq, q_m, q_d

# =====================================================================
# Parallel Processing Helpers
# =====================================================================

# Global variables to hold data in worker processes
# This avoids serializing large arrays to every worker
shared_mswx_data = None
shared_ddpm_data = None

def init_worker(mswx_data, ddpm_data):
    """
    Initializer function for the multiprocessing pool.
    Sets the global data variables in each child process.
    """
    global shared_mswx_data, shared_ddpm_data
    shared_mswx_data = mswx_data
    shared_ddpm_data = ddpm_data

def process_grid_cell(coords):
    """
    Worker function to process a single grid cell (lat_idx, lon_idx).
    Uses the shared global data to extract time series.
    """
    i, j = coords
    
    # Access the shared data
    mswx_ts = shared_mswx_data[:, i, j]
    ddpm_ts = shared_ddpm_data[:, i, j]
    
    # Check if valid (not all NaN)
    valid_mask = np.isfinite(mswx_ts) & np.isfinite(ddpm_ts)
    n_valid = np.sum(valid_mask)
    
    if n_valid < 100:  # Need sufficient data
        return (i, j, n_valid, None)
    
    # Compute Q-Q statistics for this grid cell
    try:
        qq_stats, _, _ = compute_qq_statistics(
            mswx_ts[valid_mask], 
            ddpm_ts[valid_mask]
        )
        return (i, j, n_valid, qq_stats)
    except Exception:
        return (i, j, n_valid, None)

def compute_per_gridcell_temporal_qq(ds_mswx, ds_ddpm, gdf):
    """
    Compute temporal Q-Q statistics for each grid cell using Parallel Processing.
    """
    print(f"\n=== Computing Per-Grid-Cell Temporal Q-Q Statistics (using {NUM_CORES} cores) ===")
    
    # Get dimensions
    n_time = len(ds_mswx.time)
    n_lat = len(ds_mswx.lat)
    n_lon = len(ds_mswx.lon)
    
    print(f"Grid dimensions: {n_lat} x {n_lon} = {n_lat * n_lon} cells")
    print(f"Time steps: {n_time}")
    
    # Load data into memory for shared access
    # .values triggers the computation/loading
    print("Loading data into memory for parallel distribution...")
    mswx_data = ds_mswx["air_temperature"].values
    ddpm_data = ds_ddpm["air_temperature"].values

    # Initialize output arrays
    r_squared_map = np.full((n_lat, n_lon), np.nan)
    slope_map = np.full((n_lat, n_lon), np.nan)
    p5_bias_map = np.full((n_lat, n_lon), np.nan)
    p95_bias_map = np.full((n_lat, n_lon), np.nan)
    n_valid_map = np.full((n_lat, n_lon), 0)

    # Prepare list of coordinates to process
    coords_list = [(i, j) for i in range(n_lat) for j in range(n_lon)]
    total_cells = len(coords_list)
    processed_count = 0

    print("Starting parallel pool...")
    
    # Create pool and map the worker function
    # initializer ensures we share memory efficiently on Linux systems
    with multiprocessing.Pool(processes=NUM_CORES, initializer=init_worker, initargs=(mswx_data, ddpm_data)) as pool:
        # imap_unordered is generally faster as it yields results as soon as they are ready
        results = list(tqdm(pool.imap_unordered(process_grid_cell, coords_list), 
                           total=total_cells, 
                           desc="Processing grid cells"))

    # Process results
    print("Aggregating results...")
    for i, j, n_valid, stats in results:
        n_valid_map[i, j] = n_valid
        
        if stats is not None:
            r_squared_map[i, j] = stats['r_squared']
            slope_map[i, j] = stats['slope']
            p5_bias_map[i, j] = stats['p5_bias']
            p95_bias_map[i, j] = stats['p95_bias']
            processed_count += 1

    print(f"\nProcessed {processed_count} valid grid cells out of {total_cells}")
    
    # Create xarray Dataset with results
    result_ds = xr.Dataset(
        {
            'r_squared': (['lat', 'lon'], r_squared_map),
            'slope': (['lat', 'lon'], slope_map),
            'p5_bias': (['lat', 'lon'], p5_bias_map),
            'p95_bias': (['lat', 'lon'], p95_bias_map),
            'n_valid': (['lat', 'lon'], n_valid_map),
        },
        coords={
            'lat': ds_mswx.lat,
            'lon': ds_mswx.lon,
        }
    )
    
    # Add attributes
    result_ds['r_squared'].attrs = {
        'long_name': 'Temporal Q-Q R-squared',
        'description': 'R² from Q-Q plot of daily temperatures at each grid cell'
    }
    result_ds['slope'].attrs = {
        'long_name': 'Temporal Q-Q slope',
        'description': 'Slope from Q-Q plot (ideal = 1.0)'
    }
    result_ds['p5_bias'].attrs = {
        'long_name': '5th percentile bias',
        'units': 'degree_Celsius',
        'description': 'DDPM - MSWX at 5th percentile (cold days)'
    }
    result_ds['p95_bias'].attrs = {
        'long_name': '95th percentile bias',
        'units': 'degree_Celsius',
        'description': 'DDPM - MSWX at 95th percentile (hot days)'
    }
    
    return result_ds

# =====================================================================
# MAIN EXECUTION
# =====================================================================

def main():
    # 1. Load file lists and filter by period
    f_mswx = sorted(glob.glob(os.path.join(HR_MSWX, "*.nc")))
    f_ddpm = sorted(glob.glob(os.path.join(HR_DDPM, "*_ddpm.nc")))
    f_mswx = filter_period(f_mswx, TEST_START, TEST_END)
    f_ddpm = filter_period(f_ddpm, TEST_START, TEST_END)

    if not f_mswx or not f_ddpm:
        raise RuntimeError("No files found for the requested period.")

    print(f"Found {len(f_mswx)} MSWX files and {len(f_ddpm)} DDPM files")

    # 2. Open datasets
    # Optimized: parallel=True for faster open
    print("\nLoading datasets...")
    ds_mswx = xr.open_mfdataset(f_mswx, combine="nested", concat_dim="time", 
                                 engine="netcdf4", parallel=True)
    ds_ddpm = xr.open_mfdataset(f_ddpm, combine="nested", concat_dim="time", 
                                 engine="netcdf4", parallel=True)

    # Assign time if missing
    if "time" not in ds_mswx.coords:
        times = [doy_to_datetime(get_file_date(f)) for f in f_mswx]
        ds_mswx = ds_mswx.assign_coords(time=("time", times))
    if "time" not in ds_ddpm.coords:
        times = [doy_to_datetime(get_file_date(f)) for f in f_ddpm]
        ds_ddpm = ds_ddpm.assign_coords(time=("time", times))

    print(f"Loaded datasets: {len(ds_mswx.time)} time steps")

    # Load India shapefile
    gdf = gpd.read_file(INDIA_SHP)

    # =====================================================================
    # PART A: SPATIAL Q-Q (Original analysis)
    # =====================================================================
    print("\n=== Computing SPATIAL Q-Q analysis ===")
    
    overall_mswx = ds_mswx["air_temperature"].groupby("time.year").mean("time").mean("year")
    overall_ddpm = ds_ddpm["air_temperature"].groupby("time.year").mean("time").mean("year")
    overall_bias = overall_ddpm - overall_mswx

    # Clip to India
    overall_mswx = overall_mswx.rename({"lat":"y","lon":"x"})
    overall_ddpm = overall_ddpm.rename({"lat":"y","lon":"x"})
    overall_bias = overall_bias.rename({"lat":"y","lon":"x"})
    
    overall_mswx.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
    overall_mswx.rio.write_crs("EPSG:4326", inplace=True)
    overall_ddpm.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
    overall_ddpm.rio.write_crs("EPSG:4326", inplace=True)
    overall_bias.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
    overall_bias.rio.write_crs("EPSG:4326", inplace=True)

    overall_mswx_clipped = overall_mswx.rio.clip(gdf.geometry, gdf.crs)
    overall_ddpm_clipped = overall_ddpm.rio.clip(gdf.geometry, gdf.crs)
    overall_bias_clipped = overall_bias.rio.clip(gdf.geometry, gdf.crs)

    mswx_vals_spatial = overall_mswx_clipped.values.flatten()
    ddpm_vals_spatial = overall_ddpm_clipped.values.flatten()
    bias_vals_spatial = overall_bias_clipped.values.flatten()
    
    mswx_vals_spatial = mswx_vals_spatial[np.isfinite(mswx_vals_spatial)]
    ddpm_vals_spatial = ddpm_vals_spatial[np.isfinite(ddpm_vals_spatial)]
    bias_vals_spatial = bias_vals_spatial[np.isfinite(bias_vals_spatial)]

    print(f"Spatial samples: {len(bias_vals_spatial)}")

    dof_info = compute_effective_dof(overall_bias_clipped.values, GRID_SPACING_KM)
    qq_stats_spatial, q_mswx_spatial, q_ddpm_spatial = compute_qq_statistics(
        mswx_vals_spatial, ddpm_vals_spatial, dof_info['effective_dof']
    )

    # =====================================================================
    # PART B: TEMPORAL Q-Q (Domain-averaged)
    # =====================================================================
    print("\n=== Computing TEMPORAL Q-Q analysis (domain-averaged) ===")
    
    mswx_daily_avg = ds_mswx["air_temperature"].mean(dim=['lat', 'lon']).values
    ddpm_daily_avg = ds_ddpm["air_temperature"].mean(dim=['lat', 'lon']).values
    
    mask_temporal = np.isfinite(mswx_daily_avg) & np.isfinite(ddpm_daily_avg)
    mswx_vals_temporal = mswx_daily_avg[mask_temporal]
    ddpm_vals_temporal = ddpm_daily_avg[mask_temporal]
    
    print(f"Temporal samples (days): {len(mswx_vals_temporal)}")
    print(f"Temperature range: {mswx_vals_temporal.min():.2f} to {mswx_vals_temporal.max():.2f}°C")

    qq_stats_temporal, q_mswx_temporal, q_ddpm_temporal = compute_qq_statistics(
        mswx_vals_temporal, ddpm_vals_temporal
    )

    # =====================================================================
    # PART C: PER-GRID-CELL TEMPORAL Q-Q (NEW! - Parallelized)
    # =====================================================================
    gridcell_qq_stats = compute_per_gridcell_temporal_qq(ds_mswx, ds_ddpm, gdf)
    
    # Save to NetCDF for later analysis
    print(f"\nSaving per-grid-cell statistics to: {GRIDCELL_STATS_FILE}")
    gridcell_qq_stats.to_netcdf(GRIDCELL_STATS_FILE)
    
    # Compute summary statistics
    print("\n=== Per-Grid-Cell Summary Statistics ===")
    print(f"R² mean: {np.nanmean(gridcell_qq_stats['r_squared'].values):.6f}")
    print(f"R² median: {np.nanmedian(gridcell_qq_stats['r_squared'].values):.6f}")
    print(f"R² min: {np.nanmin(gridcell_qq_stats['r_squared'].values):.6f}")
    print(f"R² max: {np.nanmax(gridcell_qq_stats['r_squared'].values):.6f}")
    print(f"\nSlope mean: {np.nanmean(gridcell_qq_stats['slope'].values):.4f}")
    print(f"Slope median: {np.nanmedian(gridcell_qq_stats['slope'].values):.4f}")
    print(f"\n5th %ile bias mean: {np.nanmean(gridcell_qq_stats['p5_bias'].values):.4f}°C")
    print(f"95th %ile bias mean: {np.nanmean(gridcell_qq_stats['p95_bias'].values):.4f}°C")

    # =====================================================================
    # SAVE STATISTICS
    # =====================================================================
    try:
        with open(STATS_FILE, "w") as f:
            f.write("COMPREHENSIVE BIAS ANALYSIS (DDPM vs MSWX)\n")
            f.write("="*70 + "\n\n")
            
            f.write("SPATIAL Q-Q ANALYSIS (Climatological Mean Field)\n")
            f.write("-"*70 + "\n")
            f.write(f"Number of spatial points: {len(bias_vals_spatial):,}\n")
            f.write(f"Effective DOF: {dof_info['effective_dof']:,}\n")
            f.write(f"Mean bias: {np.mean(bias_vals_spatial):.4f} °C\n")
            f.write(f"Std bias: {np.std(bias_vals_spatial):.4f} °C\n")
            f.write(f"Q-Q Slope: {qq_stats_spatial['slope']:.4f}\n")
            f.write(f"Q-Q Intercept: {qq_stats_spatial['intercept']:.4f} °C\n")
            f.write(f"Q-Q R²: {qq_stats_spatial['r_squared']:.6f}\n")
            f.write(f"5th percentile bias: {qq_stats_spatial['p5_bias']:.4f} °C\n")
            f.write(f"95th percentile bias: {qq_stats_spatial['p95_bias']:.4f} °C\n\n")
            
            f.write("TEMPORAL Q-Q ANALYSIS (Daily Domain Averages)\n")
            f.write("-"*70 + "\n")
            f.write(f"Number of days: {len(mswx_vals_temporal):,}\n")
            f.write(f"Temperature range (MSWX): {mswx_vals_temporal.min():.2f} to {mswx_vals_temporal.max():.2f} °C\n")
            f.write(f"Temperature range (DDPM): {ddpm_vals_temporal.min():.2f} to {ddpm_vals_temporal.max():.2f} °C\n")
            f.write(f"Q-Q Slope: {qq_stats_temporal['slope']:.4f}\n")
            f.write(f"Q-Q Intercept: {qq_stats_temporal['intercept']:.4f} °C\n")
            f.write(f"Q-Q R²: {qq_stats_temporal['r_squared']:.6f}\n")
            f.write(f"5th percentile bias: {qq_stats_temporal['p5_bias']:.4f} °C\n")
            f.write(f"95th percentile bias: {qq_stats_temporal['p95_bias']:.4f} °C\n\n")
            
            f.write("PER-GRID-CELL TEMPORAL Q-Q STATISTICS\n")
            f.write("-"*70 + "\n")
            f.write(f"R² mean: {np.nanmean(gridcell_qq_stats['r_squared'].values):.6f}\n")
            f.write(f"R² median: {np.nanmedian(gridcell_qq_stats['r_squared'].values):.6f}\n")
            f.write(f"R² std: {np.nanstd(gridcell_qq_stats['r_squared'].values):.6f}\n")
            f.write(f"R² range: [{np.nanmin(gridcell_qq_stats['r_squared'].values):.6f}, "
                    f"{np.nanmax(gridcell_qq_stats['r_squared'].values):.6f}]\n\n")
            f.write(f"Slope mean: {np.nanmean(gridcell_qq_stats['slope'].values):.4f}\n")
            f.write(f"Slope median: {np.nanmedian(gridcell_qq_stats['slope'].values):.4f}\n")
            f.write(f"Slope std: {np.nanstd(gridcell_qq_stats['slope'].values):.4f}\n\n")
            f.write(f"5th %ile bias mean: {np.nanmean(gridcell_qq_stats['p5_bias'].values):.4f} °C\n")
            f.write(f"95th %ile bias mean: {np.nanmean(gridcell_qq_stats['p95_bias'].values):.4f} °C\n")
            
        print(f"\nStatistics saved to: {STATS_FILE}")
    except Exception as e:
        print(f"Warning: failed to save stats file: {e}")

    # =====================================================================
    # PLOT 1: MAIN 3-PANEL FIGURE (unchanged)
    # =====================================================================
    print("\n=== Creating main 3-panel figure ===")
    
    fig = plt.figure(figsize=FIGSIZE_MAIN)
    gs = GridSpec(1, 3, figure=fig, wspace=0.30)

    # Panel (a): Bias Distribution
    ax1 = fig.add_subplot(gs[0, 0])
    x_range = np.linspace(np.min(bias_vals_spatial), np.max(bias_vals_spatial), 300)

    try:
        kde = gaussian_kde(bias_vals_spatial)
        kde_vals = kde(x_range)
        ax1.plot(x_range, kde_vals, color='steelblue', linewidth=2.5, label='KDE')
    except Exception:
        pass

    mu, sigma = stats.norm.fit(bias_vals_spatial)
    normal_pdf = stats.norm.pdf(x_range, mu, sigma)
    ax1.plot(x_range, normal_pdf, 'r--', linewidth=2, label='Normal fit')
    ax1.hist(bias_vals_spatial, bins=50, density=True, alpha=0.25, 
             color='gray', edgecolor='k', label='Histogram')
    ax1.axvline(0.0, color='black', linestyle='-', linewidth=1.4)
    ax1.axvline(np.mean(bias_vals_spatial), color='darkblue', linestyle='--', 
                linewidth=1.6, label=f"Mean = {np.mean(bias_vals_spatial):.3f} °C")

    ax1.set_xlabel('Spatial Bias (DDPM − MSWX) (°C)', fontsize=LABEL_FS, fontweight='bold')
    ax1.set_ylabel('Density', fontsize=LABEL_FS, fontweight='bold')
    ax1.set_title('(a) Spatial Bias Distribution', fontsize=LABEL_FS, fontweight='bold', pad=10)
    ax1.legend(fontsize=LEGEND_FS, loc='upper right')
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(axis='both', labelsize=TICK_FS)

    # Panel (b): Spatial Q-Q
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.scatter(q_mswx_spatial, q_ddpm_spatial, alpha=0.6, s=28, 
                color='steelblue', label='Quantile pairs')
    min_val = min(q_mswx_spatial.min(), q_ddpm_spatial.min())
    max_val = max(q_mswx_spatial.max(), q_ddpm_spatial.max())
    ax2.plot([min_val, max_val], [min_val, max_val], 'k--', 
             linewidth=1.6, label='1:1 reference')
    fitted_line = qq_stats_spatial['slope'] * q_mswx_spatial + qq_stats_spatial['intercept']
    ax2.plot(q_mswx_spatial, fitted_line, color='green', linewidth=2.0, 
             label=f"Fit: y={qq_stats_spatial['slope']:.3f}x{qq_stats_spatial['intercept']:+.2f}")

    ax2.set_xlabel('MSWX Spatial Quantiles (°C)', fontsize=LABEL_FS, fontweight='bold')
    ax2.set_ylabel('DDPM Spatial Quantiles (°C)', fontsize=LABEL_FS, fontweight='bold')
    ax2.set_title(f'(b) Spatial Q-Q (R²={qq_stats_spatial["r_squared"]:.4f})', 
                  fontsize=LABEL_FS, fontweight='bold', pad=10)
    ax2.legend(fontsize=LEGEND_FS, loc='lower right')
    ax2.grid(True, alpha=0.3)
    ax2.set_aspect('equal', adjustable='box')
    ax2.tick_params(axis='both', labelsize=TICK_FS)

    # Panel (c): Temporal Q-Q
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.scatter(q_mswx_temporal, q_ddpm_temporal, alpha=0.6, s=28, 
                color='coral', label='Quantile pairs')
    min_val_t = min(q_mswx_temporal.min(), q_ddpm_temporal.min())
    max_val_t = max(q_mswx_temporal.max(), q_ddpm_temporal.max())
    ax3.plot([min_val_t, max_val_t], [min_val_t, max_val_t], 'k--', 
             linewidth=1.6, label='1:1 reference')
    fitted_line_t = qq_stats_temporal['slope'] * q_mswx_temporal + qq_stats_temporal['intercept']
    ax3.plot(q_mswx_temporal, fitted_line_t, color='darkred', linewidth=2.0, 
             label=f"Fit: y={qq_stats_temporal['slope']:.3f}x{qq_stats_temporal['intercept']:+.2f}")

    ax3.set_xlabel('MSWX Temporal Quantiles (°C)', fontsize=LABEL_FS, fontweight='bold')
    ax3.set_ylabel('DDPM Temporal Quantiles (°C)', fontsize=LABEL_FS, fontweight='bold')
    ax3.set_title(f'(c) Temporal Q-Q (R²={qq_stats_temporal["r_squared"]:.4f})', 
                  fontsize=LABEL_FS, fontweight='bold', pad=10)
    ax3.legend(fontsize=LEGEND_FS, loc='lower right')
    ax3.grid(True, alpha=0.3)
    ax3.set_aspect('equal', adjustable='box')
    ax3.tick_params(axis='both', labelsize=TICK_FS)

    plt.tight_layout()
    plt.savefig(PLOT_MAIN_FILE, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Saved main figure: {PLOT_MAIN_FILE}")
    plt.close()

    # =====================================================================
    # PLOT 2: PER-GRID-CELL TEMPORAL Q-Q MAPS (NEW!)
    # =====================================================================
    print("\n=== Creating per-grid-cell temporal Q-Q maps ===")
    
    fig, axes = plt.subplots(2, 2, figsize=FIGSIZE_MAPS, subplot_kw={'projection': None})
    fig.suptitle('Per-Grid-Cell Temporal Q-Q Statistics', 
                 fontsize=TITLE_FS+2, fontweight='bold', y=0.98)

    # Plot India boundary
    for ax in axes.flat:
        gdf.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=1.5)
        ax.set_xlabel('Longitude', fontsize=LABEL_FS)
        ax.set_ylabel('Latitude', fontsize=LABEL_FS)
        ax.tick_params(labelsize=TICK_FS)

    # (a) R² map
    ax = axes[0, 0]
    im1 = gridcell_qq_stats['r_squared'].plot(
        ax=ax, 
        cmap='RdYlGn', 
        vmin=0.95, 
        vmax=1.0,
        cbar_kwargs={'label': 'R²', 'shrink': 0.8}
    )
    ax.set_title('(a) Temporal Q-Q R² (daily time series)', 
                 fontsize=LABEL_FS, fontweight='bold', pad=10)
    
    # (b) Slope map
    ax = axes[0, 1]
    im2 = gridcell_qq_stats['slope'].plot(
        ax=ax, 
        cmap='RdBu_r', 
        vmin=0.95, 
        vmax=1.05,
        cbar_kwargs={'label': 'Slope', 'shrink': 0.8}
    )
    ax.set_title('(b) Temporal Q-Q Slope (ideal = 1.0)', 
                 fontsize=LABEL_FS, fontweight='bold', pad=10)
    
    # (c) 5th percentile bias map (cold days)
    ax = axes[1, 0]
    im3 = gridcell_qq_stats['p5_bias'].plot(
        ax=ax, 
        cmap='RdBu_r', 
        vmin=-0.5, 
        vmax=0.5,
        cbar_kwargs={'label': 'Bias (°C)', 'shrink': 0.8}
    )
    ax.set_title('(c) 5th Percentile Bias (cold days)', 
                 fontsize=LABEL_FS, fontweight='bold', pad=10)
    
    # (d) 95th percentile bias map (hot days)
    ax = axes[1, 1]
    im4 = gridcell_qq_stats['p95_bias'].plot(
        ax=ax, 
        cmap='RdBu_r', 
        vmin=-0.5, 
        vmax=0.5,
        cbar_kwargs={'label': 'Bias (°C)', 'shrink': 0.8}
    )
    ax.set_title('(d) 95th Percentile Bias (hot days)', 
                 fontsize=LABEL_FS, fontweight='bold', pad=10)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(PLOT_GRIDCELL_FILE, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Saved per-grid-cell figure: {PLOT_GRIDCELL_FILE}")
    plt.close()

    print("\n=== Analysis Complete ===")
    print(f"Spatial Q-Q: R² = {qq_stats_spatial['r_squared']:.6f}, "
          f"5th/95th %ile bias = {qq_stats_spatial['p5_bias']:.3f}/{qq_stats_spatial['p95_bias']:.3f}°C")
    print(f"Temporal Q-Q: R² = {qq_stats_temporal['r_squared']:.6f}, "
          f"5th/95th %ile bias = {qq_stats_temporal['p5_bias']:.3f}/{qq_stats_temporal['p95_bias']:.3f}°C")
    print(f"Per-grid-cell R² mean: {np.nanmean(gridcell_qq_stats['r_squared'].values):.6f}")

if __name__ == "__main__":
    main()

In [ ]:
import  xarray as xr
ds = xr.open_dataset("<SET YOUR PATH HERE>")
print(ds)


In [ ]:
#!/usr/bin/env python3
"""
Plot per-grid-cell temporal Q-Q statistics from NetCDF file
"""

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Configuration
NC_FILE = "<SET YOUR PATH HERE>"
INDIA_SHP = "<SET YOUR PATH HERE>"
OUTPUT_FILE = "per_gridcell_temporal_qq_maps_publication.png"

# Font sizes
TITLE_FS = 18
LABEL_FS = 16
TICK_FS = 14
CBAR_FS = 14

# Load data
print("Loading data...")
ds = xr.open_dataset(NC_FILE)

print("\nDataset summary:")
print(ds)
print(f"\nR² statistics:")
print(f"  Mean: {np.nanmean(ds['r_squared'].values):.6f}")
print(f"  Median: {np.nanmedian(ds['r_squared'].values):.6f}")
print(f"  Min: {np.nanmin(ds['r_squared'].values):.6f}")
print(f"  Max: {np.nanmax(ds['r_squared'].values):.6f}")

# Create figure with 4 panels
fig = plt.figure(figsize=(20, 16))
gs = GridSpec(2, 2, figure=fig, wspace=0.25, hspace=0.25)

# Add overall title
fig.suptitle('Per-Grid-Cell Temporal Q-Q Statistics (1,973 daily time series per location)', 
             fontsize=TITLE_FS+2, fontweight='bold', y=0.98)

# Define subplots
axes = [fig.add_subplot(gs[i, j], projection=ccrs.PlateCarree()) 
        for i in range(2) for j in range(2)]

# India boundaries (approximate)
india_extent = [68, 98, 6, 38]  # [lon_min, lon_max, lat_min, lat_max]

# ============================================================================
# Panel (a): R² map
# ============================================================================
ax = axes[0]
ax.set_extent(india_extent, crs=ccrs.PlateCarree())

# Plot R²
im1 = ax.pcolormesh(
    ds.lon, ds.lat, ds['r_squared'],
    transform=ccrs.PlateCarree(),
    cmap='RdYlGn',
    vmin=0.995,
    vmax=1.000,
    shading='auto'
)

# Add features
ax.coastlines(resolution='10m', linewidth=0.8)

ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)

# Colorbar
cbar1 = plt.colorbar(im1, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8)
cbar1.set_label('R²', fontsize=CBAR_FS, fontweight='bold')
cbar1.ax.tick_params(labelsize=TICK_FS)

ax.set_title('(a) Temporal Q-Q R²', fontsize=LABEL_FS, fontweight='bold', pad=10)

# ============================================================================
# Panel (b): Slope map
# ============================================================================
ax = axes[1]
ax.set_extent(india_extent, crs=ccrs.PlateCarree())

# Plot slope
im2 = ax.pcolormesh(
    ds.lon, ds.lat, ds['slope'],
    transform=ccrs.PlateCarree(),
    cmap='RdBu_r',
    vmin=0.95,
    vmax=1.05,
    shading='auto'
)

ax.coastlines(resolution='10m', linewidth=0.8)

ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)

cbar2 = plt.colorbar(im2, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8)
cbar2.set_label('Slope', fontsize=CBAR_FS, fontweight='bold')
cbar2.ax.tick_params(labelsize=TICK_FS)

ax.set_title('(b) Temporal Q-Q Slope (ideal = 1.0)', fontsize=LABEL_FS, fontweight='bold', pad=10)

# ============================================================================
# Panel (c): 5th percentile bias (cold days)
# ============================================================================
ax = axes[2]
ax.set_extent(india_extent, crs=ccrs.PlateCarree())

# Plot 5th percentile bias
im3 = ax.pcolormesh(
    ds.lon, ds.lat, ds['p5_bias'],
    transform=ccrs.PlateCarree(),
    cmap='RdBu_r',
    vmin=-0.3,
    vmax=0.3,
    shading='auto'
)

ax.coastlines(resolution='10m', linewidth=0.8)

ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)

cbar3 = plt.colorbar(im3, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8)
cbar3.set_label('Bias (°C)', fontsize=CBAR_FS, fontweight='bold')
cbar3.ax.tick_params(labelsize=TICK_FS)

ax.set_title('(c) 5th Percentile Bias (cold days)', fontsize=LABEL_FS, fontweight='bold', pad=10)

# ============================================================================
# Panel (d): 95th percentile bias (hot days)
# ============================================================================
ax = axes[3]
ax.set_extent(india_extent, crs=ccrs.PlateCarree())

# Plot 95th percentile bias
im4 = ax.pcolormesh(
    ds.lon, ds.lat, ds['p95_bias'],
    transform=ccrs.PlateCarree(),
    cmap='RdBu_r',
    vmin=-0.5,
    vmax=0.1,
    shading='auto'
)

ax.coastlines(resolution='10m', linewidth=0.8)
ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)

cbar4 = plt.colorbar(im4, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8)
cbar4.set_label('Bias (°C)', fontsize=CBAR_FS, fontweight='bold')
cbar4.ax.tick_params(labelsize=TICK_FS)

ax.set_title('(d) 95th Percentile Bias (hot days)', fontsize=LABEL_FS, fontweight='bold', pad=10)

# Save
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(OUTPUT_FILE, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\nSaved publication-quality figure: {OUTPUT_FILE}")

plt.show()

# ============================================================================
# Print summary statistics
# ============================================================================
print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)

for var_name in ['r_squared', 'slope', 'p5_bias', 'p95_bias']:
    data = ds[var_name].values
    print(f"\n{var_name}:")
    print(f"  Mean:   {np.nanmean(data):.6f}")
    print(f"  Median: {np.nanmedian(data):.6f}")
    print(f"  Std:    {np.nanstd(data):.6f}")
    print(f"  Min:    {np.nanmin(data):.6f}")
    print(f"  Max:    {np.nanmax(data):.6f}")
    print(f"  5th %:  {np.nanpercentile(data, 5):.6f}")
    print(f"  95th %: {np.nanpercentile(data, 95):.6f}")

In [ ]:
# Complete cell: compute overall_mswx_clipped / overall_ddpm_clipped, extract values, plot PDFs
import glob, os
import numpy as np
import xarray as xr
import geopandas as gpd
import rioxarray
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings("ignore")

# --------- USER CONFIG: update paths if needed ----------
HR_MSWX = "<SET YOUR PATH HERE>"
HR_DDPM = "<SET YOUR PATH HERE>"
INDIA_SHP = "<SET YOUR PATH HERE>"

TEST_START = 2009219   # inclusive YYYYDOY
TEST_END   = 2014365   # inclusive YYYYDOY

OUT_PNG = "mswx_ddpm_pdf_comparison.png"  # output filename
FIGSIZE = (11,6)

# ---------------- helper functions ----------------
def get_file_date(fname):
    base = os.path.basename(fname)
    base = base.replace("_ddpm","").replace(".nc","")
    # try YYYYDOY (7 or 8 digits), then YYYY-MM-DD
    import re
    m = re.search(r"(\d{7,8})", base)
    if m:
        return int(m.group(1))
    m2 = re.search(r"(\d{4}-\d{2}-\d{2})", base)
    if m2:
        from pandas import to_datetime
        return int(to_datetime(m2.group(1)).strftime("%Y%j"))
    # fallback
    try:
        return int(''.join(ch for ch in base if ch.isdigit()))
    except:
        return None

def doy_to_datetime(yydoy):
    import pandas as pd
    s = str(int(yydoy))
    # handle both 7 and 8-digit (YYYYddd or YYYYMMDD)
    if len(s) == 7:
        year = s[:4]; doy = s[4:]
        return pd.to_datetime(f"{year}-{doy}", format="%Y-%j")
    elif len(s) == 8:
        # maybe YYYYMMDD
        return pd.to_datetime(s, format="%Y%m%d")
    else:
        # fallback - treat as year only
        return pd.to_datetime(s[:4] + "-01-01")

def filter_period(files, start, end):
    out = []
    for f in sorted(files):
        d = get_file_date(f)
        if d is None: 
            continue
        if start <= d <= end:
            out.append(f)
    return out

def find_temp_var(ds):
    # prefer common names
    for v in ("air_temperature","tas","t2m","temperature","temp"):
        if v in ds.data_vars:
            return v
    # fallback: choose first float 2D+var
    for v in ds.data_vars:
        if np.issubdtype(ds[v].dtype, np.floating) and ds[v].ndim >= 2:
            return v
    raise RuntimeError("No temperature-like variable found")

# ---------------- collect files ----------------
mswx_files = sorted(glob.glob(os.path.join(HR_MSWX, "*.nc")))
ddpm_files = sorted(glob.glob(os.path.join(HR_DDPM, "*.nc")))

mswx_files = filter_period(mswx_files, TEST_START, TEST_END)
ddpm_files = filter_period(ddpm_files, TEST_START, TEST_END)

if not mswx_files:
    raise RuntimeError(f"No MSWX files found in period {TEST_START}-{TEST_END}")
if not ddpm_files:
    raise RuntimeError(f"No DDPM files found in period {TEST_START}-{TEST_END}")

print(f"Found {len(mswx_files)} MSWX files and {len(ddpm_files)} DDPM files for the period.")

# ---------------- open datasets lazily ----------------
# Use combine="nested" since filenames are daily tiles in time ordering
ds_mswx = xr.open_mfdataset(mswx_files, combine="nested", concat_dim="time", engine="netcdf4", parallel=False)
ds_ddpm = xr.open_mfdataset(ddpm_files, combine="nested", concat_dim="time", engine="netcdf4", parallel=False)

# assign times from filenames if missing or not datetime-like
if "time" not in ds_mswx.coords or not np.issubdtype(ds_mswx["time"].dtype, np.datetime64):
    times = [doy_to_datetime(get_file_date(f)) for f in mswx_files]
    ds_mswx = ds_mswx.assign_coords(time=("time", times))
if "time" not in ds_ddpm.coords or not np.issubdtype(ds_ddpm["time"].dtype, np.datetime64):
    times = [doy_to_datetime(get_file_date(f)) for f in ddpm_files]
    ds_ddpm = ds_ddpm.assign_coords(time=("time", times))

# ---------------- pick temperature variable ----------------
var_m = find_temp_var(ds_mswx)
var_d = find_temp_var(ds_ddpm)
print("Using variables:", var_m, "and", var_d)

# ---------------- compute multi-year mean field (mean over time -> mean over years)
# approach: groupby year then mean->time per year -> mean across years
# this avoids calendar issues and matches your earlier pipeline.
try:
    yearly_m = ds_mswx[var_m].groupby("time.year").mean(dim="time", skipna=True)
    overall_mswx = yearly_m.mean(dim="year", skipna=True)
except Exception:
    # fallback: straight mean over time
    overall_mswx = ds_mswx[var_m].mean(dim="time", skipna=True)

try:
    yearly_d = ds_ddpm[var_d].groupby("time.year").mean(dim="time", skipna=True)
    overall_ddpm = yearly_d.mean(dim="year", skipna=True)
except Exception:
    overall_ddpm = ds_ddpm[var_d].mean(dim="time", skipna=True)

# ---------------- rename coords to x/y for rioxarray, set CRS ----------------
# Expect coordinates named 'lat' & 'lon' (or 'y' & 'x'). Normalize to x/y
def ensure_rio(da):
    da = da.copy()
    if "lat" in da.coords and "lon" in da.coords:
        da = da.rename({"lat":"y","lon":"x"})
    # set spatial dims & CRS for rioxarray (EPSG:4326), skip if already set
    try:
        da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
        da.rio.write_crs("EPSG:4326", inplace=True)
    except Exception:
        pass
    return da

overall_mswx = ensure_rio(overall_mswx)
overall_ddpm = ensure_rio(overall_ddpm)

# ---------------- read India shapefile and clip ----------------
gdf = gpd.read_file(INDIA_SHP)
# ensure same CRS
if gdf.crs is None:
    gdf = gdf.set_crs("EPSG:4326")

overall_mswx_clipped = overall_mswx.rio.clip(gdf.geometry, gdf.crs, drop=False)
overall_ddpm_clipped = overall_ddpm.rio.clip(gdf.geometry, gdf.crs, drop=False)

# ---------------- extract flattened arrays and convert to °C if needed ----------------
mswx_vals = overall_mswx_clipped.values.flatten()
ddpm_vals = overall_ddpm_clipped.values.flatten()

# Keep only finite
mswx_vals = mswx_vals[np.isfinite(mswx_vals)]
ddpm_vals = ddpm_vals[np.isfinite(ddpm_vals)]

# If values appear to be in Kelvin (mean > 200), convert to Celsius
if mswx_vals.size and np.nanmean(mswx_vals) > 200:
    print("Converting MSWX from K -> °C")
    mswx_vals = mswx_vals - 273.15
if ddpm_vals.size and np.nanmean(ddpm_vals) > 200:
    print("Converting DDPM from K -> °C")
    ddpm_vals = ddpm_vals - 273.15

print("Final sample sizes (after clipping & dropping NaN): MSWX =", mswx_vals.size, ", DDPM =", ddpm_vals.size)

# ---------------- Plot PDFs (KDE line + histograms) ----------------
xmin = min(mswx_vals.min(), ddpm_vals.min())
xmax = max(mswx_vals.max(), ddpm_vals.max())
xgrid = np.linspace(xmin, xmax, 400)

kde_mswx = gaussian_kde(mswx_vals)
kde_ddpm = gaussian_kde(ddpm_vals)
pdf_mswx = kde_mswx(xgrid)
pdf_ddpm = kde_ddpm(xgrid)

plt.figure(figsize=FIGSIZE)
plt.plot(xgrid, pdf_mswx, color='black', linewidth=2.4, label="MSWX (KDE)")
plt.plot(xgrid, pdf_ddpm, color='royalblue', linewidth=2.4, label="DDPM (KDE)")

plt.hist(mswx_vals, bins=50, density=True, alpha=0.20, color='gray', edgecolor='k', label='MSWX histogram')
plt.hist(ddpm_vals, bins=50, density=True, alpha=0.20, color='skyblue', edgecolor='k', label='DDPM histogram')

plt.xlabel("Temperature (°C)", fontsize=14, fontweight='bold')
plt.ylabel("Probability density", fontsize=14, fontweight='bold')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.25)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig(OUT_PNG, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("Saved PDF comparison to:", OUT_PNG)

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import rioxarray
import geopandas as gpd
import seaborn as sns
import statsmodels.api as sm
import warnings

# Suppress warnings from xarray/rioxarray
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# -----------------------------
# 1. Paths
# -----------------------------
hr_mswx = "<SET YOUR PATH HERE>"
hr_ddpm = "<SET YOUR PATH HERE>"
india_shp = "<SET YOUR PATH HERE>"
output_plot_dir = "<SET YOUR PATH HERE>"
os.makedirs(output_plot_dir, exist_ok=True)

# -----------------------------
# 2. Helper functions
# -----------------------------
def get_file_date(fname):
    """Extracts YYYYDOY from filename."""
    base = os.path.basename(fname)
    base = base.replace("_ddpm", "").replace(".nc", "")
    return int(base)

def doy_to_datetime(yydoy):
    """Converts YYYYDOY to datetime object."""
    year = int(str(yydoy)[:4])
    doy = int(str(yydoy)[4:])
    return pd.to_datetime(f"{year}-{doy:03d}", format="%Y-%j")

def filter_period(files, start=2009219, end=2014365): # Testing period
    """Filters a list of files to be within the start/end YYYYDOY range."""
    valid = []
    for f in files:
        try:
            d = get_file_date(f)
            if start <= d <= end:
                valid.append(f)
        except:
            continue
    return valid

# -----------------------------
# 3. Load files
# -----------------------------
print("Loading files...")
f_mswx = sorted(glob.glob(os.path.join(hr_mswx, "*.nc")))
f_ddpm = sorted(glob.glob(os.path.join(hr_ddpm, "*_ddpm.nc")))
f_mswx = filter_period(f_mswx)
f_ddpm = filter_period(f_ddpm)

# -----------------------------
# 4. Open and Process Data
# -----------------------------
print("Opening datasets...")
ds_mswx = xr.open_mfdataset(f_mswx, combine="nested", concat_dim="time", engine="netcdf4")
ds_ddpm = xr.open_mfdataset(f_ddpm, combine="nested", concat_dim="time", engine="netcdf4")

# Assign time
if "time" not in ds_mswx.coords:
    doy_list = [get_file_date(f) for f in f_mswx]
    times = [doy_to_datetime(d) for d in doy_list]
    ds_mswx = ds_mswx.assign_coords(time=("time", times))
    ds_ddpm = ds_ddpm.assign_coords(time=("time", times))

print("✅ Data loaded and time coordinates assigned")

# -----------------------------
# 5. Load Shapefile
# -----------------------------
print("Loading shapefile...")
gdf = gpd.read_file(india_shp)

# -----------------------------
# 6. Define Seasons
# -----------------------------
# IMD Defined Seasons
seasons = {
    'Winter (DJF)': [12, 1, 2],
    'Pre-Monsoon (MAM)': [3, 4, 5],
    'Monsoon (JJAS)': [6, 7, 8, 9],
    'Post-Monsoon (ON)': [10, 11]
}

# -----------------------------
# 7. Main Seasonal Loop
# -----------------------------
print("Starting seasonal analysis loop...")

for season_name, months in seasons.items():
    
    print("\n" + "="*50)
    print(f"Processing Season: {season_name}")
    print("="*50)

    # --- 7.1: Filter data by season ---
    print("  Calculating seasonal means...")
    seasonal_mswx_data = ds_mswx.sel(time=ds_mswx.time.dt.month.isin(months))
    seasonal_ddpm_data = ds_ddpm.sel(time=ds_ddpm.time.dt.month.isin(months))
    
    # Calculate the mean *over all days in that season*
    overall_mswx = seasonal_mswx_data["air_temperature"].mean('time')
    overall_ddpm = seasonal_ddpm_data["air_temperature"].mean('time')
    overall_bias = overall_ddpm - overall_mswx

    # --- 7.2: Clip data ---
    print("  Clipping data to shapefile...")
    overall_mswx = overall_mswx.rename({"lat": "y", "lon": "x"})
    overall_ddpm = overall_ddpm.rename({"lat": "y", "lon": "x"})
    overall_bias = overall_bias.rename({"lat": "y", "lon": "x"})

    for da in [overall_mswx, overall_ddpm, overall_bias]:
        da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
        da.rio.write_crs("EPSG:4326", inplace=True)

    overall_mswx_clipped = overall_mswx.rio.clip(gdf.geometry, gdf.crs)
    overall_ddpm_clipped = overall_ddpm.rio.clip(gdf.geometry, gdf.crs)
    overall_bias_clipped = overall_bias.rio.clip(gdf.geometry, gdf.crs)

    # --- 7.3: Extract values ---
    print("  Extracting values for plotting...")
    mswx_vals = overall_mswx_clipped.values.flatten()
    mswx_vals = mswx_vals[~np.isnan(mswx_vals)]

    ddpm_vals = overall_ddpm_clipped.values.flatten()
    ddpm_vals = ddpm_vals[~np.isnan(ddpm_vals)]

    bias_vals = overall_bias_clipped.values.flatten()
    bias_vals = bias_vals[~np.isnan(bias_vals)]

    if len(mswx_vals) == 0:
        print(f"  No valid data found for {season_name}. Skipping plot.")
        continue
        
    # ----------------------------------------------------
    # 7.4: Calculate & Print Summary Statistics
    # ----------------------------------------------------
    print("\n" + "="*50)
    print(f"     SUMMARY: Distributional Statistics for {season_name}")
    print("="*50 + "\n")

    # --- Bias Distribution Statistics (for Plot a) ---
    stats_bias_mean = float(np.mean(bias_vals))
    stats_bias_median = float(np.median(bias_vals))
    stats_bias_std = float(np.std(bias_vals))
    stats_mab = float(np.mean(np.abs(bias_vals)))
    stats_rmse = float(np.sqrt(np.mean(np.square(bias_vals))))
    stats_bias_min = float(np.min(bias_vals))
    stats_bias_max = float(np.max(bias_vals))

    print("--- (a) Spatial Bias Distribution Statistics ---")
    print(f"Mean Bias (MB):   {stats_bias_mean:.4f} °C")
    print(f"Median Bias:      {stats_bias_median:.4f} °C")
    print(f"Std Dev of Bias:  {stats_bias_std:.4f} °C")
    print(f"Mean Absolute (MAB): {stats_mab:.4f} °C")
    print(f"RMSE of Bias:     {stats_rmse:.4f} °C")
    print(f"Min Bias:         {stats_bias_min:.4f} °C")
    print(f"Max Bias:         {stats_bias_max:.4f} °C")
    print("\n")

    # --- Q-Q Plot Statistics (for Plot b) ---
    percentiles = np.arange(1, 100)
    q_mswx = np.percentile(mswx_vals, q=percentiles)
    q_ddpm = np.percentile(ddpm_vals, q=percentiles)

    slope, intercept = np.polyfit(q_mswx, q_ddpm, 1)
    corr = np.corrcoef(q_mswx, q_ddpm)[0, 1]
    p5_bias = np.percentile(ddpm_vals, 5) - np.percentile(mswx_vals, 5)
    p95_bias = np.percentile(ddpm_vals, 95) - np.percentile(mswx_vals, 95)

    print("--- (b) Quantile-Quantile (Q-Q) Plot Statistics ---")
    print(f"Quantile Correlation (r): {corr:.6f}")
    print(f"Quantile Linear Fit:      y = {slope:.4f}x + {intercept:.4f}")
    print(f"Bias at 5th Percentile:   {p5_bias:.4f} °C")
    print(f"Bias at 95th Percentile:  {p95_bias:.4f} °C")
    print("\n" + "="*50 + "\n")


    # --- 7.5: Create Plots ---
    print("  Generating distribution plots...")
    sns.set_style("whitegrid")
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Add an overall title for the figure
    fig.suptitle(f"Seasonal Analysis: {season_name}", fontsize=18, y=1.02)

    # --- Plot (a): Bias Distribution ---
    ax1 = axes[0]
    sns.kdeplot(bias_vals, ax=ax1, label="Bias Distribution", color="green", fill=True)
    ax1.axvline(x=0, color="black", linestyle="--", label="Zero Bias")
    ax1.set_title("Spatial Bias Distribution", fontsize=16)
    ax1.set_xlabel("Bias (DDPM − MSWX) (°C)", fontsize=14)
    ax1.set_ylabel("Density", fontsize=14)
    ax1.legend(fontsize=12)
    ax1.tick_params(axis='both', which='major', labelsize=12) 
    ax1.text(0.05, 0.95, "(a)",
             transform=ax1.transAxes, 
             fontsize=16, 
             fontweight='bold', 
             va='top', 
             ha='left')

    # --- Plot (b): Quantile-Quantile (Q-Q) Plot ---
    ax2 = axes[1]
    sm.qqplot_2samples(
        mswx_vals, 
        ddpm_vals, 
        ax=ax2, 
        line='45',
        xlabel="MSWX Quantiles (°C)",
        ylabel="DDPM Quantiles (°C)"
    )
    ax2.set_title("Quantile-Quantile (Q-Q) Plot", fontsize=16)
    ax2.get_xaxis().label.set_size(14)
    ax2.get_yaxis().label.set_size(14)
    ax2.tick_params(axis='both', which='major', labelsize=12)

    for line in ax2.get_lines():
        if line.get_linestyle() == 'None': # Scatter plot
            line.set_marker('o')
            line.set_markersize(3)       
            line.set_alpha(0.3)          
            line.set_color('blue')
        else: # 1:1 line
            line.set_color('red')
            line.set_linestyle('--')
            line.set_label('1:1 Line')

    ax2.legend(fontsize=12, loc='lower right')
    ax2.text(0.05, 0.95, "(b)",
             transform=ax2.transAxes, 
             fontsize=16, 
             fontweight='bold', 
             va='top', 
             ha='left')

    plt.tight_layout(rect=[0, 0, 1, 0.95]) # Adjust layout for suptitle

    # --- 7.6: Save and Show ---
    plt.show()

    # # Save the figure with high DPI
    # safe_season_name = season_name.replace(" ", "_").replace("(", "").replace(")", "")
    # save_path = os.path.join(output_plot_dir, f"DDPM_Bias_QQ_Plot_{safe_season_name}.png")
    # fig.savefig(save_path, dpi=300, bbox_inches='tight')
    # print(f"✅ Distribution plot complete. Figure saved to {save_path}")

print("\nAll seasonal analyses complete.")